# SINCON ensemble_v3 재현

4-fold expanding-origin CV → 로그 OOF 볼록 앙상블 → 2개 시드 전체 재학습 → 제출 파일.
각 fold는 80만 행, 최종 학습은 100만 행이며 기본 설정을 축소하지 않습니다.
LightGBM은 CPU, CatBoost는 CUDA 0, XGBoost는 CUDA 1(한 장이면 0), TabM은 CUDA 0/MPS/CPU를 자동 선택합니다.
TabM은 CUDA에서만 FP16 + GradScaler를 사용합니다. CUDA CatBoost의 비트 단위 결정성은 보장되지 않습니다.
Kaggle에서는 4개 대회 CSV를 Add Input하고 GPU T4 × 2를 선택한 뒤 Run All 하세요.
Colab에서는 CSV 폴더를 업로드하거나 마운트하고 아래 DATA_DIR_OVERRIDE를 설정하세요.
실행시간은 장치와 조기 종료 시점에 따라 달라집니다. 현재 런타임의 세션 제한은 별도로 확인하세요.
외부 사전학습 가중치는 사용하지 않으며, 달력 공개시점 필터 as_of=origin을 유지합니다.
노트북은 제출 파일을 만들며 Kaggle에 자동 제출하지 않습니다.
최종 앙상블은 Fold 3+4 합산 OOF로 적합하며, 전체 4-fold와 날짜 중복 제거 교차 점검도 저장합니다.
KBO 일정·맥주축제는 제외합니다. 펜타포트는 인천시 공공누리 제1유형 자료의 공개일 이후만 허용합니다.
개최 결과·우천 취소·미래 실측 날씨를 사용하지 않습니다. 지역 학사력과 귀가 시각은 proxy입니다.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
IS_KAGGLE = Path('/kaggle/working').exists()
PROJECT_ROOT = Path('/kaggle/working/SINCON' if IS_KAGGLE else '/content/SINCON')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
EMBEDDED_FILES = {'configs/ensemble_v3.json': '{\n  "version": "ensemble_v3",\n  "seed": 2026,\n  "final_seeds": [\n    2026,\n    2027\n  ],\n  "fold_origins": [\n    "2025-08-30",\n    "2025-10-30",\n    "2025-12-30",\n    "2026-02-27"\n  ],\n  "fold_weights": [\n    0.15,\n    0.15,\n    0.35,\n    0.35\n  ],\n  "horizon_days": 123,\n  "final_cutoff": "2026-06-30",\n  "max_training_rows": 800000,\n  "final_training_rows": 1000000,\n  "threads": 8,\n  "max_cpu_workers": 2,\n  "device": "auto",\n  "models": [\n    "lightgbm",\n    "catboost",\n    "xgboost",\n    "tabm"\n  ],\n  "patience": 150,\n  "lightgbm": {\n    "num_leaves": 159,\n    "min_child_samples": 25,\n    "learning_rate": 0.03,\n    "subsample": 0.8,\n    "subsample_freq": 1,\n    "colsample_bytree": 0.8,\n    "reg_alpha": 0.5,\n    "reg_lambda": 5.0,\n    "n_estimators": 3000,\n    "cat_smooth": 50.0,\n    "cat_l2": 20.0,\n    "extra_trees": true\n  },\n  "catboost": {\n    "depth": 8,\n    "learning_rate": 0.04,\n    "l2_leaf_reg": 5.0,\n    "random_strength": 1.0,\n    "iterations": 3500\n  },\n  "xgboost": {\n    "tree_method": "hist",\n    "enable_categorical": true,\n    "max_cat_to_onehot": 4,\n    "max_depth": 6,\n    "min_child_weight": 80,\n    "learning_rate": 0.03,\n    "subsample": 0.8,\n    "colsample_bytree": 0.75,\n    "reg_alpha": 0.5,\n    "reg_lambda": 5.0,\n    "n_estimators": 3000\n  },\n  "tabm": {\n    "k": 8,\n    "hidden": 256,\n    "blocks": 3,\n    "frequencies": 4,\n    "dropout": 0.15,\n    "learning_rate": 0.001,\n    "weight_decay": 0.0001,\n    "batch_size": 4096,\n    "epochs": 30,\n    "patience": 5,\n    "auxiliary_weight": 0.2\n  },\n  "final_iteration_multiplier": 1.12,\n  "final_iteration_fold_weights": [\n    0.4,\n    0.6\n  ],\n  "selection_metric": "fold34_pooled_rmsle",\n  "ensemble_bias": false,\n  "input_data": "data/raw",\n  "output": "outputs/ensemble_v3",\n  "feature_workers": 4\n}\n', 'reports/holiday_sources.md': '# 공휴일 외부 데이터 출처와 사용 시점\n\n조회일: **2026-11-11 (Asia/Seoul)**. 사용한 달력 사실·공식 발표는 모두 **2026-06-30 이전 공개**된 자료다. 조회일과 정보 공개일을 구별한다. 실현된 교통량·날씨·행사 결과·투표 결과는 수집하거나 사용하지 않았다.\n\n대회에서 허용한 2025-01-01~2026-10-31 공휴일 특징을 `src/holiday_calendar.py`에 정적으로 인코딩했다. 모듈은 같은 공개 달력에 있는 2026-12-25까지 지원하지만 모델의 실제 예측 범위는 10월 31일까지다. 외부 API·인터넷 없이 재현된다.\n\n## 근거 자료\n\n| 코드의 source_id | 기관·문서 | 공개일 | 채택한 사실·조건 |\n| --- | --- | --- | --- |\n| annual_2025 | [한국천문연구원에 게시된 우주항공청 2025년도 월력요항 발표](https://www.kasi.re.kr/kor/publication/post/newsMaterial/30071) | 2024-06-20 | 2025년 설·추석·부처님오신날 및 대체공휴일. 세부 날짜는 [KASI 2025 달력자료](https://astro.kasi.re.kr/kor/life/post/calendarData?search_year=2025), 2024-05-07 생성 자료와 대조. 사전 달력자료의 미확정 표시를 인지하고 최종 월력요항 발표로 확인. |\n| annual_2026 | [우주항공청 2026년 월력요항 발표](https://www.kasa.go.kr/prog/plcyBrf/brief/kor/sub01_01_04/view.do?plcyBrfNo=202) | 2025-06-30 | 2026년 기본 공휴일·대체공휴일. 본문에 설 2/16~18, 대체 3/2·5/25·8/17·10/5, 추석 9/24~26 및 9/27 일요일까지의 연휴 확인. 이 발표 이후 추가된 노동절·제헌절은 아래 공지로 보완. |\n| annual_2026 보조 확인 | [한국천문연구원 2026 달력자료](https://astro.kasi.re.kr/kor/life/post/calendarData?search_year=2026) | 2024-07-25 생성 | 기본 날짜 표와 6/3 지방선거일 확인. 이 자료는 최종 공식 월력요항 발표 전 사전자료이며 그 자체를 최신 공휴일 확정본으로 취급하지 않음. [최종 월력요항 안내](https://astro.kasi.re.kr/kor/life/post/almanac?search_year=2026)와 우주항공청 발표를 함께 사용. |\n| temporary_20250127 | [인사혁신처 1월 27일 임시공휴일 지정 국무회의 의결, 정책브리핑 전재](https://www.korea.kr/briefing/pressReleaseView.do?newsId=156670135) | 2025-01-14 | 2025-01-27 임시공휴일. [정식 공고 제2025-28호](https://www.mpm.go.kr/mpm/comm/noti/mpmNotice/?boardId=bbs_0000000000000205&category=&cntId=904&mode=view&pageIdx=8)는 1/15 게시. CV 가용일은 공개 의결 발표인 1/14. |\n| election_20250603 | [행정안전부 제21대 대통령 선거일 브리핑](https://m.korea.kr/briefing/policyBriefingView.do?newsId=156683072) | 2025-04-08 | 2025-06-03 선거일 및 임시공휴일 지정. [행안부 보도자료](https://mois.go.kr/frt/bbs/type010/commonSelectBoardArticle.do?bbsId=BBSMSTR_000000000008&nttId=116942)와 대조. |\n| annual_2026 선거 보조 확인 | [서울특별시선거관리위원회 제9회 전국동시지방선거 주요일정](https://su.nec.go.kr/su/bbs/B0000263/view.do?category1=su&category2=&deleteCd=0&menuNo=100176&nttId=273718&pageIndex=1) | 2026-01-30 | 선거일 2026-06-03 확인. 선거 결과는 열람·사용하지 않음. 날짜 자체는 KASI의 2024-07-25 달력자료에도 있어 모듈의 보수적 연간 달력 가용일 2025-06-30보다 먼저 공개됨. |\n| constitution_2026 | [인사혁신처 제헌절, 18년 만에 공휴일 재지정](https://www.mpm.go.kr/mpm/comm/newsPress/newsPressRelease/?boardId=bbs_0000000000000029&category=&cntId=4205&mode=view&pageIdx=) | 2026-02-03 | 2026-07-17부터 공휴일임을 공개 발표. [정책브리핑 전재](https://www.korea.kr/briefing/pressReleaseView.do?newsId=156742873)에서도 날짜·배포일 확인. |\n| labor_2026 | [관계부처 합동 노동절 제정 63년 만에 공휴일 지정](https://www.korea.kr/briefing/pressReleaseView.do?newsId=156753075) | 2026-04-06 | 2026-05-01부터 공휴일. 발표 전 CV origin에서는 관공서 공휴일로 표시하지 않음. |\n| 2026 신규 지정 최종 보완 | [인사혁신처 올해부터 노동절, 제헌절 공휴일로](https://www.mpm.go.kr/mpm/comm/newsPress/newsPressRelease/?boardId=bbs_0000000000000029&cntId=4250&mode=view) | 2026-04-29 게시, 4/28 의결 | 노동절·제헌절의 관공서 공휴일 편입과 대체공휴일 적용. 제헌절 법률 2/10 공포·5/11 시행, 노동절 법률 4/9 공포·5/1 시행 경과 확인. 예측에 쓰는 `known_since`는 앞선 공식 공개 발표일이다. |\n\n## 라이선스·재사용 기록\n\n- 우주항공청 **2026년 월력요항 발표** 페이지는 **공공누리 제1유형(출처표시)**를 표시한다.\n- 우주항공청 **2025년 월력요항**의 [원 게시물](https://www.kasa.go.kr/prog/bbsArticle/BBSMSTR_000000000010/view.do?bbsId=BBSMSTR_000000000010&nttId=B000000000633Wg5pO7)은 **공공누리 제2유형(출처표시·상업적 이용금지)**를 표시한다. 대회가 요구하는 비상업적 이용 범위로 출처를 남긴다.\n- 정책브리핑의 인사혁신처·고용노동부 보도자료 전재 페이지는 **텍스트에 한해 공공누리 제1유형**을 표시한다. 본 프로젝트는 날짜·명칭·공개일을 정리하며 사진·그림·서식 파일을 재배포하지 않는다.\n- 행정안전부 보도자료는 **공공누리 제1유형**을 표시한다.\n- 서울선관위 일정 안내는 **공공누리 제4유형(출처표시·상업적 이용금지·변경금지)**를 표시한다. 원 이미지·홍보물은 복제하거나 수정하지 않았으며 공개 선거일의 교차 확인에만 사용했다. 모델에 쓰는 날짜는 이전에 공개된 KASI 달력에도 있다.\n- KASI의 [공공데이터포털 특일 정보](https://www.data.go.kr/data/15012690/openapi.do)는 **이용허락범위 제한 없음**, 무료, 등록 2015-12-21·수정 2023-03-29로 표시한다. [천문우주지식정보 카탈로그](https://www.data.go.kr/catalog/2810264/fileData.json) 역시 월력요항 등을 포함하며 **이용허락범위 제한 없음**, 수정 2023-11-15를 명시한다. 이는 달력 데이터 이용 조건의 참고 기록이며 이번 작업에서 인증 API를 호출했다고 주장하지 않는다.\n- 코드는 위 달력의 사실을 직접 작성한 정적 날짜 목록으로 보관한다. 보도자료 본문이나 디자인을 통째로 복제하지 않는다. 출처 URL·기관·공개일·조회일은 이 문서에 남긴다.\n\n## 예측 시점과 Local CV\n\n기본 호출 `calendar_features(dates, as_of="2026-06-30")`은 최종 예측 시점에 공개된 달력을 적용한다. 검증 또는 과거 학습 origin을 만들 때에는 **그 origin을 as_of로 전달**한다. 임시공휴일·신규 공휴일은 다음 조건으로 활성화된다.\n\n| 대상 날짜 | 내용 | 최초 사용 가능한 공개일 |\n| --- | --- | --- |\n| 2025-01-27 | 임시공휴일 | 2025-01-14 |\n| 2025-06-03 | 대통령선거 임시공휴일 | 2025-04-08 |\n| 2026-05-01 | 노동절의 관공서 공휴일 적용 | 2026-04-06 |\n| 2026-07-17 | 제헌절 공휴일 재지정 | 2026-02-03 |\n\n공개일 전이면 해당 날짜의 `is_holiday=0`, `holiday_type=0`으로 처리한다. ‘나중에 지정될 휴일’이라는 별도 마스크를 모델에 주지 않는다. `is_labor_day`는 두 해 모두 5/1이라는 기존 달력·노동 기념일 표시다. **2025-05-01은 관공서 공휴일로 처리하지 않는다.**\n\n주말은 named holiday와 구별한다. `is_holiday`에는 정기 일요일을 넣지 않고 `calendar_is_weekend`, `is_rest_day`에서 토·일을 표현한다. `is_holiday_eve`, `is_after_holiday`, `is_bridge_day`도 같은 as_of의 달력만 이용한다. 승차량을 휴일이라는 이유로 0으로 강제하지 않는다.\n\n## 경계 조건 확인\n\n- 2025-05-05는 어린이날과 부처님오신날이 겹치지만 한 날짜 한 행으로 인코딩한다. 5/6은 대체공휴일이다.\n- 2025-10-08은 추석 대체공휴일이다.\n- 2026-05-25, 8/17, 10/5는 대체공휴일이다.\n- **2026-09-28은 공휴일이 아니다.** 공식 연휴는 추석 9/24~26과 일요일 9/27까지다.\n- 2026-06-06 토요일 현충일은 공휴일이지만 **6/8을 대체공휴일로 추가하지 않는다.**\n- 최종 달력의 날짜 수는 2025년 19일, 2026년 22일이다(정기 일요일 제외, 중복 공휴일은 한 날짜). 테스트 2026년 7~10월에는 9개 날짜가 포함된다.\n\n상세 날짜 목록은 `reports/holiday_calendar.csv`에 있다. 이 목록은 감사·재현용이며 `known_since`, `source_id`를 모델 입력으로 넣지 않는다.\n', 'requirements.txt': 'numpy==2.3.5\npandas==2.2.3\npolars==2.1.0\nscikit-learn==1.9.1\nscipy==1.18.1\nlightgbm==4.7.0\ncatboost==1.2.10\nxgboost==3.4.1\ntorch==2.15.0\njoblib==1.6.0\nmatplotlib==3.11.2\n', 'src/__init__.py': '"""SINCON reproducible forecasting pipeline."""\n', 'src/domain_events.py': '"""v3 origin-filtered schedule facts and narrowly specified historical masks.\n\nSchedules describe announced plans, never realized attendance/cancellations.\nSchool/time-of-exit variables are explicit proxies, not measured future facts.\n"""\nimport json\nfrom functools import lru_cache\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nOUTAGE_KEYS = {\n    (\'2025-08-13\', 3112, 12), (\'2025-08-13\', 3112, 13),\n    *{(\'2026-01-14\', 3222, h) for h in range(19, 24)},\n    (\'2026-01-18\', 3129, 22), (\'2026-02-09\', 3123, 23),\n}\n\ndef oneoff_mask(frame):\n    dates = pd.to_datetime(frame.date)\n    keys = pd.MultiIndex.from_arrays([dates.dt.strftime(\'%Y-%m-%d\'), frame.station_id, frame.hour])\n    outage = keys.isin(list(OUTAGE_KEYS))\n    opening = frame.station_id.isin([3107, 3108, 3109]) & dates.between(\'2025-06-28\', \'2025-06-30\')\n    return pd.Series(outage | opening, index=frame.index)\n\n@lru_cache(maxsize=1)\ndef schedules():\n    return json.loads(Path(__file__).with_name(\'domain_schedule_v3.json\').read_text())\n\ndef add_domain_features(frame, as_of):\n    out = frame.copy()\n    origin = pd.Timestamp(as_of)\n    if origin > pd.Timestamp(\'2026-06-30\'):\n        raise ValueError(\'Forecast origin exceeds competition information cutoff\')\n    d = pd.to_datetime(out.date); sid = out.station_id; hour = out.hour\n    dow = d.dt.dayofweek; month = d.dt.month\n    scheduled = np.zeros(len(out), bool); covered = np.zeros(len(out), bool)\n    for record in schedules()[\'kbo\']:\n        if pd.Timestamp(record[\'known_since\']) <= origin:\n            scheduled |= d.isin(pd.to_datetime(record[\'home_dates\'])).to_numpy()\n            covered |= d.between(record[\'coverage_start\'], record[\'coverage_end\']).to_numpy()\n    munhak = sid.eq(3127)\n    out[\'munhak_kbo_scheduled\'] = munhak & scheduled\n    out[\'munhak_schedule_known\'] = munhak & covered\n    out[\'munhak_inseason_unannounced\'] = munhak & month.between(3, 10) & ~covered\n    holiday = out.is_holiday.astype(bool)\n    night = ((dow < 5) & ~holiday) | (month.isin([7, 8]) & dow.eq(5))\n    out[\'munhak_exit_peak_proxy\'] = np.where(munhak & scheduled,\n        np.where(night & hour.between(20, 22), 2,\n                 np.where(~night & hour.between(17, 21), 1, 0)), 0)\n    for kind, start_hour in [(\'pentaport\', 18), (\'beer\', 19)]:\n        active = np.zeros(len(out), bool)\n        for record in schedules()[\'festivals\']:\n            if record[\'kind\'] == kind and pd.Timestamp(record[\'known_since\']) <= origin:\n                active |= d.between(record[\'start\'], record[\'end\']).to_numpy()\n        out[f\'songdo_{kind}_scheduled\'] = sid.eq(3139) & active\n        out[f\'songdo_{kind}_evening_proxy\'] = sid.eq(3139) & active & hour.ge(start_hour)\n    # Lunar flags already pass the holiday calendar\'s publication filter.\n    out[\'memorial_park_lunar_peak\'] = np.where(sid.eq(3122) & hour.between(9, 17),\n        np.where(out.lunar_day_0.eq(1), 2, np.where((out.lunar_m1 + out.lunar_p1) > 0, 1, 0)), 0)\n    md = d.dt.month * 100 + d.dt.day\n    university_break = md.between(622, 831) | (md >= 1222) | (md <= 228)\n    highschool_break = md.between(720, 816) | md.between(101, 228)\n    weekday = (dow < 5) & ~holiday\n    out[\'university_semester_proxy\'] = sid.isin([3136, 3133, 3115]) & ~university_break & weekday\n    out[\'highschool_dismissal_proxy\'] = sid.eq(3208) & ~highschool_break & weekday & hour.between(15, 16)\n    out[\'is_external_transfer\'] = sid.isin([3110, 3207, 3120, 3218, 3130])\n    return out\n', 'src/domain_schedule_v3.json': '{\n  "kbo": [],\n  "festivals": [\n    {\n      "kind": "pentaport",\n      "start": "2025-08-01",\n      "end": "2025-08-03",\n      "known_since": "2025-07-30",\n      "source_url": "https://www.incheon.go.kr/IC010205/view?repSeq=DOM_0000000012828449",\n      "license": "KOGL Type 1 (attribution)",\n      "retrieved_on": "2026-11-12",\n      "attribution": "인천광역시"\n    },\n    {\n      "kind": "pentaport",\n      "start": "2026-07-31",\n      "end": "2026-08-02",\n      "known_since": "2026-05-31",\n      "source_url": "https://www.incheon.go.kr/moo/MOO030301/3074869",\n      "license": "KOGL Type 1 (attribution)",\n      "retrieved_on": "2026-11-12",\n      "attribution": "인천광역시"\n    }\n  ],\n  "exclusions": [\n    "KBO original schedule: dates verified; reuse authorization unconfirmed; excluded per user 2026-11-12",\n    "Beer festival: dated pre-cutoff event schedule and reuse conditions not both confirmed"\n  ]\n}\n', 'src/eda.py': '"""Reproducible, read-only SINCON CSV exploratory analysis.\n\nUsage: python src/eda.py --data-dir /path/to/sincon-2026 --out-dir reports\nRequires numpy and pandas; the compact SVG chart uses only the standard library.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport html\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\ndef native(value):\n    if isinstance(value, dict):\n        return {str(k): native(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [native(v) for v in value]\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (np.floating,)):\n        return float(value) if np.isfinite(value) else None\n    if isinstance(value, pd.Timestamp):\n        return value.strftime(\'%Y-%m-%d\')\n    if pd.isna(value):\n        return None\n    return value\n\n\ndef table(frame, limit=20):\n    f = frame.head(limit).copy()\n    rows = [\'| \' + \' | \'.join(map(str, f.columns)) + \' |\',\n            \'| \' + \' | \'.join([\'---\'] * len(f.columns)) + \' |\']\n    for record in f.itertuples(index=False, name=None):\n        vals = []\n        for v in record:\n            if isinstance(v, (float, np.floating)):\n                vals.append(f\'{v:,.4f}\' if abs(v) < 10 else f\'{v:,.2f}\')\n            else:\n                vals.append(str(native(v)).replace(\'|\', \'\\\\|\'))\n        rows.append(\'| \' + \' | \'.join(vals) + \' |\')\n    return \'\\n\'.join(rows)\n\n\ndef draw_chart(path, series, title, ylabel):\n    """Small dependency-free SVG with explicit axes and all source points."""\n    width, height = 1100, 340\n    left, top, plot_w, plot_h = 82, 55, 980, 220\n    values = np.asarray(series.values, dtype=float)\n    ymax = max(float(np.nanmax(values)) * 1.08, 1.0)\n    xs = np.linspace(left, left + plot_w, len(values))\n    ys = top + plot_h - values / ymax * plot_h\n    points = \' \'.join(f\'{x:.2f},{y:.2f}\' for x, y in zip(xs, ys))\n    pieces = [f\'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">\',\n              \'<rect width="100%" height="100%" fill="#fff"/>\',\n              \'<g font-family="Arial, sans-serif" fill="#172b4d">\',\n              f\'<text x="{left}" y="28" font-size="20" font-weight="bold">{html.escape(title)}</text>\',\n              f\'<text x="{left}" y="46" font-size="12">{html.escape(ylabel)}</text>\']\n    for i in range(5):\n        v = ymax * i / 4\n        y = top + plot_h - plot_h * i / 4\n        pieces += [f\'<line x1="{left}" y1="{y}" x2="{left+plot_w}" y2="{y}" stroke="#e3e8ef"/>\',\n                   f\'<text x="{left-10}" y="{y+4}" font-size="12" text-anchor="end">{v:,.0f}</text>\']\n    pieces.append(f\'<polyline points="{points}" fill="none" stroke="#007e87" stroke-width="1.7"/>\')\n    for i in sorted(set(np.linspace(0, len(values)-1, min(7, len(values))).astype(int))):\n        label = series.index[i]\n        if isinstance(label, pd.Timestamp):\n            label = label.strftime(\'%Y-%m-%d\')\n        pieces.append(f\'<text x="{xs[i]}" y="{top+plot_h+25}" font-size="12" text-anchor="middle">{html.escape(str(label))}</text>\')\n    pieces += [\'<text x="82" y="325" font-size="11" fill="#56657a">Source: supplied train.csv; no target values from test.csv are available.</text>\', \'</g></svg>\']\n    path.write_text(\'\\n\'.join(pieces), encoding=\'utf-8\')\n\n\ndef analyze(data_dir: Path, out: Path):\n    out.mkdir(parents=True, exist_ok=True)\n    files = {name: data_dir / f\'{name}.csv\' for name in [\'train\', \'test\', \'metaData\', \'sample_submission\']}\n    train = pd.read_csv(files[\'train\'], parse_dates=[\'date\'], dtype={\'station_id\': \'int32\', \'hour\': \'int8\', \'boardings\': \'int32\', \'alightings\': \'int32\'})\n    test = pd.read_csv(files[\'test\'], parse_dates=[\'date\'], dtype={\'station_id\': \'int32\', \'hour\': \'int8\'})\n    meta = pd.read_csv(files[\'metaData\'], parse_dates=[\'first_date\', \'last_date\'], dtype={\'station_id\': \'int32\'})\n    sample = pd.read_csv(files[\'sample_submission\'])\n    frames = {\'train\': train, \'test\': test, \'metaData\': meta, \'sample_submission\': sample}\n    report = {\'source_directory\': str(data_dir.resolve()), \'files\': {}}\n    for name, frame in frames.items():\n        report[\'files\'][name] = {\n            \'rows\': len(frame), \'columns\': list(frame.columns), \'column_count\': frame.shape[1],\n            \'sha256\': hashlib.sha256(files[name].read_bytes()).hexdigest(),\n            \'bytes\': files[name].stat().st_size,\n            \'missing_cells\': frame.isna().sum().to_dict(),\n            \'memory_bytes\': int(frame.memory_usage(deep=True).sum()),\n            \'dtypes\': frame.dtypes.astype(str).to_dict(),\n        }\n    key = [\'date\', \'station_id\', \'hour\']\n    train = train.sort_values([\'station_id\', \'date\', \'hour\']).reset_index(drop=True)\n    hours = sorted(train.hour.unique().tolist())\n    station_coverage = train.groupby(\'station_id\').agg(\n        first_date=(\'date\', \'min\'), last_date=(\'date\', \'max\'),\n        observed_dates=(\'date\', \'nunique\'), rows=(\'date\', \'size\'))\n    station_coverage[\'calendar_dates\'] = (station_coverage.last_date - station_coverage.first_date).dt.days + 1\n    station_coverage[\'expected_rows_within_observed_lifetime\'] = station_coverage.calendar_dates * len(hours)\n    station_coverage[\'missing_rows_within_observed_lifetime\'] = station_coverage.expected_rows_within_observed_lifetime - station_coverage.rows\n    station_coverage = station_coverage.reset_index().merge(meta[[\'station_id\', \'station_name\', \'canonical_line\']], on=\'station_id\', validate=\'one_to_one\')\n    station_coverage.to_csv(out / \'station_coverage.csv\', index=False)\n    actual_metadata = station_coverage.merge(meta[[\'station_id\', \'first_date\', \'last_date\']], on=\'station_id\', suffixes=(\'_observed\', \'_metadata\'))\n    metadata_matches = bool(((actual_metadata.first_date_observed == actual_metadata.first_date_metadata) & (actual_metadata.last_date_observed == actual_metadata.last_date_metadata)).all())\n    train_timestamp = train.date + pd.to_timedelta(train.hour.astype(int), unit=\'h\')\n    delta = train_timestamp.groupby(train.station_id).diff().dt.total_seconds().dropna() / 3600\n    expected_grid = train.date.nunique() * train.station_id.nunique() * len(hours)\n    report[\'coverage\'] = {\n        \'train_start\': train.date.min(), \'train_end\': train.date.max(), \'train_days\': train.date.nunique(),\n        \'test_start\': test.date.min(), \'test_end\': test.date.max(), \'test_days\': test.date.nunique(),\n        \'station_count\': train.station_id.nunique(), \'hours\': hours,\n        \'train_duplicate_keys\': int(train.duplicated(key).sum()), \'test_duplicate_keys\': int(test.duplicated(key).sum()),\n        \'train_test_key_overlap\': len(train[key].merge(test[key], on=key)),\n        \'test_id_duplicates\': int(test.example_id.duplicated().sum()),\n        \'sample_id_duplicates\': int(sample.example_id.duplicated().sum()),\n        \'new_test_stations\': sorted(set(test.station_id) - set(train.station_id)),\n        \'metadata_unique_station_ids\': bool(meta.station_id.is_unique),\n        \'metadata_dates_match_observed\': metadata_matches,\n        \'global_full_grid_rows\': expected_grid,\n        \'global_grid_absent_rows\': expected_grid - len(train),\n        \'within_observed_lifetime_missing_rows\': station_coverage.missing_rows_within_observed_lifetime.sum(),\n        \'station_start_date_counts\': station_coverage.first_date.value_counts().sort_index().to_dict(),\n        \'observed_timestamp_gaps_hours\': delta.value_counts().sort_index().to_dict(),\n        \'test_complete_grid\': bool(len(test) == test.date.nunique() * test.station_id.nunique() * test.hour.nunique() and not test.duplicated(key).any()),\n        \'canonical_line_station_counts\': meta.canonical_line.value_counts().sort_index().to_dict(),\n    }\n    sample_keys = sample[[\'example_id\']].merge(test, on=\'example_id\', how=\'left\', validate=\'one_to_one\', indicator=True)\n    uncovered = test.loc[~test.example_id.isin(sample.example_id)]\n    report[\'submission_mismatch\'] = {\n        \'test_rows\': len(test), \'sample_rows\': len(sample), \'difference\': len(test)-len(sample),\n        \'sample_ids_absent_from_test\': int((sample_keys[\'_merge\'] != \'both\').sum()),\n        \'sample_period_start\': sample_keys.date.min(), \'sample_period_end\': sample_keys.date.max(),\n        \'sample_days\': sample_keys.date.nunique(),\n        \'uncovered_start\': uncovered.date.min(), \'uncovered_end\': uncovered.date.max(),\n        \'sample_matches_test_prefix_order\': bool(sample.example_id.tolist() == test.example_id.head(len(sample)).tolist()),\n        \'sample_values_all_zero\': bool((sample.boardings == 0).all()),\n        \'resolution\': \'User-supplied official rules confirm 157194 rows for July through October. Use full test IDs in their original order; the local 117576-row sample is stale/inconsistent and is not the final row-count authority.\'\n    }\n    for target in [\'boardings\', \'alightings\']:\n        y = train[target]\n        report[target] = {\n            \'min\': y.min(), \'max\': y.max(), \'mean\': y.mean(), \'std\': y.std(), \'median\': y.median(),\n            \'skewness\': y.skew(), \'negative_rows\': int((y < 0).sum()), \'zero_rows\': int((y == 0).sum()),\n            \'zero_fraction\': (y == 0).mean(), \'quantiles\': y.quantile([.01, .05, .25, .5, .75, .9, .95, .99, .999]).to_dict(),\n        }\n    report[\'same_row_boardings_alightings_correlation\'] = train.boardings.corr(train.alightings)\n    stats = train.groupby(\'station_id\').boardings.agg([\'size\', \'mean\', \'median\', \'std\', \'min\', \'max\']).reset_index().merge(meta[[\'station_id\',\'station_name\',\'canonical_line\']], on=\'station_id\')\n    stats[\'zero_fraction\'] = train.groupby(\'station_id\').boardings.apply(lambda y: (y == 0).mean()).reindex(stats.station_id).to_numpy()\n    stats.to_csv(out / \'station_target_stats.csv\', index=False)\n    report[\'highest_mean_stations\'] = stats.nlargest(10, \'mean\').to_dict(\'records\')\n    report[\'lowest_mean_stations\'] = stats.nsmallest(10, \'mean\').to_dict(\'records\')\n    train[\'weekday\'] = train.date.dt.dayofweek.astype(\'int8\')\n    train[\'month\'] = train.date.dt.strftime(\'%Y-%m\')\n    weekday = train.groupby(\'weekday\').boardings.agg([\'mean\',\'median\',\'size\']).reset_index()\n    hourly = train.groupby(\'hour\').boardings.mean()\n    monthly = train.groupby(\'month\').agg(rows=(\'boardings\',\'size\'), mean=(\'boardings\',\'mean\'), total=(\'boardings\',\'sum\')).reset_index()\n    daily = train.groupby(\'date\').boardings.sum()\n    daily_mean = train.groupby(\'date\').boardings.mean()\n    report[\'weekday_profiles\'] = weekday.to_dict(\'records\')\n    report[\'hour_mean_boardings\'] = hourly.to_dict()\n    report[\'monthly_profiles\'] = monthly.to_dict(\'records\')\n    report[\'aggregate_daily_autocorrelation\'] = {lag: daily.corr(daily.shift(lag)) for lag in [1, 7, 14, 28]}\n    weekday.to_csv(out / \'weekday_profile.csv\', index=False)\n    monthly.to_csv(out / \'monthly_profile.csv\', index=False)\n    daily.rename(\'boardings_total\').to_csv(out / \'daily_boardings.csv\')\n    maxima = train.nlargest(12,\'boardings\')[key+[\'boardings\',\'alightings\']].merge(meta[[\'station_id\',\'station_name\',\'canonical_line\']],on=\'station_id\')\n    report[\'largest_single_slot_counts\'] = maxima.to_dict(\'records\')\n    # Past-only weekly reference used strictly for an EDA spike flag, never as\n    # a whole-data training feature. A 3x / 1000 threshold is heuristic.\n    group = train.groupby([\'station_id\',\'hour\',\'weekday\'], sort=False).boardings\n    past = group.transform(lambda y: y.shift(1).rolling(8, min_periods=4).median())\n    ratios = train.boardings / past.replace(0,np.nan)\n    spikes = train.loc[(ratios > 3) & (train.boardings >= 1000), key+[\'boardings\']].copy()\n    spikes[\'past_8_same_weekday_median\'] = past.loc[spikes.index]\n    spikes[\'ratio\'] = ratios.loc[spikes.index]\n    spikes = spikes.merge(meta[[\'station_id\',\'station_name\']],on=\'station_id\').sort_values(\'ratio\',ascending=False)\n    spikes.to_csv(out / \'spike_candidates.csv\',index=False)\n    report[\'spike_heuristic\'] = {\'definition\':\'boardings >= 1000 and > 3x median of up to eight prior same-weekday observations at this station/hour; at least four prior observations\', \'rows\':len(spikes),\'top_rows\':spikes.head(15).to_dict(\'records\')}\n    # Synchrony is descriptive. A shared calendar/trend can explain correlations;\n    # no coordinates or physical adjacency are present in the input.\n    panel = train.pivot_table(index=\'date\',columns=\'station_id\',values=\'boardings\',aggfunc=\'sum\')\n    corr = panel.corr(min_periods=100)\n    residual = panel - panel.groupby(panel.index.dayofweek).transform(\'mean\')\n    adjusted = residual.corr(min_periods=100)\n    pairs=[]\n    names=meta.set_index(\'station_id\').station_name.to_dict()\n    lines=meta.set_index(\'station_id\').canonical_line.to_dict()\n    for a_i,a in enumerate(corr.columns):\n        for b in corr.columns[a_i+1:]:\n            pairs.append({\'station_a\':int(a),\'station_b\':int(b),\'name_a\':names[a],\'name_b\':names[b], \'same_canonical_line\':bool(lines[a]==lines[b]), \'raw_daily_correlation\':corr.loc[a,b], \'weekday_demeaned_correlation\':adjusted.loc[a,b]})\n    pair_df=pd.DataFrame(pairs).sort_values(\'raw_daily_correlation\',ascending=False)\n    pair_df.to_csv(out / \'station_correlations.csv\',index=False)\n    lag_rows=[]\n    for row in pair_df.head(10).itertuples(index=False):\n        a,b=row.station_a,row.station_b\n        lag_corr={str(lag):residual[a].corr(residual[b].shift(lag)) for lag in range(-3,4)}\n        lag_rows.append({\'station_a\':a,\'station_b\':b,\'name_a\':names[a],\'name_b\':names[b],\'daily_lag_correlations\':lag_corr, \'highest_correlation_lag_days\':int(max(lag_corr,key=lambda k: lag_corr[k]))})\n    report[\'spatial_and_synchrony\']={\n        \'coordinates_available\':False,\'adjacency_available\':False,\'canonical_lines\': sorted(meta.canonical_line.unique().tolist()),\n        \'pair_count\':len(pair_df), \'raw_pair_correlation_quantiles\':pair_df.raw_daily_correlation.quantile([0,.25,.5,.75,.9,1]).to_dict(),\n        \'weekday_demeaned_pair_correlation_quantiles\':pair_df.weekday_demeaned_correlation.quantile([0,.25,.5,.75,.9,1]).to_dict(),\n        \'top_pairs\':pair_df.head(10).to_dict(\'records\'), \'lag_diagnostics_top_10_pairs\':lag_rows,\n        \'interpretation\':\'Daily synchrony and coarse day-lag association only. Not a physical travel-time/propagation estimate. Station ID numeric proximity does not establish adjacency.\'}\n    report[\'feature_availability\']={\n        \'future_known_from_files\':[\'date\',\'station_id\',\'hour\',\'canonical_line\'],\n        \'train_only_observations\':[\'boardings\',\'alightings\'],\n        \'absent_from_files\':[\'weather\',\'holiday indicator\',\'discount/event indicator\',\'station coordinates\',\'physical adjacency\',\'service disruption labels\'],\n        \'calendar_derivable\':[\'weekday\',\'month\',\'day of year\',\'cyclic hour/weekday/day of year\'],\n        \'leakage_notes\':[\'No contemporaneous test alightings; use only values observed by forecast origin.\', \'Metadata last_date is a dataset coverage boundary, not a verified station closing date.\', \'No holdout target data may enter rolling means, scale normalization, correlations used as features or fitted calibration.\']}\n    report[\'official_task\'] = {\n        \'evidence\': \'User-supplied official Overview/Data/Rules; reports/sources/rules_user_provided.txt\',\n        \'metric\': \'RMSLE\', \'horizon_days\': 123, \'required_submission_rows\': 157194,\n        \'training_loss\': \'MSE on log1p(boardings); invert with expm1 then clip at zero\',\n        \'external_data_cutoff\': \'2026-06-30\',\n        \'holidays_allowed\': True, \'external_sources_and_license_record_required\': True,\n    }\n    (out/\'eda.json\').write_text(json.dumps(native(report),ensure_ascii=False,indent=2,allow_nan=False),encoding=\'utf-8\')\n    draw_chart(out/\'daily_boardings.svg\',daily,\'Daily boardings across observed stations\',\'Boardings / day (68 stations initially, 71 from 2025-06-28)\')\n    draw_chart(out/\'daily_mean_boardings.svg\',daily_mean,\'Daily mean boardings per station-hour\',\'Mean boardings / observed station-hour\')\n    draw_chart(out/\'hourly_boardings.svg\',hourly,\'Average boardings by hour\',\'Mean boardings / station-hour; all training dates\')\n    r=report\n    introduction=\'\'\'# SINCON 2026 데이터 EDA\n\n이 보고서는 제공된 CSV의 실제 값에서 재현한 관찰 결과다. 사용자가 제공한 [공식 Overview·Data·Rules 사본](sources/rules_user_provided.txt)에 따라 과제는 **인천 지하철 71개 역의 시간대별 승차 인원(boardings)을 2026-06-30 기준으로 7~10월 123일간 한 번에 예측하는 패널 시계열 회귀**다. 공식 지표는 **RMSLE**, 제출 행 수는 **157,194행**이다. `alightings`는 학습 데이터에만 있는 하차 인원 보조 변수다.\n\n## 1. 파일과 예측 범위\n\n\'\'\'\n    summary=pd.DataFrame([{\'파일\':f\'{name}.csv\',\'행\':len(df),\'열\':df.shape[1],\'셀 결측\':int(df.isna().sum().sum())} for name,df in frames.items()])\n    md=introduction+table(summary)+f\'\'\'\n\n- 학습: **2025-01-01~2026-06-30, {train.date.nunique()}일**.\n- 테스트: **2026-07-01~2026-10-31, {test.date.nunique()}일**, 71역 × 18시간 × 123일.\n- 제출 예시: 테스트 앞 **92일(2026-07-01~09-30)**의 ID 117,576개와 순서까지 일치한다.\n- 테스트에는 **10월 39,618행**이 추가로 있다. sample ID의 test 외 항목은 없고, 모든 ID와 (date, station_id, hour) 키는 고유하다.\n- **공식 규정은 157,194행을 요구하므로 전체 test의 123일을 예측한다.** 로컬 sample의 117,576행은 구버전/불일치로 기록하고, 최종 제출의 행 수는 공식 규정과 test ID를 기준으로 검수한다. 원본 sample은 수정하지 않는다.\n- `sample_submission.boardings`의 0은 양식 placeholder이며 실제 테스트 타겟이 아니다.\n\n## 2. 규칙성·결측·역별 관측 기간\n\n- 71개 역, 매일 **06~23시 18개 슬롯**이 제공된다. 각 역의 최초~최종 관측일 내부에는 누락 행이 **{int(station_coverage.missing_rows_within_observed_lifetime.sum())}개**, 중복 키도 0개다.\n- timestamp 차이는 **1시간**(같은 날) 또는 **7시간**(23시→다음 날 06시)뿐이다. 00~05시가 파일에 없다는 뜻이며 실제 운휴를 확인한 것은 아니다. 그 시간에 0을 채워 24시간 자료로 바꾸지 않는다.\n- 전체 71역 × 546일 × 18시간 격자에서 빈 {expected_grid-len(train):,}행은 3개 역의 첫 관측일 이전 구간이다. 이 값들은 결측 승객 수 또는 0명으로 취급하지 않는다.\n- 68개 역은 2025-01-01, 3개 역(검단호수공원·신검단중앙·아라)은 **2025-06-28**부터 관측된다. 테스트에는 완전히 새로운 station_id가 없다.\n- 메타데이터 `first_date`, `last_date`는 실제 관측 범위와 일치한다. 특히 `last_date=2026-06-30`은 데이터 종료일이지 폐역일이 아니다.\n- 노선별 역 수: {json.dumps(native(r[\'coverage\'][\'canonical_line_station_counts\']),ensure_ascii=False)}. 같은 역명이라도 다른 station_id를 자동 합치지 않는다.\n\n## 3. 타겟 분포와 급등\n\n\'\'\'\n    ys=r[\'boardings\']\n    md+=table(pd.DataFrame([{\'통계\':\'평균\',\'값\':ys[\'mean\']},{\'통계\':\'중앙값\',\'값\':ys[\'median\']},{\'통계\':\'표준편차\',\'값\':ys[\'std\']},{\'통계\':\'최솟값\',\'값\':ys[\'min\']},{\'통계\':\'최댓값\',\'값\':ys[\'max\']},{\'통계\':\'왜도\',\'값\':ys[\'skewness\']},{\'통계\':\'0인 행 수\',\'값\':ys[\'zero_rows\']},{\'통계\':\'0 비율\',\'값\':ys[\'zero_fraction\']}]))\n    md+=f\'\'\'\n\n우측 꼬리가 길다. 99% 분위수는 {ys[\'quantiles\'][.99]:,.0f}, 99.9% 분위수는 {ys[\'quantiles\'][.999]:,.2f}다. 음수는 없다. 공식 RMSLE에 맞춰 **log1p(boardings)의 MSE**를 기본 학습 손실로 사용한다. 예측은 expm1로 복원한 뒤 음수를 0으로 자른다. 원단위 MAE·RMSE·편향도 진단용으로 병행한다. 양수 카운트라는 사실만으로 Poisson 분포를 가정하지 않는다.\n\n최대 관측값:\n\n\'\'\'+table(maxima,12)+f\'\'\'\n\n동일 역·시간·요일의 과거 4~8주 중앙값 대비 3배 초과이면서 승차 1,000명 이상인 휴리스틱 급등 후보는 **{len(spikes):,}행**이다. 이벤트·오류·수송 변화 여부는 원자료 설명 없이는 확정할 수 없다. 무조건 삭제하거나 상한을 자르지 않는다. 상세 목록은 `spike_candidates.csv`에 있다.\n\n\'\'\'+table(spikes.head(10),10)+\'\'\'\n\n## 4. 달력 패턴과 시점 변화\n\n요일 0=월요일, 6=일요일이다. 다음 수치는 역·시간당 평균이며 총량과 구별한다.\n\n\'\'\'+table(weekday)+\'\'\'\n\n월별 표는 그 달에 관측된 행의 평균이다. 3개 역 추가에 따른 구성 변화와 달력 효과를 함께 고려해야 한다.\n\n\'\'\'+table(monthly,30)+f\'\'\'\n\n전체 일별 승차 총량의 자기상관: {json.dumps(native(r[\'aggregate_daily_autocorrelation\']),ensure_ascii=False)}. 공통 요일과 추세가 포함된 EDA 상관이며 모델 잔차 ACF가 아니다. CV 후에는 역·시간별 잔차를 날짜순으로 다시 평가한다.\n\n![일별 총승차](daily_boardings.svg)\n\n![역·시간당 일별 평균](daily_mean_boardings.svg)\n\n![시간대별 평균 승차](hourly_boardings.svg)\n\n## 5. 역 규모와 다개체 연결성\n\n평균 승차가 큰 역:\n\n\'\'\'+table(stats.nlargest(10,\'mean\')[[\'station_id\',\'station_name\',\'canonical_line\',\'mean\',\'median\',\'max\']])+\'\'\'\n\n제공 메타데이터에는 노선은 있으나 좌표·역 간 거리·연결 목록이 없다. ID의 숫자 차이로 거리나 인접 관계를 만들지 않는다. 같은 노선 구분과 역별 이력만 즉시 사용할 수 있다.\n\n일별 총량의 역 간 상관 상위쌍(최소 동시 관측 100일):\n\n\'\'\'+table(pair_df.head(10),10)+f\'\'\'\n\n- 전체 {len(pair_df):,}개 역 쌍의 원자료 상관 중앙값: {pair_df.raw_daily_correlation.median():.4f}.\n- 각 역의 요일 평균을 제거한 상관 중앙값: {pair_df.weekday_demeaned_correlation.median():.4f}.\n- 상위 10쌍의 ±3일 지연 상관은 `eda.json`에 기록했다. 이는 날짜 단위의 동조 점검이다. 1시간보다 짧은 열차 이동/승객 전파를 검증하지 못하며 인과적 연결 또는 물리적 전파 지연의 근거가 아니다.\n- 전체 학습 구간으로 만든 상관은 EDA 전용이다. 모델의 그룹/이웃 특징으로 쓰려면 각 CV origin 이전 이력에서만 다시 계산한다.\n\n## 6. 입력 가용성과 누수 방지\n\n| 구분 | 변수/처리 |\n| --- | --- |\n| 미래에도 주어짐 | date, station_id, hour, canonical_line |\n| 파일에서 직접 파생 가능 | 요일·주말·월·연중일·시간 및 달력 주기 sin/cos |\n| 학습에만 있음 | boardings, alightings |\n| 없음 | 날씨, 공휴일 라벨, 할인·행사, 운행 장애, 좌표, 물리적 인접 정보 |\n| 규정상 허용된 외부 입력 | 2026-06-30 이전 공개되고 비상업 이용이 허용된 공휴일·일정 등. 출처·조회일·라이선스 기록 필요 |\n\n공휴일은 `src/holiday_calendar.py`의 검증된 정적 달력에서 생성한다. 공식 출처와 공개시점은 [holiday_sources.md](holiday_sources.md)에 기록했다. 원본 CSV에는 공휴일이 없으며 외부 특징으로 추가한 것이다. 임시공휴일과 신규 지정일은 각 CV 예측 기준일에 알려진 정보만 활성화한다. 2026-05-01 노동절과 2026-07-17 제헌절을 포함하고, 2026-09-28은 공휴일로 표시하지 않는다.\n\n같은 행의 승차·하차 상관은 {r[\'same_row_boardings_alightings_correlation\']:.4f}다. **미래 alightings는 test에 없으므로 동시점 하차 값을 입력하면 누수/추론 불일치다.** origin 이전의 요약·지연 하차만 쓸 수 있다. 123일 전체를 한 번에 예측할 때 day 8의 실제 lag7처럼 origin 이후 관측을 참조해서는 안 된다. 직접 다중 horizon 특징이나 origin에 고정한 계절/이동 통계가 필요하다.\n\n## 7. 재현 및 보존\n\n```bash\npython src/eda.py --data-dir data/raw --out-dir reports\n```\n\n이 스크립트는 원본을 수정하지 않는다. `eda.json`에 입력 SHA-256, 자료형, 정확한 수치와 정의가 있고, 보조 CSV에 역별 범위·분포·상관·급등 후보가 있다. 그림은 집계값을 그대로 사용한다. 파일 크기는 약 24 MB이므로 이 단계의 원본만으로 29 GB RAM이 부족하지 않다. 다중 origin × horizon × 파생변수를 확장하면 메모리가 증가하므로 float32·정수 다운캐스팅, origin 배치 생성, 메모리 사용량 계측이 필요하다.\n\'\'\'\n    (out/\'EDA.md\').write_text(md,encoding=\'utf-8\')\n    print(json.dumps(native({\'train_rows\':len(train),\'test_rows\':len(test),\'sample_rows\':len(sample),\'missing_rows\':station_coverage.missing_rows_within_observed_lifetime.sum(),\'spikes\':len(spikes),\'out_dir\':str(out.resolve())}),ensure_ascii=False))\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--data-dir\', type=Path, default=Path(__file__).resolve().parents[1]/\'data\'/\'raw\')\n    parser.add_argument(\'--out-dir\', type=Path, default=Path(__file__).resolve().parents[1]/\'reports\')\n    args=parser.parse_args()\n    analyze(args.data_dir,args.out_dir)\n', 'src/ensemble.py': '"""Development-OOF convex blends: primary-constrained log-MSE or legacy greedy."""\nimport numpy as np\nfrom .ensemble_v2 import convex_weights, evaluate_ensemble\nimport pandas as pd\nfrom .metrics import score\n\ndef fit_ensemble(y,predictions,metric=\'rmsle\',correlation_cutoff=0.95,policy=None):\n    if policy and policy.get(\'method\')==\'primary_constrained_log_mse\':\n        if metric!=\'rmsle\':raise ValueError(\'Constrained log-MSE blending requires RMSLE\')\n        return fit_primary_ensemble(y,predictions,policy)\n    if policy:raise ValueError(f\'Unknown ensemble policy: {policy}\')\n    names=list(predictions)\n    scores={n:score(y,predictions[n],metric) for n in names}\n    space={n:np.log1p(predictions[n]) if metric==\'rmsle\' else np.asarray(predictions[n]) for n in names}\n    target=np.log1p(y) if metric==\'rmsle\' else y\n    invert=lambda values:np.maximum(0,np.expm1(values)) if metric==\'rmsle\' else np.maximum(0,values)\n    residual=pd.DataFrame({n:space[n]-target for n in names})\n    corr=residual.corr()\n    ranked=sorted(names,key=scores.get)\n    retained=[];excluded={}\n    for name in ranked:\n        redundant=[other for other in retained if np.array_equal(space[name],space[other]) or corr.loc[name,other]>correlation_cutoff]\n        if redundant:excluded[name]={\'reason\':\'residual_correlation_above_threshold\',\'with\':redundant}\n        else:retained.append(name)\n    weights={n:0.0 for n in retained};weights[retained[0]]=1.0\n    blend=space[retained[0]].copy();best=score(y,invert(blend),metric)\n    for step in [0.25,0.1,0.05,0.02,0.01]:\n        for _ in range(40):\n            options=[]\n            for name in retained:\n                proposed=(1-step)*blend+step*space[name]\n                options.append((score(y,invert(proposed),metric),name,proposed))\n            trial,name,proposed=min(options,key=lambda item:item[0])\n            if trial>=best-1e-8:break\n            weights={n:w*(1-step) for n,w in weights.items()};weights[name]+=step\n            blend=proposed;best=trial\n    # A scalar adjustment is estimated only on development OOF, audit is untouched.\n    bias=float(np.mean(target-blend)) if metric==\'rmsle\' else float(np.median(y-blend)) if metric==\'mae\' else 0.0\n    corrected=invert(blend+bias)\n    if score(y,corrected,metric)>=best:bias=0.0\n    return {\'weights\':weights,\'bias\':bias,\'bias_space\':\'log1p\' if metric==\'rmsle\' else \'raw\',\'blend_space\':\'log1p\' if metric==\'rmsle\' else \'raw\',\'retained\':retained,\'excluded\':excluded,\'individual_scores\':scores,\'development_score\':score(y,invert(blend+bias),metric),\'metric\':metric},corr\n\ndef fit_primary_ensemble(y,predictions,policy):\n    """Minimize development log-MSE with a fixed minimum primary-model share.\n\n    The caller supplies development OOF only. Correlation filtering and intercept\n    calibration are disabled; the primary model cannot be silently removed.\n    """\n    from scipy.optimize import minimize\n    names=list(predictions)\n    primary=policy[\'primary_model\']\n    minimum=float(policy[\'primary_min_weight\'])\n    if primary not in names:raise ValueError(\'Primary model missing from predictions\')\n    if not .5<=minimum<=1:raise ValueError(\'Primary minimum weight must be in [0.5, 1]\')\n    if policy.get(\'bias_correction\',False):raise ValueError(\'This policy disables bias correction\')\n    truth=np.asarray(y,dtype=np.float64)\n    if truth.ndim!=1 or len(truth)==0 or not np.isfinite(truth).all() or np.any(truth<0):\n        raise ValueError(\'Expected finite nonnegative one-dimensional targets\')\n    arrays=[np.asarray(predictions[name],dtype=np.float64) for name in names]\n    if any(p.shape!=truth.shape or not np.isfinite(p).all() or np.any(p<0) for p in arrays):\n        raise ValueError(\'Predictions must match targets and be finite and nonnegative\')\n    residual=np.column_stack([np.log1p(p)-np.log1p(truth) for p in arrays])\n    gram=residual.T@residual/len(truth)\n    ix=names.index(primary)\n    lower=np.zeros(len(names));lower[ix]=minimum\n    start=lower+(1-minimum)/len(names)\n    result=minimize(lambda w:float(w@gram@w),start,jac=lambda w:2*gram@w,\n                    method=\'SLSQP\',bounds=[(float(lo),1.) for lo in lower],\n                    constraints=[{\'type\':\'eq\',\'fun\':lambda w:w.sum()-1.,\n                                  \'jac\':lambda w:np.ones_like(w)}],\n                    options={\'ftol\':1e-13,\'maxiter\':500})\n    if not result.success:raise RuntimeError(f\'Ensemble optimization failed: {result.message}\')\n    w=np.asarray(result.x)\n    if np.any(w<lower-1e-10) or abs(w.sum()-1)>1e-10:\n        raise RuntimeError(\'Ensemble optimizer violated convex constraints\')\n    # SLSQP may return a boundary value a few ulps below its lower bound.\n    w=np.maximum(w,lower)\n    w[ix]+=1-w.sum()\n    scores={name:score(truth,p,\'rmsle\') for name,p in zip(names,arrays)}\n    plan={\'weights\':dict(zip(names,map(float,w))),\'bias\':0.,\'bias_space\':\'log1p\',\n          \'blend_space\':\'log1p\',\'retained\':names,\'excluded\':{},\n          \'individual_scores\':scores,\'metric\':\'rmsle\',\'policy\':dict(policy),\n          \'optimizer\':\'SLSQP: convex quadratic log-MSE\',\n          \'correlation_filter_applied\':False,\'bias_correction_applied\':False}\n    plan[\'development_score\']=score(truth,predict_ensemble(predictions,plan),\'rmsle\')\n    return plan,pd.DataFrame(residual,columns=names).corr()\n\ndef predict_ensemble(predictions,plan):\n    active=[(name,weight) for name,weight in plan[\'weights\'].items() if weight>0]\n    if len(active)==1 and active[0][1]==1.0 and plan[\'bias\']==0.0:\n        # Preserve exact standalone predictions, avoiding log/expm1 roundoff.\n        return np.maximum(0,np.asarray(predictions[active[0][0]],dtype=np.float64))\n    log_space=plan.get(\'blend_space\')==\'log1p\'\n    pred=sum(weight*(np.log1p(predictions[name]) if log_space else np.asarray(predictions[name])) for name,weight in plan[\'weights\'].items() if weight>0)\n    return np.maximum(0,np.expm1(pred+plan[\'bias\']) if log_space else pred+plan[\'bias\'])\n', 'src/ensemble_v2.py': '"""Convex log-space OOF blending without an intercept, including LOFO checks."""\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import minimize\n\ndef row_weights(folds,fold_weights,exclude=None):\n    weights=np.zeros(len(folds),dtype=\'float64\')\n    for fold,weight in enumerate(fold_weights,1):\n        if fold==exclude:continue\n        mask=folds==fold\n        if mask.any():weights[mask]=weight/mask.sum()\n    return weights/weights.sum()\n\ndef convex_weights(truth,predictions,weights=None):\n    y=np.asarray(truth,dtype=\'float64\');p=np.asarray(predictions,dtype=\'float64\')\n    if p.ndim!=2 or len(y)!=len(p) or not np.isfinite(p).all() or not np.isfinite(y).all():raise ValueError(\'Invalid finite log prediction matrix\')\n    weights=np.full(len(y),1/len(y)) if weights is None else np.asarray(weights)/np.sum(weights)\n    residual=p-y[:,None];gram=residual.T@(weights[:,None]*residual)\n    n=p.shape[1]\n    result=minimize(lambda w:float(w@gram@w),np.full(n,1/n),jac=lambda w:2*gram@w,method=\'SLSQP\',bounds=[(0,1)]*n,constraints=[{\'type\':\'eq\',\'fun\':lambda w:w.sum()-1,\'jac\':lambda w:np.ones(n)}],options={\'ftol\':1e-13,\'maxiter\':1000})\n    if not result.success:raise RuntimeError(result.message)\n    w=np.maximum(0,result.x);w/=w.sum()\n    return w\n\ndef evaluate_ensemble(truth,predictions,folds,names,fold_weights):\n    y=np.asarray(truth);p=np.asarray(predictions);folds=np.asarray(folds)\n    weights=row_weights(folds,fold_weights);w=convex_weights(y,p,weights)\n    per_model={name:{\'oof_rmsle\':float(np.sqrt(np.mean((p[:,i]-y)**2))), \'weighted_rmsle\':float(np.sqrt(np.sum(weights*(p[:,i]-y)**2))), \'fold34_rmsle\':float(np.sqrt(np.mean((p[folds>=3,i]-y[folds>=3])**2)))} for i,name in enumerate(names)}\n    lofo=np.zeros(len(y));reference=np.zeros(len(y));checks=[]\n    for fold in range(1,5):\n        fitweights=row_weights(folds,fold_weights,exclude=fold)\n        wf=convex_weights(y,p,fitweights);ix=folds==fold\n        best=int(np.argmin(np.sum(fitweights[:,None]*(p-y[:,None])**2,axis=0)))\n        lofo[ix]=p[ix]@wf;reference[ix]=p[ix,best]\n        checks.append({\'held_out_fold\':fold,\'weights\':dict(zip(names,wf.tolist())),\'training_selected_best_model\':names[best], \'ensemble_rmsle\':float(np.sqrt(np.mean((lofo[ix]-y[ix])**2))), \'selected_single_rmsle\':float(np.sqrt(np.mean((reference[ix]-y[ix])**2)))})\n    pred=p@w\n    report={\'method\':\'SLSQP convex mean in log1p space; no intercept\',\'weights\':dict(zip(names,w.tolist())),\'models\':per_model,\'ensemble\':{\'oof_rmsle\':float(np.sqrt(np.mean((pred-y)**2))),\'weighted_rmsle\':float(np.sqrt(np.sum(weights*(pred-y)**2))),\'fold34_rmsle\':float(np.sqrt(np.mean((pred[folds>=3]-y[folds>=3])**2)))},\'lofo\':checks,\'lofo_weighted_rmsle\':float(np.sqrt(np.sum(weights*(lofo-y)**2))),\'lofo_selected_single_weighted_rmsle\':float(np.sqrt(np.sum(weights*(reference-y)**2))),\'evaluation_limit\':\'Validation dates overlap between folds; LOFO is a sensitivity check, not an independent future holdout. Base models use their fold validation for early stopping.\'}\n    corr=pd.DataFrame(y[:,None]-p,columns=names).corr()\n    return pred,lofo,report,corr\n', 'src/ensemble_v3.py': '"""Recent-regime convex blend, with all-fold and purged sensitivity checks."""\nimport numpy as np\nimport pandas as pd\nfrom .ensemble_v2 import convex_weights, row_weights, evaluate_ensemble as evaluate_v2\n\ndef rmsle_log(y, p, weights=None):\n    return float(np.sqrt(np.average((np.asarray(y)-np.asarray(p))**2, weights=weights)))\n\ndef evaluate_ensemble(truth, predictions, folds, names, fold_weights, dates=None):\n    y=np.asarray(truth); p=np.asarray(predictions); folds=np.asarray(folds)\n    all_pred, lofo, report, corr = evaluate_v2(y,p,folds,names,fold_weights)\n    recent=folds>=3; w=convex_weights(y[recent],p[recent]); pred=p@w\n    all_comparison={\'weights\':report[\'weights\'],\'ensemble\':report[\'ensemble\']}\n    report[\'all_fold_comparison\']=all_comparison\n    report[\'weights\']=dict(zip(names,w.tolist()))\n    report[\'selection_rows\']=int(recent.sum())\n    report[\'selection_scope\']=\'Fold 3 + Fold 4 pooled OOF, equal row weights\'\n    report[\'ensemble\']={\n        \'oof_rmsle\':rmsle_log(y,pred),\n        \'weighted_rmsle\':rmsle_log(y,pred,row_weights(folds,fold_weights)),\n        \'fold34_rmsle\':rmsle_log(y[recent],pred[recent]),\n    }\n    report[\'recent_cross_checks\']=[]\n    for source,target in [(3,4),(4,3)]:\n        fit=folds==source; test=folds==target\n        # Also report strict target-date exclusion because validation windows overlap.\n        for purge in [False,True]:\n            fit_mask=fit.copy()\n            if purge:\n                if dates is None:continue\n                date_array=np.asarray(dates,dtype=\'datetime64[D]\')\n                fit_mask &= ~np.isin(date_array,date_array[test])\n            if not fit_mask.any():continue\n            wf=convex_weights(y[fit_mask],p[fit_mask])\n            best=int(np.argmin(np.mean((p[fit_mask]-y[fit_mask,None])**2,axis=0)))\n            report[\'recent_cross_checks\'].append({\n                \'fit_fold\':source,\'test_fold\':target,\'purged_overlapping_target_dates\':purge,\n                \'fit_rows\':int(fit_mask.sum()),\'weights\':dict(zip(names,wf.tolist())),\n                \'ensemble_rmsle\':rmsle_log(y[test],p[test]@wf),\n                \'training_selected_single\':names[best],\n                \'single_rmsle\':rmsle_log(y[test],p[test,best]),\n                \'chronological\':source<target,\n            })\n    # The fitting objective includes every single-model vertex.\n    best_single=min(report[\'models\'][n][\'fold34_rmsle\'] for n in names)\n    if report[\'ensemble\'][\'fold34_rmsle\'] > best_single + 1e-7:\n        raise RuntimeError(\'Convex optimizer is worse than a feasible single model\')\n    report[\'residual_correlations_fold34\']=pd.DataFrame(y[recent,None]-p[recent],columns=names).corr().to_dict()\n    report[\'evaluation_limit\'] += \' Recent-fold optimized scores reuse labels for weights; purged weight checks still reuse fold-selected early stopping and feature-design decisions. Fold 3/4 are not season-identical to July-October.\'\n    return pred,lofo,report,corr\n', 'src/evaluate_postprocessing.py': '"""Choose a postprocessor on temporal development OOF and evaluate it once later."""\nfrom __future__ import annotations\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom .features import load_data,make_features\nfrom .ensemble import predict_ensemble\nfrom .postprocessing import build_context,candidate_specs,fit_plan,apply_plan\nfrom .metrics import regression_metrics\nfrom .pipeline import save_json,export_predictions,diagnostics\n\nROOT=Path(__file__).resolve().parents[1]\n\ndef rmsle(actual,pred):\n    return float(np.sqrt(np.mean((np.log1p(pred)-np.log1p(actual))**2)))\n\ndef run(args):\n    started=time.perf_counter();out=args.output;base=args.base_output\n    if (out/\'model_bundle.joblib\').exists():raise FileExistsError(\'Choose a new output directory to preserve the previous run\')\n    out.mkdir(parents=True,exist_ok=True)\n    data=load_data(args.data_dir);train=data[\'train\'];cfg=json.loads((base/\'config_resolved.json\').read_text())\n    origins=cfg[\'fold_origins\']\n    frames=[];contexts=[];preds=[];truth=[]\n    # Only these two OOF files are loaded until the selection plan is frozen.\n    for fold in [1,2]:\n        f=pd.read_csv(base/f\'fold_{fold}/oof.csv.gz\',parse_dates=[\'date\'])\n        f=f.loc[f.known_at_origin].reset_index(drop=True)\n        context=build_context(train,f,origins[fold-1])\n        frames.append(f);contexts.append(context)\n        preds.append(f.lightgbm_large.to_numpy());truth.append(f.boardings.to_numpy())\n    raw=[rmsle(y,p) for y,p in zip(truth,preds)]\n    design={\'calibration_fit_fold\':1,\'temporal_selection_fold\':2,\'fit_scope\':\'known stations only\',\n            \'final_calibration_refit_folds\':[1,2],\'audit_selection_uses_labels\':False,\n            \'max_historical_anchor_weight\':.2,\'anchor_windows_days\':[28,56,112],\n            \'holiday_gate\':\'known-as-of-origin holidays remain exactly as base model predictions\',\n            \'candidate_count\':len(candidate_specs()),\'decision_rule\':\'minimum forward fold-2 RMSLE among eligible candidates that do not worsen fold1; identity included\',\n            \'caps_note\':\'95th-percentile caps are diagnostic-only: no physical capacity bound is provided\',\n            \'context\':\'Development folds were already used for LightGBM tuning. Fold3 was analyzed earlier; all results are conditional reused-validation evidence.\'}\n    save_json(out/\'selection_design.json\',design)\n    results=[];plans={}\n    for spec in candidate_specs():\n        plan=fit_plan(contexts[0],preds[0],truth[0],spec)\n        p1=apply_plan(contexts[0],preds[0],plan);p2=apply_plan(contexts[1],preds[1],plan)\n        a,b=rmsle(truth[0],p1),rmsle(truth[1],p2)\n        allowed=spec[\'eligible\'] and a<=raw[0]+1e-12\n        results.append({\'name\':spec[\'name\'],\'development1_in_sample_rmsle\':a,\'development2_forward_rmsle\':b,\n                        \'change_dev1\':a-raw[0],\'change_dev2\':b-raw[1],\'eligible_for_selection\':bool(allowed),\n                        \'calibration\':spec[\'calibration\'][\'kind\'],\'calibration_scale\':spec[\'calibration\'][\'scale\'],\n                        \'anchor_days\':spec[\'anchor\'][\'days\'],\'anchor_weight\':spec[\'anchor\'][\'weight\'],\'cap_days\':spec[\'cap_days\']})\n        plans[spec[\'name\']]=plan\n    candidates=pd.DataFrame(results)\n    candidates.to_csv(out/\'candidate_development_scores.csv\',index=False)\n    chosen=candidates.loc[candidates.eligible_for_selection].sort_values([\'development2_forward_rmsle\',\'anchor_weight\',\'calibration_scale\']).iloc[0]\n    spec=plans[chosen[\'name\']][\'spec\']\n    pooled=pd.concat(contexts,ignore_index=True)\n    pooled_pred=np.concatenate(preds);pooled_y=np.concatenate(truth)\n    fold_ids=np.concatenate([np.full(len(contexts[i]),i+1) for i in range(2)])\n    plan=fit_plan(pooled,pooled_pred,pooled_y,spec,fold_ids)\n    plan[\'selection\']=design;plan[\'selected_forward_scores\']=chosen.to_dict()\n    plan[\'source_base_bundle_sha256\']=hashlib.sha256((base/\'model_bundle.joblib\').read_bytes()).hexdigest()\n    save_json(out/\'postprocessing_plan.json\',plan)\n    plan_digest=hashlib.sha256((out/\'postprocessing_plan.json\').read_bytes()).hexdigest()\n    print(f\'Frozen postprocessing: {chosen["name"]}; forward dev2 {raw[1]:.9f} -> {chosen["development2_forward_rmsle"]:.9f}\',flush=True)\n    for i in range(2):\n        f=frames[i].copy();f[\'postprocessed\']=apply_plan(contexts[i],preds[i],plan)\n        folder=out/f\'fold_{i+1}\';folder.mkdir(exist_ok=True)\n        f.to_csv(folder/\'oof.csv.gz\',index=False,compression=\'gzip\')\n    # This file is read only after all candidates and parameters have been frozen.\n    f=pd.read_csv(base/\'fold_3/oof.csv.gz\',parse_dates=[\'date\'])\n    context=build_context(train,f,origins[2]);original=f.lightgbm_large.to_numpy()\n    corrected=apply_plan(context,original,plan)\n    raw_score=rmsle(f.boardings,original);new_score=rmsle(f.boardings,corrected)\n    folder=out/\'fold_3\';folder.mkdir(exist_ok=True)\n    f[\'postprocessed\']=corrected;f.to_csv(folder/\'oof.csv.gz\',index=False,compression=\'gzip\')\n    summary=diagnostics(train,f,corrected,pd.Timestamp(origins[2]),folder)\n    save_json(out/\'audit_summary.json\',summary)\n    assert plan_digest==hashlib.sha256((out/\'postprocessing_plan.json\').read_bytes()).hexdigest()\n    segments=[]\n    f[\'is_holiday\']=context.is_holiday.to_numpy()\n    for label,mask in [(\'all\',np.ones(len(f),dtype=bool)),(\'nonholiday\',~f.is_holiday),(\'holiday\',f.is_holiday)]:\n        if np.sum(mask):segments.append({\'segment\':label,\'rows\':int(np.sum(mask)),\'raw_rmsle\':rmsle(f.loc[mask,\'boardings\'],original[mask]),\'postprocessed_rmsle\':rmsle(f.loc[mask,\'boardings\'],corrected[mask])})\n    for month,rows in f.groupby(f.date.dt.to_period(\'M\')):\n        ix=rows.index.to_numpy();segments.append({\'segment\':str(month),\'rows\':len(rows),\'raw_rmsle\':rmsle(rows.boardings,original[ix]),\'postprocessed_rmsle\':rmsle(rows.boardings,corrected[ix])})\n    pd.DataFrame(segments).to_csv(out/\'audit_segments.csv\',index=False)\n    # Fixed component ablations explain the outcome; no coefficients are refit\n    # using the reused comparison targets and no alternative is auto-promoted.\n    ablations=[]\n    for name in [\'raw\',\'calibration_only\',\'anchor_only\',\'combined\']:\n        component=copy.deepcopy(plan)\n        if name==\'calibration_only\':component[\'spec\'][\'anchor\']={\'days\':0,\'weight\':0.}\n        if name==\'anchor_only\':component[\'spec\'][\'calibration\']={\'kind\':\'none\',\'scale\':0.}\n        values=original if name==\'raw\' else apply_plan(context,original,component)\n        error=np.log1p(values)-np.log1p(f.boardings)\n        ablations.append({\'component\':name,\'rmsle\':float(np.sqrt(np.mean(error**2))),\n                          \'mean_log_error\':float(error.mean()),\'log_error_variance\':float(np.var(error))})\n    pd.DataFrame(ablations).to_csv(out/\'component_ablation.csv\',index=False)\n    daily=pd.DataFrame({\'date\':f.date,\'raw\':(np.log1p(original)-np.log1p(f.boardings))**2,\'post\':(np.log1p(corrected)-np.log1p(f.boardings))**2}).groupby(\'date\').agg(raw=(\'raw\',\'sum\'),post=(\'post\',\'sum\'),n=(\'date\',\'size\'))\n    rng=np.random.default_rng(2026);ix=rng.integers(0,len(daily),size=(2000,len(daily)))\n    n=daily.n.to_numpy()[ix].sum(axis=1)\n    delta=np.sqrt(daily.post.to_numpy()[ix].sum(axis=1)/n)-np.sqrt(daily.raw.to_numpy()[ix].sum(axis=1)/n)\n    # Keep original weights immutable; attach the frozen postprocessor to a new bundle.\n    bundle=joblib.load(base/\'model_bundle.joblib\')\n    for name,digest in bundle[\'input_sha256\'].items():\n        if hashlib.sha256((args.data_dir/name).read_bytes()).hexdigest()!=digest:raise ValueError(f\'Input changed: {name}\')\n    xt=make_features(train,data[\'test\'],data[\'metaData\'],bundle[\'training_cutoff\'])\n    model_preds={name:model.predict(xt) for name,model in bundle[\'models\'].items()}\n    before=predict_ensemble(model_preds,bundle[\'ensemble\'])\n    test_context=build_context(train,data[\'test\'],bundle[\'training_cutoff\'])\n    after=apply_plan(test_context,before,plan)\n    bundle=copy.copy(bundle);bundle[\'postprocessing\']=plan\n    bundle[\'postprocessing_source\']=str(base)\n    joblib.dump(bundle,out/\'model_bundle.joblib\',compress=3)\n    export_predictions(data,after,out,corrected)\n    raw_output=out/\'unprocessed_reference\';export_predictions(data,before,raw_output,original)\n    assert hashlib.sha256((raw_output/\'submission.csv\').read_bytes()).hexdigest()==hashlib.sha256((base/\'submission.csv\').read_bytes()).hexdigest()\n    loaded=joblib.load(out/\'model_bundle.joblib\')\n    reloaded=apply_plan(test_context,before,loaded[\'postprocessing\'])\n    np.testing.assert_array_equal(after,reloaded)\n    assert hashlib.sha256((base/\'model_bundle.joblib\').read_bytes()).hexdigest()==plan[\'source_base_bundle_sha256\']\n    report={\'status\':\'completed\',\'base_rmsle\':raw_score,\'postprocessed_rmsle\':new_score,\n            \'relative_rmsle_reduction_percent\':100*(1-new_score/raw_score),\'selected_candidate\':spec[\'name\'],\n            \'improved_reused_comparison\':new_score<raw_score,\'seconds\':time.perf_counter()-started,\n            \'paired_date_bootstrap_delta_ci95\':list(map(float,np.quantile(delta,[.025,.975]))),\n            \'bootstrap_note\':\'Descriptive paired date resampling; does not preserve all serial dependence or prove future performance.\',\n            \'postprocessing_plan_frozen_before_comparison\':True,\'original_weights_unchanged\':True,\n            \'test_rows\':len(after),\'test_holiday_predictions_unchanged\':bool(np.array_equal(after[test_context.is_holiday],before[test_context.is_holiday])),\n            \'prediction_reloaded_identical\':True,\'evaluation_context\':design[\'context\']}\n    save_json(out/\'run_summary.json\',report)\n    save_json(out/\'adoption_decision.json\',{\'adopted_as_default\':False,\n              \'reason\':\'Experimental evaluation only. Keep the base model unless a separate documented adoption decision is made.\',\n              \'base_rmsle\':raw_score,\'candidate_rmsle\':new_score,\'base_model_bundle\':str(base/\'model_bundle.joblib\')})\n    (out/\'EXPERIMENT_ONLY.md\').write_text(\'This directory contains postprocessing experiments. It is not the default submission. See adoption_decision.json and reports/POSTPROCESSING.md.\\n\')\n    print(json.dumps(report,ensure_ascii=False,indent=2))\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--data-dir\',type=Path,default=ROOT/\'data/raw\')\n    p.add_argument(\'--base-output\',type=Path,default=ROOT/\'outputs/lightgbm_tuned\')\n    p.add_argument(\'--output\',type=Path,default=ROOT/\'outputs/lightgbm_postprocessed\')\n    run(p.parse_args())\n\nif __name__==\'__main__\':main()\n', 'src/features.py': '"""Origin-frozen direct forecasting features. Never read labels after origin."""\nfrom pathlib import Path\nfrom .features_v2 import make_features_v2, training_matrix_v2, partial_origins\nimport numpy as np\nimport pandas as pd\nfrom .holiday_calendar import calendar_features\n\nKEYS = [\'station_id\', \'hour\']\n\ndef load_data(data_dir):\n    import polars as pl\n    root = Path(data_dir)\n    result = {}\n    for name in [\'train\', \'test\', \'sample_submission\', \'metaData\']:\n        lf = pl.scan_csv(root / f\'{name}.csv\')\n        schema = lf.collect_schema()\n        casts = [pl.col(c).cast(pl.Int32) for c in [\'station_id\', \'hour\', \'boardings\', \'alightings\', \'canonical_line\'] if c in schema]\n        df = pd.DataFrame(lf.with_columns(casts).collect().to_dict(as_series=False))\n        if \'date\' in df: df[\'date\'] = pd.to_datetime(df[\'date\'])\n        for c in [\'station_id\', \'hour\', \'boardings\', \'alightings\', \'canonical_line\']:\n            if c in df: df[c] = df[c].astype(\'int32\')\n        result[name] = df\n    tr, te = result[\'train\'], result[\'test\']\n    for name, frame in [(\'train\',tr),(\'test\',te)]:\n        if frame.duplicated([\'date\'] + KEYS).any(): raise ValueError(f\'{name}: duplicate keys\')\n        if frame.isna().any().any(): raise ValueError(f\'{name}: input cells missing; inspect first\')\n    if tr.date.max() >= te.date.min(): raise ValueError(\'Test is not strictly after training\')\n    if not np.isfinite(tr[[\'boardings\',\'alightings\']]).all().all() or (tr[[\'boardings\',\'alightings\']] < 0).any().any():\n        raise ValueError(\'Training counts must be finite and nonnegative\')\n    return result\n\ndef make_features(history, query, metadata, origin):\n    origin = pd.Timestamp(origin)\n    hist = history.loc[history.date <= origin].copy()\n    if hist.empty: raise ValueError(\'No history at this origin\')\n    if (query.date <= origin).any(): raise ValueError(\'Queries must follow origin\')\n    q = query[[\'date\'] + KEYS].reset_index(drop=True).copy()\n    q[\'_order\'] = np.arange(len(q))\n    q[\'dow\'] = q.date.dt.dayofweek.astype(\'int8\')\n    hist[\'dow\'] = hist.date.dt.dayofweek.astype(\'int8\')\n    q = q.merge(metadata[[\'station_id\',\'canonical_line\']],on=\'station_id\',how=\'left\',validate=\'many_to_one\')\n    if q.canonical_line.isna().any(): raise ValueError(\'Station absent from metadata\')\n    hist = hist.merge(metadata[[\'station_id\',\'canonical_line\']],on=\'station_id\',how=\'left\',validate=\'many_to_one\')\n    q[\'horizon\'] = (q.date - origin).dt.days\n    q[\'month\'] = q.date.dt.month\n    q[\'day_of_month\'] = q.date.dt.day\n    q[\'day_of_year\'] = q.date.dt.dayofyear\n    q[\'date_index\'] = (q.date - pd.Timestamp(\'2025-01-01\')).dt.days\n    q[\'weekend\'] = (q.dow >= 5).astype(\'int8\')\n    q = pd.concat([q,calendar_features(q.date,as_of=origin)],axis=1)\n    for col, period in [(\'hour\',24),(\'dow\',7),(\'day_of_year\',365.25)]:\n        for trig in [\'sin\',\'cos\']:\n            q[f\'{col}_{trig}\'] = getattr(np,trig)(2*np.pi*q[col]/period)\n    inception = hist.groupby(\'station_id\',as_index=False).date.min().rename(columns={\'date\':\'observed_first_date\'})\n    q = q.merge(inception,on=\'station_id\',how=\'left\',validate=\'many_to_one\')\n    q[\'history_days\'] = (origin - q.observed_first_date).dt.days + 1\n    q[\'cold_start\'] = q.observed_first_date.isna().astype(\'int8\')\n    q = q.drop(columns=\'observed_first_date\')\n    # All aggregations are computed only once per origin, never using query labels.\n    for days in [28,56,112]:\n        h = hist.loc[hist.date > origin - pd.Timedelta(days=days)]\n        stats = h.groupby(KEYS+[\'dow\']).agg(\n            **{f\'board_mean_{days}\':(\'boardings\',\'mean\'),\n               f\'board_median_{days}\':(\'boardings\',\'median\'),\n               f\'board_std_{days}\':(\'boardings\',\'std\'),\n               f\'alight_mean_{days}\':(\'alightings\',\'mean\'),\n               f\'n_{days}\':(\'boardings\',\'size\')}).reset_index()\n        q = q.merge(stats,on=KEYS+[\'dow\'],how=\'left\',validate=\'many_to_one\')\n    short = hist.loc[hist.date > origin - pd.Timedelta(days=28)]\n    stats = short.groupby(KEYS).agg(recent_mean=(\'boardings\',\'mean\'),recent_std=(\'boardings\',\'std\'),recent_alight=(\'alightings\',\'mean\')).reset_index()\n    q = q.merge(stats,on=KEYS,how=\'left\',validate=\'many_to_one\')\n    for keys,label in [([\'canonical_line\',\'hour\',\'dow\'],\'line\'),([\'hour\',\'dow\'],\'global\')]:\n        agg = short.groupby(keys,as_index=False).boardings.median().rename(columns={\'boardings\':f\'{label}_median\'})\n        q = q.merge(agg,on=keys,how=\'left\',validate=\'many_to_one\')\n    # Weekly seasonal copy for arbitrary horizon: repeat the last observed weekday.\n    q[\'_lookup_date\'] = origin - pd.to_timedelta((origin.dayofweek - q.dow)%7,unit=\'D\')\n    lookup = hist[[\'date\']+KEYS+[\'boardings\',\'alightings\']].rename(columns={\'date\':\'_lookup_date\',\'boardings\':\'seasonal7\',\'alightings\':\'seasonal7_alight\'})\n    q = q.merge(lookup,on=[\'_lookup_date\']+KEYS,how=\'left\',validate=\'many_to_one\').drop(columns=\'_lookup_date\')\n    for days in [1,7,28]:\n        snap = hist.loc[hist.date == origin - pd.Timedelta(days=days-1),KEYS+[\'boardings\']].rename(columns={\'boardings\':f\'origin_lag_{days}\'})\n        q = q.merge(snap,on=KEYS,how=\'left\',validate=\'many_to_one\')\n    q[\'_lookup_date\'] = q.date - pd.Timedelta(days=364)\n    # This explicit guard keeps the feature valid if a longer forecast is requested.\n    q.loc[q._lookup_date > origin,\'_lookup_date\'] = pd.NaT\n    lookup = hist[[\'date\']+KEYS+[\'boardings\']].rename(columns={\'date\':\'_lookup_date\',\'boardings\':\'year364\'})\n    q = q.merge(lookup,on=[\'_lookup_date\']+KEYS,how=\'left\',validate=\'many_to_one\').drop(columns=\'_lookup_date\')\n    q[\'trend_28_112\'] = q.board_mean_28 / q.board_mean_112.clip(lower=1)\n    q[\'boarding_alighting_ratio\'] = q.board_mean_56 / q.alight_mean_56.clip(lower=1)\n    q[\'baseline\'] = q.board_median_56.fillna(q.line_median).fillna(q.global_median).fillna(float(hist.boardings.median()))\n    q[\'seasonal7\'] = q.seasonal7.fillna(q.baseline)\n    q[\'weekday_mean\'] = q.board_mean_56.fillna(q.baseline)\n    q = q.sort_values(\'_order\').drop(columns=[\'_order\',\'date\']).reset_index(drop=True)\n    return q.astype(\'float32\')\n\ndef training_matrix(train, metadata, cutoff, horizon=123, origin_stride=28, max_rows=220000, seed=2026):\n    """Whole label windows end by cutoff; last training origin is horizon days earlier."""\n    cutoff = pd.Timestamp(cutoff)\n    start = train.date.min() + pd.Timedelta(days=27)\n    end = cutoff - pd.Timedelta(days=horizon)\n    origins = pd.date_range(start,end,freq=f\'{origin_stride}D\')\n    if len(origins) == 0: raise ValueError(\'Insufficient history for full-horizon training\')\n    # Include the exact last permissible anchor so the newest observed labels\n    # are represented even when the 28-day grid misses the cutoff boundary.\n    if origins[-1] != end: origins = origins.append(pd.DatetimeIndex([end]))\n    rng = np.random.default_rng(seed)\n    per_origin = max(1,max_rows//len(origins))\n    xs, ys, audit = [], [], []\n    for origin in origins:\n        labels = train.loc[(train.date > origin)&(train.date <= origin + pd.Timedelta(days=horizon))].copy()\n        if len(labels)>per_origin:\n            labels = labels.iloc[np.sort(rng.choice(len(labels),per_origin,replace=False))]\n        x = make_features(train,labels,metadata,origin)\n        xs.append(x); ys.append(labels.boardings.to_numpy(dtype=\'float32\'))\n        audit.append({\'origin\':str(origin.date()),\'feature_max_date\':str(origin.date()),\'label_end\':str((origin+pd.Timedelta(days=horizon)).date()),\'rows\':len(labels)})\n    return pd.concat(xs,ignore_index=True),np.concatenate(ys),audit\n', 'src/features_v2.py': '"""Causal partial-horizon snapshots and cleaned log demand features for v2."""\nimport numpy as np\nimport pandas as pd\nfrom .holiday_calendar import calendar_features\n\nCATS=[\'station_id\',\'canonical_line\',\'hour\',\'dow\',\'eff_dow\',\'holiday_subtype\',\'station_hour\',\'station_eff_dow\']\nNEW=[3107,3108,3109]\nTRANSFERS={3118:3761,3761:3118,3124:3221,3221:3124,3213:3763,3763:3213}\n\ndef calendar(dates,origin):\n    d=pd.Series(pd.to_datetime(dates)).reset_index(drop=True)\n    c=calendar_features(d,as_of=origin).reset_index(drop=True)\n    c[\'eff_dow\']=np.where(c.is_holiday.astype(bool)&~c.is_labor_day.astype(bool),6,d.dt.dayofweek)\n    c[\'holiday_subtype\']=c.holiday_type\n    for key in [\'lunar_day_0\',\'lunar_m1\',\'lunar_p1\']:c[key]=0\n    centers=[\'2025-01-29\',\'2025-10-06\',\'2026-02-17\',\'2026-09-25\']\n    for center in centers:\n        for key,offset in [(\'lunar_day_0\',0),(\'lunar_m1\',-1),(\'lunar_p1\',1)]:\n            c.loc[(d==pd.Timestamp(center)+pd.Timedelta(days=offset))&c.is_lunar_holiday.astype(bool),key]=1\n    # First ordinary weekday after a holiday block, allowing an intervening weekend.\n    previous=[calendar_features(d-pd.Timedelta(days=k),as_of=origin) if (d-pd.Timedelta(days=k)).min()>=pd.Timestamp(\'2025-01-01\') else None for k in range(1,5)]\n    post=np.zeros(len(d),bool);between=np.ones(len(d),bool)\n    for cprev in previous:\n        if cprev is None:continue\n        post|=between&cprev.is_holiday.to_numpy(dtype=bool)\n        between&=cprev.is_rest_day.to_numpy(dtype=bool)\n    c[\'post_holiday_workday\']=post&~c.is_rest_day.to_numpy(dtype=bool)\n    md=d.dt.month*100+d.dt.day\n    # Generic seasonal proxies requested by user, not verified university schedules.\n    c[\'summer_break_proxy\']=md.between(622,831)\n    c[\'winter_break_proxy\']=(md>=1222)|(md<=228)\n    c[\'semester_first2w_proxy\']=md.between(302,315)|md.between(901,914)\n    return c\n\ndef cleaned(history,origin):\n    h=history.loc[history.date<=pd.Timestamp(origin)].copy().reset_index(drop=True)\n    c=calendar_features(h.date,as_of=origin).reset_index(drop=True)\n    h[\'dow\']=h.date.dt.dayofweek;h[\'eff_dow\']=h.dow\n    h[\'holiday\']=c.is_holiday.to_numpy(dtype=bool);h[\'rest\']=c.is_rest_day.to_numpy(dtype=int)\n    h[\'opening\']=h.station_id.isin(NEW)&h.date.between(\'2025-06-28\',\'2025-06-30\')\n    h[\'clean\']=~(h.holiday|c.is_labor_day.to_numpy(dtype=bool)|h.opening)\n    if pd.Timestamp(origin)>=pd.Timestamp(\'2025-07-28\'):\n        h.loc[h.station_id.isin([3110,3111])&(h.date<\'2025-07-01\'),\'clean\']=False\n    h[\'board_log\']=np.log1p(h.boardings);h[\'alight_log\']=np.log1p(h.alightings)\n    return h\n\ndef lookup(table,q,keys,col):\n    return table[col].reindex(pd.MultiIndex.from_frame(q[keys]) if len(keys)>1 else q[keys[0]]).to_numpy()\n\ndef make_features_v2(history,query,meta,origin):\n    origin=pd.Timestamp(origin)\n    q=query[[\'date\',\'station_id\',\'hour\']].copy().reset_index(drop=True)\n    if (q.date<=origin).any():raise ValueError(\'Query must be after origin\')\n    h=cleaned(history,origin);line_map=meta.set_index(\'station_id\').canonical_line\n    h[\'canonical_line\']=h.station_id.map(line_map)\n    q[\'canonical_line\']=q.station_id.map(line_map);q[\'dow\']=q.date.dt.dayofweek\n    q=pd.concat([q,calendar(q.date,origin)],axis=1)\n    q[\'horizon\']=(q.date-origin).dt.days;q[\'month\']=q.date.dt.month;q[\'day_of_month\']=q.date.dt.day\n    q[\'day_of_year\']=q.date.dt.dayofyear\n    for col,period in [(\'hour\',24),(\'dow\',7),(\'day_of_year\',365.25)]:\n        q[col+\'_sin\']=np.sin(2*np.pi*q[col]/period);q[col+\'_cos\']=np.cos(2*np.pi*q[col]/period)\n    first=h.groupby(\'station_id\').date.min();q[\'history_days\']=(origin-q.station_id.map(first)).dt.days+1\n    q[\'cold_start\']=q.history_days.isna().astype(int)\n    keys=[\'station_id\',\'hour\',\'eff_dow\'];profiles={}\n    clean=h.loc[h.clean]\n    for days in [28,56,112]:\n        w=clean.loc[clean.date>origin-pd.Timedelta(days=days)]\n        tab=w.groupby(keys).agg(board_mean=(\'board_log\',\'mean\'),board_median=(\'board_log\',\'median\'),board_std=(\'board_log\',\'std\'),alight_mean=(\'alight_log\',\'mean\'),alight_median=(\'alight_log\',\'median\'),alight_std=(\'alight_log\',\'std\'),count=(\'board_log\',\'size\'))\n        profiles[days]=tab\n        for col in tab:q[f\'{col}_{days}\']=lookup(tab,q,keys,col)\n    recent=clean.loc[clean.date>origin-pd.Timedelta(days=112)]\n    fallback_line=recent.groupby([\'canonical_line\',\'hour\',\'eff_dow\'])[[\'board_log\',\'alight_log\']].mean()\n    fallback_global=recent.groupby([\'hour\',\'eff_dow\'])[[\'board_log\',\'alight_log\']].mean()\n    def anchor(rows,target):\n        a=lookup(profiles[56],rows,keys,target+\'_mean\');b=lookup(profiles[112],rows,keys,target+\'_mean\')\n        a=np.where(np.isfinite(a),a,b)\n        b=lookup(fallback_line,rows,[\'canonical_line\',\'hour\',\'eff_dow\'],target+\'_log\');a=np.where(np.isfinite(a),a,b)\n        b=lookup(fallback_global,rows,[\'hour\',\'eff_dow\'],target+\'_log\');a=np.where(np.isfinite(a),a,b)\n        a=np.where(np.isfinite(a),a,float(clean[target+\'_log\'].mean()))\n        short=lookup(profiles[28],rows,keys,target+\'_mean\')\n        return .6*a+.4*np.where(np.isfinite(short),short,a)\n    q[\'anchor_board_log\']=anchor(q,\'board\');q[\'anchor_alight_log\']=anchor(q,\'alight\')\n    cal=q.copy();cal[\'eff_dow\']=cal.dow;q[\'cal_dow_log_mean\']=lookup(profiles[56],cal,keys,\'board_mean\')\n    cal[\'eff_dow\']=6;q[\'sun_log_mean\']=lookup(profiles[56],cal,keys,\'board_mean\')\n    q[\'sun_minus_wkd_log\']=q.sun_log_mean-q.cal_dow_log_mean\n    q[\'growth_log\']=q.board_mean_28-q.board_mean_112\n    q[\'board_alight_asymmetry\']=q.board_mean_56-q.alight_mean_56\n    # Historical holiday elasticity, never future labels.\n    ordinary=h.loc[h.clean&(h.dow<5)].groupby([\'station_id\',\'hour\']).board_log.mean().to_frame(\'ordinary\')\n    holiday_mask=h.holiday&~h.opening&~((h.date.dt.month==5)&(h.date.dt.day==1))\n    if origin>=pd.Timestamp(\'2025-07-28\'):\n        holiday_mask&=~(h.station_id.isin([3110,3111])&(h.date<\'2025-07-01\'))\n    # Holiday observations are needed only for this explicit contrast feature;\n    # the ordinary side uses the same cleaned structural regime as the anchors.\n    holiday=h.loc[holiday_mask].groupby([\'station_id\',\'hour\']).board_log.mean().to_frame(\'holiday\')\n    q[\'holiday_elasticity\']=lookup(holiday,q,[\'station_id\',\'hour\'],\'holiday\')-lookup(ordinary,q,[\'station_id\',\'hour\'],\'ordinary\')\n    shares=recent.groupby([\'station_id\',\'hour\']).boardings.mean();shares=shares/shares.groupby(level=0).transform(\'sum\')\n    q[\'hour_share\']=lookup(shares.to_frame(\'share\'),q,[\'station_id\',\'hour\'],\'share\')\n    # Exact prior-year lookup, replace holiday/opening observations with adjacent weeks.\n    eligible=h.loc[~h.holiday&~h.opening&~((h.date.dt.month==5)&(h.date.dt.day==1))]\n    prior=eligible.set_index([\'date\',\'station_id\',\'hour\'])[[\'board_log\']]\n    year=np.full(len(q),np.nan)\n    for days in [364,357,371]:\n        dates=q[[\'date\',\'station_id\',\'hour\']].copy();dates[\'date\']=dates.date-pd.Timedelta(days=days)\n        val=lookup(prior,dates,[\'date\',\'station_id\',\'hour\'],\'board_log\')\n        val[dates.date>origin]=np.nan;year=np.where(np.isfinite(year),year,val)\n    q[\'year364_clean_log\']=year;q[\'year364_missing\']=~np.isfinite(year)\n    # Previous target month minus the 56-day window preceding origin one year earlier.\n    baseline_end=origin-pd.DateOffset(years=1)\n    base=clean.loc[(clean.date>baseline_end-pd.Timedelta(days=56))&(clean.date<=baseline_end)]\n    base_station=base.groupby([\'station_id\',\'rest\']).agg(value=(\'board_log\',\'mean\'),days=(\'date\',\'nunique\'))\n    base_line=base.groupby([\'canonical_line\',\'hour\',\'rest\']).board_log.mean().to_frame(\'value\')\n    q[\'yoy_seas_log_delta\']=np.nan;q[\'yoy_seas_fallback\']=1\n    for (year_now,month),indices in q.groupby([q.date.dt.year,q.date.dt.month]).groups.items():\n        part=q.loc[indices].copy();part[\'rest\']=part.is_rest_day\n        previous=clean.loc[(clean.date.dt.year==year_now-1)&(clean.date.dt.month==month)]\n        month_station=previous.groupby([\'station_id\',\'rest\']).board_log.mean().to_frame(\'value\')\n        month_line=previous.groupby([\'canonical_line\',\'hour\',\'rest\']).board_log.mean().to_frame(\'value\')\n        a=lookup(month_station,part,[\'station_id\',\'rest\'],\'value\')-lookup(base_station,part,[\'station_id\',\'rest\'],\'value\')\n        enough=part.station_id.map(base.groupby(\'station_id\').date.nunique()).to_numpy()>=14\n        b=lookup(month_line,part,[\'canonical_line\',\'hour\',\'rest\'],\'value\')-lookup(base_line,part,[\'canonical_line\',\'hour\',\'rest\'],\'value\')\n        use=enough&np.isfinite(a);delta=np.where(use,a,b)\n        q.loc[indices,\'yoy_seas_fallback\']=(~use).astype(int)\n        q.loc[indices,\'yoy_seas_log_delta\']=delta\n    q[\'yoy_seas_missing\']=q.yoy_seas_log_delta.isna();q[\'yoy_seas_log_delta\']=q.yoy_seas_log_delta.fillna(0)\n    for label,ids in [(\'previous\',q.station_id-1),(\'next\',q.station_id+1),(\'transfer\',q.station_id.map(TRANSFERS))]:\n        other=q[[\'station_id\',\'canonical_line\',\'hour\',\'eff_dow\']].copy();other[\'station_id\']=ids\n        other[\'canonical_line\']=ids.map(line_map)\n        valid=other.canonical_line.notna()\n        if label!=\'transfer\':valid&=other.canonical_line==q.canonical_line\n        value=anchor(other,\'board\');value[~valid]=np.nan\n        q[f\'{label}_anchor_log\']=value\n        q[f\'{label}_growth_log\']=lookup(profiles[28],other,keys,\'board_mean\')-lookup(profiles[112],other,keys,\'board_mean\')\n        q.loc[~valid,f\'{label}_growth_log\']=np.nan\n        q[f\'{label}_log_ratio\']=q.anchor_board_log-q[f\'{label}_anchor_log\']\n    q[\'station_hour\']=q.station_id*24+q.hour;q[\'station_eff_dow\']=q.station_id*7+q.eff_dow\n    return q.drop(columns=\'date\').astype(\'float32\')\n\ndef partial_origins(cutoff,start=\'2025-02-11\'):\n    end=pd.Timestamp(cutoff)-pd.Timedelta(days=14)\n    origins=list(pd.date_range(start,end,freq=\'14D\'))\n    if origins and origins[-1]!=end:origins.append(end)\n    return origins\n\ndef sampled_pairs(train,cutoff,max_rows,seed=2026):\n    cutoff=pd.Timestamp(cutoff);origins=partial_origins(cutoff)\n    dates={d:idx.to_numpy() for d,idx in train.loc[train.date<=cutoff].groupby(\'date\').groups.items()}\n    rng=np.random.default_rng(seed);pairs=[]\n    for horizon in range(1,124):\n        rows=[];os=[];weights=[]\n        for oi,o in enumerate(origins):\n            d=o+pd.Timedelta(days=horizon)\n            if d>cutoff or d not in dates:continue\n            ids=dates[d];rows.extend(ids);os.extend([oi]*len(ids))\n            w=.5 if o<pd.Timestamp(\'2025-07-01\') else 1.5 if o<pd.Timestamp(\'2025-12-30\') else 2.5\n            weights.extend([w]*len(ids))\n        n=min(len(rows),max_rows//123+(horizon<=max_rows%123))\n        if n:\n            w=np.asarray(weights);ix=rng.choice(len(rows),n,replace=False,p=w/w.sum())\n            pairs.append(pd.DataFrame({\'row\':np.asarray(rows)[ix],\'origin_index\':np.asarray(os)[ix],\'horizon\':horizon}))\n    return pd.concat(pairs,ignore_index=True),origins\n\ndef training_matrix_v2(train,meta,cutoff,max_rows,seed=2026):\n    pairs,origins=sampled_pairs(train,cutoff,max_rows,seed);xs=[];ys=[];audit=[]\n    for oi,p in pairs.groupby(\'origin_index\',sort=True):\n        o=origins[oi];labels=train.loc[p.row].reset_index(drop=True)\n        assert labels.date.min()>o and labels.date.max()<=pd.Timestamp(cutoff)\n        x=make_features_v2(train,labels,meta,o)\n        y=np.column_stack([np.log1p(labels.boardings)-x.anchor_board_log,np.log1p(labels.alightings)-x.anchor_alight_log]).astype(\'float32\')\n        xs.append(x);ys.append(y)\n        audit.append({\'origin\':str(o.date()),\'rows\':len(p),\'label_max\':str(labels.date.max().date()),\'min_horizon\':int(p.horizon.min()),\'max_horizon\':int(p.horizon.max())})\n    return pd.concat(xs,ignore_index=True),np.concatenate(ys),audit,pairs.horizon.value_counts().sort_index().to_dict()\n', 'src/features_v3.py': '"""Causal partial-horizon snapshots and cleaned log demand features for v3."""\nimport numpy as np\nimport pandas as pd\nfrom .holiday_calendar import calendar_features\nfrom .domain_events import oneoff_mask, add_domain_features\n\nCATS=[\'station_id\',\'canonical_line\',\'hour\',\'dow\',\'eff_dow\',\'holiday_subtype\',\'station_hour\',\'station_eff_dow\']\nNEW=[3107,3108,3109]\nTRANSFERS={3118:3761,3761:3118,3124:3221,3221:3124,3213:3763,3763:3213}\n\ndef calendar(dates,origin):\n    d=pd.Series(pd.to_datetime(dates)).reset_index(drop=True)\n    c=calendar_features(d,as_of=origin).reset_index(drop=True)\n    c[\'eff_dow\']=np.where(c.is_holiday.astype(bool)&~c.is_labor_day.astype(bool),6,d.dt.dayofweek)\n    c[\'holiday_subtype\']=c.holiday_type\n    for key in [\'lunar_day_0\',\'lunar_m1\',\'lunar_p1\']:c[key]=0\n    centers=[\'2025-01-29\',\'2025-10-06\',\'2026-02-17\',\'2026-09-25\']\n    for center in centers:\n        for key,offset in [(\'lunar_day_0\',0),(\'lunar_m1\',-1),(\'lunar_p1\',1)]:\n            c.loc[(d==pd.Timestamp(center)+pd.Timedelta(days=offset))&c.is_lunar_holiday.astype(bool),key]=1\n    # First ordinary weekday after a holiday block, allowing an intervening weekend.\n    previous=[calendar_features(d-pd.Timedelta(days=k),as_of=origin) if (d-pd.Timedelta(days=k)).min()>=pd.Timestamp(\'2025-01-01\') else None for k in range(1,5)]\n    post=np.zeros(len(d),bool);between=np.ones(len(d),bool)\n    for cprev in previous:\n        if cprev is None:continue\n        post|=between&cprev.is_holiday.to_numpy(dtype=bool)\n        between&=cprev.is_rest_day.to_numpy(dtype=bool)\n    c[\'post_holiday_workday\']=post&~c.is_rest_day.to_numpy(dtype=bool)\n    md=d.dt.month*100+d.dt.day\n    # Generic seasonal proxies requested by user, not verified university schedules.\n    c[\'summer_break_proxy\']=md.between(622,831)\n    c[\'winter_break_proxy\']=(md>=1222)|(md<=228)\n    c[\'semester_first2w_proxy\']=md.between(302,315)|md.between(901,914)\n    return c\n\ndef cleaned(history,origin):\n    h=history.loc[history.date<=pd.Timestamp(origin)].copy().reset_index(drop=True)\n    c=calendar_features(h.date,as_of=origin).reset_index(drop=True)\n    h[\'dow\']=h.date.dt.dayofweek;h[\'eff_dow\']=h.dow\n    h[\'holiday\']=c.is_holiday.to_numpy(dtype=bool);h[\'rest\']=c.is_rest_day.to_numpy(dtype=int)\n    h[\'opening\']=h.station_id.isin(NEW)&h.date.between(\'2025-06-28\',\'2025-06-30\')\n    h[\'oneoff\']=oneoff_mask(h)\n    h[\'clean\']=~(h.holiday|c.is_labor_day.to_numpy(dtype=bool)|h.oneoff)\n    if pd.Timestamp(origin)>=pd.Timestamp(\'2025-07-28\'):\n        h.loc[h.station_id.isin([3110,3111])&(h.date<\'2025-07-01\'),\'clean\']=False\n    h[\'board_log\']=np.log1p(h.boardings);h[\'alight_log\']=np.log1p(h.alightings)\n    return h\n\ndef lookup(table,q,keys,col):\n    return table[col].reindex(pd.MultiIndex.from_frame(q[keys]) if len(keys)>1 else q[keys[0]]).to_numpy()\n\ndef make_features_v3(history,query,meta,origin):\n    origin=pd.Timestamp(origin)\n    q=query[[\'date\',\'station_id\',\'hour\']].copy().reset_index(drop=True)\n    if (q.date<=origin).any():raise ValueError(\'Query must be after origin\')\n    h=cleaned(history,origin);line_map=meta.set_index(\'station_id\').canonical_line\n    h[\'canonical_line\']=h.station_id.map(line_map)\n    q[\'canonical_line\']=q.station_id.map(line_map);q[\'dow\']=q.date.dt.dayofweek\n    q=pd.concat([q,calendar(q.date,origin)],axis=1)\n    q[\'horizon\']=(q.date-origin).dt.days;q[\'month\']=q.date.dt.month;q[\'day_of_month\']=q.date.dt.day\n    q[\'day_of_year\']=q.date.dt.dayofyear\n    for col,period in [(\'hour\',24),(\'dow\',7),(\'day_of_year\',365.25)]:\n        q[col+\'_sin\']=np.sin(2*np.pi*q[col]/period);q[col+\'_cos\']=np.cos(2*np.pi*q[col]/period)\n    first=h.groupby(\'station_id\').date.min();q[\'history_days\']=(origin-q.station_id.map(first)).dt.days+1\n    q[\'cold_start\']=q.history_days.isna().astype(int)\n    keys=[\'station_id\',\'hour\',\'eff_dow\'];profiles={}\n    clean=h.loc[h.clean]\n    for days in [28,56,112]:\n        w=clean.loc[clean.date>origin-pd.Timedelta(days=days)]\n        tab=w.groupby(keys).agg(board_mean=(\'board_log\',\'mean\'),board_median=(\'board_log\',\'median\'),board_std=(\'board_log\',\'std\'),alight_mean=(\'alight_log\',\'mean\'),alight_median=(\'alight_log\',\'median\'),alight_std=(\'alight_log\',\'std\'),count=(\'board_log\',\'size\'))\n        profiles[days]=tab\n        for col in tab:q[f\'{col}_{days}\']=lookup(tab,q,keys,col)\n    recent=clean.loc[clean.date>origin-pd.Timedelta(days=112)]\n    fallback_line=recent.groupby([\'canonical_line\',\'hour\',\'eff_dow\'])[[\'board_log\',\'alight_log\']].mean()\n    fallback_global=recent.groupby([\'hour\',\'eff_dow\'])[[\'board_log\',\'alight_log\']].mean()\n    def anchor(rows,target):\n        a=lookup(profiles[56],rows,keys,target+\'_mean\');b=lookup(profiles[112],rows,keys,target+\'_mean\')\n        a=np.where(np.isfinite(a),a,b)\n        b=lookup(fallback_line,rows,[\'canonical_line\',\'hour\',\'eff_dow\'],target+\'_log\');a=np.where(np.isfinite(a),a,b)\n        b=lookup(fallback_global,rows,[\'hour\',\'eff_dow\'],target+\'_log\');a=np.where(np.isfinite(a),a,b)\n        a=np.where(np.isfinite(a),a,float(clean[target+\'_log\'].mean()))\n        short=lookup(profiles[28],rows,keys,target+\'_mean\')\n        return .6*a+.4*np.where(np.isfinite(short),short,a)\n    q[\'anchor_board_log\']=anchor(q,\'board\');q[\'anchor_alight_log\']=anchor(q,\'alight\')\n    cal=q.copy();cal[\'eff_dow\']=cal.dow;q[\'cal_dow_log_mean\']=lookup(profiles[56],cal,keys,\'board_mean\')\n    cal[\'eff_dow\']=6;q[\'sun_log_mean\']=lookup(profiles[56],cal,keys,\'board_mean\')\n    q[\'sun_minus_wkd_log\']=q.sun_log_mean-q.cal_dow_log_mean\n    q[\'growth_log\']=q.board_mean_28-q.board_mean_112\n    q[\'board_alight_asymmetry\']=q.board_mean_56-q.alight_mean_56\n    # Historical holiday elasticity, never future labels.\n    ordinary=h.loc[h.clean&(h.dow<5)].groupby([\'station_id\',\'hour\']).board_log.mean().to_frame(\'ordinary\')\n    holiday_mask=h.holiday&~h.oneoff&~((h.date.dt.month==5)&(h.date.dt.day==1))\n    if origin>=pd.Timestamp(\'2025-07-28\'):\n        holiday_mask&=~(h.station_id.isin([3110,3111])&(h.date<\'2025-07-01\'))\n    # Holiday observations are needed only for this explicit contrast feature;\n    # the ordinary side uses the same cleaned structural regime as the anchors.\n    holiday=h.loc[holiday_mask].groupby([\'station_id\',\'hour\']).board_log.mean().to_frame(\'holiday\')\n    q[\'holiday_elasticity\']=lookup(holiday,q,[\'station_id\',\'hour\'],\'holiday\')-lookup(ordinary,q,[\'station_id\',\'hour\'],\'ordinary\')\n    shares=recent.groupby([\'station_id\',\'hour\']).boardings.mean();shares=shares/shares.groupby(level=0).transform(\'sum\')\n    q[\'hour_share\']=lookup(shares.to_frame(\'share\'),q,[\'station_id\',\'hour\'],\'share\')\n    # Exact prior-year lookup, replace holiday/opening observations with adjacent weeks.\n    eligible=h.loc[~h.holiday&~h.oneoff&~((h.date.dt.month==5)&(h.date.dt.day==1))]\n    prior=eligible.set_index([\'date\',\'station_id\',\'hour\'])[[\'board_log\']]\n    year=np.full(len(q),np.nan)\n    for days in [364,357,371]:\n        dates=q[[\'date\',\'station_id\',\'hour\']].copy();dates[\'date\']=dates.date-pd.Timedelta(days=days)\n        val=lookup(prior,dates,[\'date\',\'station_id\',\'hour\'],\'board_log\')\n        val[dates.date>origin]=np.nan;year=np.where(np.isfinite(year),year,val)\n    q[\'year364_clean_log\']=year;q[\'year364_missing\']=~np.isfinite(year)\n    # Previous target month minus the 56-day window preceding origin one year earlier.\n    baseline_end=origin-pd.DateOffset(years=1)\n    base=clean.loc[(clean.date>baseline_end-pd.Timedelta(days=56))&(clean.date<=baseline_end)]\n    base_station=base.groupby([\'station_id\',\'rest\']).agg(value=(\'board_log\',\'mean\'),days=(\'date\',\'nunique\'))\n    base_line=base.groupby([\'canonical_line\',\'hour\',\'rest\']).board_log.mean().to_frame(\'value\')\n    q[\'yoy_seas_log_delta\']=np.nan;q[\'yoy_seas_fallback\']=1\n    for (year_now,month),indices in q.groupby([q.date.dt.year,q.date.dt.month]).groups.items():\n        part=q.loc[indices].copy();part[\'rest\']=part.is_rest_day\n        previous=clean.loc[(clean.date.dt.year==year_now-1)&(clean.date.dt.month==month)]\n        month_station=previous.groupby([\'station_id\',\'rest\']).board_log.mean().to_frame(\'value\')\n        month_line=previous.groupby([\'canonical_line\',\'hour\',\'rest\']).board_log.mean().to_frame(\'value\')\n        a=lookup(month_station,part,[\'station_id\',\'rest\'],\'value\')-lookup(base_station,part,[\'station_id\',\'rest\'],\'value\')\n        enough=part.station_id.map(base.groupby(\'station_id\').date.nunique()).to_numpy()>=14\n        b=lookup(month_line,part,[\'canonical_line\',\'hour\',\'rest\'],\'value\')-lookup(base_line,part,[\'canonical_line\',\'hour\',\'rest\'],\'value\')\n        use=enough&np.isfinite(a);delta=np.where(use,a,b)\n        q.loc[indices,\'yoy_seas_fallback\']=(~use).astype(int)\n        q.loc[indices,\'yoy_seas_log_delta\']=delta\n    q[\'yoy_seas_missing\']=q.yoy_seas_log_delta.isna();q[\'yoy_seas_log_delta\']=q.yoy_seas_log_delta.fillna(0)\n    for label,ids in [(\'previous\',q.station_id-1),(\'next\',q.station_id+1),(\'transfer\',q.station_id.map(TRANSFERS))]:\n        other=q[[\'station_id\',\'canonical_line\',\'hour\',\'eff_dow\']].copy();other[\'station_id\']=ids\n        other[\'canonical_line\']=ids.map(line_map)\n        valid=other.canonical_line.notna()\n        if label!=\'transfer\':valid&=other.canonical_line==q.canonical_line\n        value=anchor(other,\'board\');value[~valid]=np.nan\n        q[f\'{label}_anchor_log\']=value\n        q[f\'{label}_growth_log\']=lookup(profiles[28],other,keys,\'board_mean\')-lookup(profiles[112],other,keys,\'board_mean\')\n        q.loc[~valid,f\'{label}_growth_log\']=np.nan\n        q[f\'{label}_log_ratio\']=q.anchor_board_log-q[f\'{label}_anchor_log\']\n    q[\'station_hour\']=q.station_id*24+q.hour;q[\'station_eff_dow\']=q.station_id*7+q.eff_dow\n    q=add_history_extras(clean,q,origin)\n    q=add_domain_features(q,origin)\n    return q.drop(columns=\'date\').astype(\'float32\')\n\ndef add_history_extras(clean, q, origin):\n    """Daily prior-year curve; both sides exclude all post-origin observations."""\n    clean = clean.loc[clean.date <= origin].copy()\n    base_end = origin - pd.DateOffset(years=1)\n    base = clean.loc[clean.date.between(base_end-pd.Timedelta(days=55), base_end)]\n    keys = [\'station_id\', \'rest\']\n    base_station = base.groupby(keys).board_log.mean().to_frame(\'value\')\n    base_days = base.groupby(\'station_id\').date.nunique()\n    line_keys = [\'canonical_line\', \'hour\', \'rest\']\n    base_line = base.groupby(line_keys).board_log.mean().to_frame(\'value\')\n    delta = np.full(len(q), np.nan); fallback = np.ones(len(q), dtype=bool)\n    # Queries share only 123 dates; grouping before the window scan bounds memory.\n    for date, ix in q.groupby(\'date\', sort=False).groups.items():\n        center = date - pd.Timedelta(days=364)\n        past = clean.loc[clean.date.between(center-pd.Timedelta(days=7), min(center+pd.Timedelta(days=7), origin))]\n        part = q.loc[ix].copy(); part[\'rest\'] = part.is_rest_day\n        a = past.groupby(keys).board_log.mean().to_frame(\'value\')\n        b = past.groupby(line_keys).board_log.mean().to_frame(\'value\')\n        station_delta = lookup(a, part, keys, \'value\') - lookup(base_station, part, keys, \'value\')\n        line_delta = lookup(b, part, line_keys, \'value\') - lookup(base_line, part, line_keys, \'value\')\n        use = (part.station_id.map(base_days).to_numpy() >= 14) & np.isfinite(station_delta)\n        delta[ix] = np.where(use, station_delta, line_delta); fallback[ix] = ~use\n    q[\'yoy_smooth15_missing\'] = ~np.isfinite(delta)\n    q[\'yoy_smooth15_line_fallback\'] = fallback\n    q[\'yoy_smooth15_log_delta\'] = np.where(np.isfinite(delta), delta, 0)\n    weekday = clean.loc[clean.dow < 5]\n    for name, hours in [(\'am\', [7, 8]), (\'pm\', [17, 18, 19])]:\n        group = weekday.loc[weekday.hour.isin(hours)].groupby(\'station_id\')[[\'board_log\', \'alight_log\']].mean()\n        q[f\'commuter_asym_{name}\'] = q.station_id.map(group.board_log - group.alight_log)\n    return q\n\ndef partial_origins(cutoff,start=\'2025-02-11\'):\n    end=pd.Timestamp(cutoff)-pd.Timedelta(days=14)\n    origins=list(pd.date_range(start,end,freq=\'14D\'))\n    if origins and origins[-1]!=end:origins.append(end)\n    return origins\n\ndef sampled_pairs(train,cutoff,max_rows,seed=2026):\n    cutoff=pd.Timestamp(cutoff);origins=partial_origins(cutoff)\n    dates={d:idx.to_numpy() for d,idx in train.loc[(train.date<=cutoff)&~oneoff_mask(train)].groupby(\'date\').groups.items()}\n    rng=np.random.default_rng(seed);pairs=[]\n    for horizon in range(1,124):\n        rows=[];os=[];weights=[]\n        for oi,o in enumerate(origins):\n            d=o+pd.Timedelta(days=horizon)\n            if d>cutoff or d not in dates:continue\n            ids=dates[d];rows.extend(ids);os.extend([oi]*len(ids))\n            w=.5 if o<pd.Timestamp(\'2025-07-01\') else 1.5 if o<pd.Timestamp(\'2025-12-30\') else 2.5\n            weights.extend([w]*len(ids))\n        n=min(len(rows),max_rows//123+(horizon<=max_rows%123))\n        if n:\n            w=np.asarray(weights);ix=rng.choice(len(rows),n,replace=False,p=w/w.sum())\n            pairs.append(pd.DataFrame({\'row\':np.asarray(rows)[ix],\'origin_index\':np.asarray(os)[ix],\'horizon\':horizon}))\n    return pd.concat(pairs,ignore_index=True),origins\n\ndef training_matrix_v3(train,meta,cutoff,max_rows,seed=2026):\n    pairs,origins=sampled_pairs(train,cutoff,max_rows,seed);xs=[];ys=[];audit=[]\n    for oi,p in pairs.groupby(\'origin_index\',sort=True):\n        o=origins[oi];labels=train.loc[p.row].reset_index(drop=True)\n        assert labels.date.min()>o and labels.date.max()<=pd.Timestamp(cutoff)\n        x=make_features_v3(train,labels,meta,o)\n        y=np.column_stack([np.log1p(labels.boardings)-x.anchor_board_log,np.log1p(labels.alightings)-x.anchor_alight_log]).astype(\'float32\')\n        xs.append(x);ys.append(y)\n        audit.append({\'origin\':str(o.date()),\'rows\':len(p),\'label_max\':str(labels.date.max().date()),\'min_horizon\':int(p.horizon.min()),\'max_horizon\':int(p.horizon.max())})\n    return pd.concat(xs,ignore_index=True),np.concatenate(ys),audit,pairs.horizon.value_counts().sort_index().to_dict()\n', 'src/holiday_calendar.py': '"""South Korean known-in-advance holiday covariates, version 2026-06-30.\n\nOnly calendar facts/public declarations issued by the competition cutoff are\nencoded. No ridership, weather realization or event outcome is used. Sources,\npublication dates and reuse notes: reports/holiday_sources.md.\n\n`is_holiday` means a named national holiday, substitute or election holiday;\nordinary Sundays are represented separately by `is_weekend` / `is_rest_day`.\nPass the forecast origin as `as_of` for historical CV. This hides a temporary\nholiday/new designation until the cited public announcement was available.\nNo feature reveals that a not-yet-announced designation will occur.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom functools import lru_cache\nfrom typing import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nKNOWLEDGE_CUTOFF = "2026-06-30"\nCOVERAGE_START = "2025-01-01"\nCOVERAGE_END = "2026-12-31"\n\nHOLIDAY_TYPES = {\n    "none": 0, "new_year": 1, "seollal": 2, "independence": 3,\n    "children": 4, "buddha": 5, "children_and_buddha": 6, "memorial": 7,\n    "liberation": 8, "chuseok": 9, "national_foundation": 10,\n    "hangul": 11, "christmas": 12, "temporary": 13,\n    "presidential_election": 14, "local_election": 15,\n    "labor": 16, "constitution": 17,\n}\n\n\n@dataclass(frozen=True)\nclass Holiday:\n    name: str\n    kind: str\n    known_since: str\n    source: str\n    substitute: bool = False\n\n\ndef _h(name, kind, year, substitute=False):\n    # Conservative availability of the final annual calendar announcement.\n    known = "2024-06-20" if year == 2025 else "2025-06-30"\n    return Holiday(name, kind, known, f"annual_{year}", substitute)\n\n\nHOLIDAYS = {\n    "2025-01-01": _h("신정", "new_year", 2025),\n    "2025-01-27": Holiday("임시공휴일", "temporary", "2025-01-14", "temporary_20250127"),\n    "2025-01-28": _h("설날 전날", "seollal", 2025),\n    "2025-01-29": _h("설날", "seollal", 2025),\n    "2025-01-30": _h("설날 다음날", "seollal", 2025),\n    "2025-03-01": _h("삼일절", "independence", 2025),\n    "2025-03-03": _h("삼일절 대체공휴일", "independence", 2025, True),\n    "2025-05-05": _h("어린이날·부처님오신날", "children_and_buddha", 2025),\n    "2025-05-06": _h("부처님오신날 대체공휴일", "buddha", 2025, True),\n    "2025-06-03": Holiday("제21대 대통령선거", "presidential_election", "2025-04-08", "election_20250603"),\n    "2025-06-06": _h("현충일", "memorial", 2025),\n    "2025-08-15": _h("광복절", "liberation", 2025),\n    "2025-10-03": _h("개천절", "national_foundation", 2025),\n    "2025-10-05": _h("추석 전날", "chuseok", 2025),\n    "2025-10-06": _h("추석", "chuseok", 2025),\n    "2025-10-07": _h("추석 다음날", "chuseok", 2025),\n    "2025-10-08": _h("추석 대체공휴일", "chuseok", 2025, True),\n    "2025-10-09": _h("한글날", "hangul", 2025),\n    "2025-12-25": _h("기독탄신일", "christmas", 2025),\n    "2026-01-01": _h("신정", "new_year", 2026),\n    "2026-02-16": _h("설날 전날", "seollal", 2026),\n    "2026-02-17": _h("설날", "seollal", 2026),\n    "2026-02-18": _h("설날 다음날", "seollal", 2026),\n    "2026-03-01": _h("삼일절", "independence", 2026),\n    "2026-03-02": _h("삼일절 대체공휴일", "independence", 2026, True),\n    "2026-05-01": Holiday("노동절", "labor", "2026-04-06", "labor_2026"),\n    "2026-05-05": _h("어린이날", "children", 2026),\n    "2026-05-24": _h("부처님오신날", "buddha", 2026),\n    "2026-05-25": _h("부처님오신날 대체공휴일", "buddha", 2026, True),\n    "2026-06-03": _h("제9회 전국동시지방선거", "local_election", 2026),\n    "2026-06-06": _h("현충일", "memorial", 2026),\n    "2026-07-17": Holiday("제헌절", "constitution", "2026-02-03", "constitution_2026"),\n    "2026-08-15": _h("광복절", "liberation", 2026),\n    "2026-08-17": _h("광복절 대체공휴일", "liberation", 2026, True),\n    "2026-09-24": _h("추석 전날", "chuseok", 2026),\n    "2026-09-25": _h("추석", "chuseok", 2026),\n    "2026-09-26": _h("추석 다음날", "chuseok", 2026),\n    # 2026-09-28 is NOT a substitute holiday. Chuseok on Saturday alone does\n    # not add a substitute day; the official calendar says Sep 24-27 break.\n    "2026-10-03": _h("개천절", "national_foundation", 2026),\n    "2026-10-05": _h("개천절 대체공휴일", "national_foundation", 2026, True),\n    "2026-10-09": _h("한글날", "hangul", 2026),\n    "2026-12-25": _h("기독탄신일", "christmas", 2026),\n}\n\n\ndef holiday_records() -> pd.DataFrame:\n    """All encoded facts and provenance; not a model feature table."""\n    return pd.DataFrame([\n        {"date": date, "name": h.name, "kind": h.kind,\n         "holiday_type": HOLIDAY_TYPES[h.kind], "known_since": h.known_since,\n         "source_id": h.source, "is_substitute": int(h.substitute)}\n        for date, h in sorted(HOLIDAYS.items())\n    ])\n\n\n@lru_cache(maxsize=128)\ndef _known_records(as_of: str):\n    return {pd.Timestamp(d): h for d, h in HOLIDAYS.items() if h.known_since <= as_of}\n\n\ndef calendar_features(dates: Iterable, as_of=KNOWLEDGE_CUTOFF) -> pd.DataFrame:\n    """Return numeric int8 features in the input row order.\n\n    A Series input preserves its index. Other iterables receive a RangeIndex.\n    `as_of` is one forecast origin, not a vector; it must be <= 2026-06-30.\n    Scope is 2025-2026. Reject invalid/missing/out-of-range dates rather than\n    silently labeling an unsupported year\'s holidays as ordinary days.\n    """\n    origin = pd.Timestamp(as_of).normalize()\n    if pd.isna(origin) or origin > pd.Timestamp(KNOWLEDGE_CUTOFF):\n        raise ValueError(f"as_of must be a valid date <= {KNOWLEDGE_CUTOFF}")\n    index = dates.index if isinstance(dates, pd.Series) else None\n    d = pd.DatetimeIndex(pd.to_datetime(dates)).normalize()\n    if d.isna().any():\n        raise ValueError("calendar dates must not contain NaT")\n    if len(d) and (d.min() < pd.Timestamp(COVERAGE_START) or d.max() > pd.Timestamp(COVERAGE_END)):\n        raise ValueError(f"calendar only covers {COVERAGE_START} through {COVERAGE_END}")\n    known = _known_records(origin.strftime("%Y-%m-%d"))\n    # Unique-date lookup makes repeated station/hour rows inexpensive.\n    unique = d.unique()\n    lookup = {}\n    for day in unique:\n        h = known.get(day)\n        before = day - pd.Timedelta(days=1)\n        after = day + pd.Timedelta(days=1)\n        weekend = day.dayofweek >= 5\n        rest = weekend or h is not None\n        prev_rest = before.dayofweek >= 5 or before in known\n        next_rest = after.dayofweek >= 5 or after in known\n        kind = h.kind if h else "none"\n        lookup[day] = (\n            int(h is not None), HOLIDAY_TYPES[kind], int(h.substitute if h else False),\n            int(kind in {"temporary", "presidential_election"}),\n            int(kind in {"presidential_election", "local_election"}),\n            int(kind in {"seollal", "buddha", "children_and_buddha", "chuseok"}),\n            int(day.month == 5 and day.day == 1), int(weekend), int(rest),\n            int(after in known), int(before in known),\n            int(not rest and prev_rest and next_rest),\n        )\n    columns = [\n        "is_holiday", "holiday_type", "is_substitute_holiday", "is_temporary_holiday",\n        "is_election_day", "is_lunar_holiday", "is_labor_day", "calendar_is_weekend",\n        "is_rest_day", "is_holiday_eve", "is_after_holiday", "is_bridge_day",\n    ]\n    if len(d):\n        values = np.asarray([lookup[x] for x in d], dtype=np.int8)\n    else:\n        values = np.empty((0, len(columns)), dtype=np.int8)\n    return pd.DataFrame(values, columns=columns, index=index)\n\n\nif __name__ == "__main__":\n    print(holiday_records().to_string(index=False))\n', 'src/metrics.py': '"""Forecast metrics and calendar-aware diagnostics for SINCON.\n\nRMSLE is the official metric in the user-provided competition rules. ``smape`` uses\n200 * |prediction - target| / (|target| + |prediction|), with 0/0 := 0.\nMASE scales are calculated only from the supplied training history using\nexact seven-calendar-day matches within station/hour, never adjacent rows.\n"""\nfrom __future__ import annotations\n\nfrom typing import Sequence\n\nimport numpy as np\nimport pandas as pd\n\n\nSUPPORTED_METRICS = ("mae", "rmse", "smape", "rmsle")\nGROUP_COLS = ("station_id", "hour")\n\n\ndef _arrays(y_true, y_pred):\n    y = np.asarray(y_true, dtype=np.float64)\n    p = np.asarray(y_pred, dtype=np.float64)\n    if y.ndim != 1 or p.ndim != 1 or y.shape != p.shape or y.size == 0:\n        raise ValueError("Targets and predictions must be nonempty 1-D arrays of equal length.")\n    if not np.isfinite(y).all() or not np.isfinite(p).all():\n        raise ValueError("Targets and predictions must contain only finite numbers.")\n    return y, p\n\n\ndef regression_metrics(y_true, y_pred) -> dict[str, float]:\n    """Return unweighted row-level metrics; undefined values are NaN.\n\n    RMSLE is undefined for negative values. No implicit clipping is performed.\n    ``rmse_mae_ratio`` is undefined for perfect predictions (MAE == 0).\n    MBE is prediction minus truth, so positive means overprediction.\n    """\n    y, p = _arrays(y_true, y_pred)\n    err = p - y\n    abs_err = np.abs(err)\n    mae = float(abs_err.mean())\n    rmse = float(np.sqrt(np.mean(err * err)))\n    denom = np.abs(y) + np.abs(p)\n    smape_terms = np.divide(200.0 * abs_err, denom, out=np.zeros_like(err), where=denom > 0)\n    rmsle = float(np.sqrt(np.mean((np.log1p(p) - np.log1p(y)) ** 2))) if (y >= 0).all() and (p >= 0).all() else float("nan")\n    return {\n        "mae": mae,\n        "rmse": rmse,\n        "smape": float(smape_terms.mean()),\n        "rmsle": rmsle,\n        "mbe": float(err.mean()),\n        "rmse_mae_ratio": rmse / mae if mae > 0 else float("nan"),\n    }\n\n\ndef score(y_true, y_pred, metric: str = "rmsle") -> float:\n    """Select an objective explicitly; default is official RMSLE, all minimize."""\n    metric = metric.lower()\n    if metric not in SUPPORTED_METRICS:\n        raise ValueError(f"Unknown metric {metric!r}; choose {SUPPORTED_METRICS}.")\n    # Ensemble weight search calls this repeatedly: compute only its objective.\n    y, p = _arrays(y_true, y_pred)\n    if metric == "mae":\n        result = float(np.abs(p - y).mean())\n    elif metric == "rmse":\n        result = float(np.sqrt(np.mean((p - y) ** 2)))\n    elif metric == "smape":\n        denominator = np.abs(y) + np.abs(p)\n        result = float(np.divide(200.0 * np.abs(p - y), denominator, out=np.zeros_like(y), where=denominator > 0).mean())\n    else:\n        result = float(np.sqrt(np.mean((np.log1p(p) - np.log1p(y)) ** 2))) if (y >= 0).all() and (p >= 0).all() else float("nan")\n    if not np.isfinite(result):\n        raise ValueError(f"{metric} is undefined for these targets/predictions.")\n    return result\n\n\ndef _calendar_frame(frame: pd.DataFrame, group_cols: Sequence[str], target: str) -> pd.DataFrame:\n    needed = [*group_cols, "date", target]\n    missing = set(needed) - set(frame.columns)\n    if missing:\n        raise ValueError(f"Missing columns: {sorted(missing)}")\n    result = frame[needed].copy()\n    result["date"] = pd.to_datetime(result["date"], errors="raise").dt.normalize()\n    if result[needed].isna().any().any():\n        raise ValueError("Calendar keys and target cannot contain missing values.")\n    if result.duplicated([*group_cols, "date"]).any():\n        raise ValueError("Duplicate station/hour/date keys are not valid for calendar diagnostics.")\n    values = pd.to_numeric(result[target], errors="raise").to_numpy(dtype=float)\n    if not np.isfinite(values).all():\n        raise ValueError("Target values must be finite.")\n    result[target] = values\n    return result\n\n\ndef seasonal_scales(\n    train: pd.DataFrame,\n    seasonality_days: int = 7,\n    target: str = "boardings",\n    group_cols: Sequence[str] = GROUP_COLS,\n) -> pd.DataFrame:\n    """Training-only MASE denominator and exact seasonal pair counts per series.\n\n    Zero denominators are retained as zero so the caller can distinguish a\n    constant series from a series with no historical seasonal pairs (NaN).\n    """\n    if not isinstance(seasonality_days, int) or seasonality_days <= 0:\n        raise ValueError("seasonality_days must be a positive integer.")\n    t = _calendar_frame(train, group_cols, target)\n    previous = t.rename(columns={target: "seasonal_value"})\n    previous["date"] += pd.Timedelta(days=seasonality_days)\n    paired = t.merge(previous, on=[*group_cols, "date"], how="left", validate="one_to_one")\n    paired["seasonal_abs_difference"] = (paired[target] - paired["seasonal_value"]).abs()\n    return paired.groupby(list(group_cols), observed=True, dropna=False).agg(\n        mase_scale=("seasonal_abs_difference", "mean"),\n        seasonal_pairs=("seasonal_abs_difference", "count"),\n    ).reset_index()\n\n\ndef residual_autocorrelation(\n    predictions: pd.DataFrame,\n    lags: Sequence[int] = (1, 7, 28),\n    group_cols: Sequence[str] = GROUP_COLS,\n) -> pd.DataFrame:\n    """Pearson residual correlation at exact calendar-day lags, per series.\n\n    Input needs date/group columns and residual=prediction-truth. This is a\n    pairwise calendar-lag correlation, not an ACF across concatenated stations.\n    Fewer than three matched pairs or zero variance produces NaN.\n    """\n    t = _calendar_frame(predictions, group_cols, "residual")\n    records = []\n    for lag in lags:\n        if not isinstance(lag, int) or lag <= 0:\n            raise ValueError("ACF lags must be positive integer calendar days.")\n        previous = t.rename(columns={"residual": "residual_lag"})\n        previous["date"] += pd.Timedelta(days=lag)\n        paired = t.merge(previous, on=[*group_cols, "date"], how="inner", validate="one_to_one")\n        values_by_group = {}\n        for keys, frame in paired.groupby(list(group_cols), observed=True, dropna=False):\n            keys = keys if isinstance(keys, tuple) else (keys,)\n            left = frame["residual"].to_numpy()\n            right = frame["residual_lag"].to_numpy()\n            corr = float(np.corrcoef(left, right)[0, 1]) if len(frame) >= 3 and left.std() > 0 and right.std() > 0 else float("nan")\n            values_by_group[keys] = (len(frame), corr)\n        for keys in t[list(group_cols)].drop_duplicates().itertuples(index=False, name=None):\n            n, corr = values_by_group.get(keys, (0, float("nan")))\n            records.append({**dict(zip(group_cols, keys)), "lag_days": lag, "pairs": n, "acf": corr})\n    return pd.DataFrame.from_records(records)\n\n\ndef forecast_diagnostics(\n    train: pd.DataFrame,\n    valid: pd.DataFrame,\n    predictions,\n    origin=None,\n    horizon_col: str = "horizon",\n    target: str = "boardings",\n    group_cols: Sequence[str] = GROUP_COLS,\n    seasonality_days: int = 7,\n) -> dict:\n    """Return summary and DataFrames for one fixed-origin forecast fold.\n\n    ``train`` must end no later than origin; all validation dates must follow\n    origin. Predictions must have the same row order as ``valid``. The summary\n    contains two MASE aggregations: mean per-series MASE (macro), and the mean\n    per-row absolute error divided by that row\'s training-series scale. Both\n    exclude unavailable/zero scales and report their coverage explicitly.\n    Per-step metrics use calendar-day horizon, not row position or hour number.\n    """\n    t = _calendar_frame(train, group_cols, target)\n    v = _calendar_frame(valid, group_cols, target)\n    if t.empty or v.empty:\n        raise ValueError("Training and validation must both be nonempty.")\n    cutoff = pd.Timestamp(origin).normalize() if origin is not None else t["date"].max()\n    if t["date"].max() > cutoff:\n        raise ValueError("Training includes observations after forecast origin (leakage).")\n    if v["date"].min() <= cutoff:\n        raise ValueError("Validation must lie strictly after forecast origin.")\n    y, p = _arrays(v[target], predictions)\n    v["prediction"] = p\n    v["residual"] = p - y\n    v["absolute_error"] = np.abs(p - y)\n    actual_horizon = (v["date"] - cutoff).dt.days.to_numpy()\n    if horizon_col in valid:\n        supplied_horizon = pd.to_numeric(valid[horizon_col], errors="raise").to_numpy()\n        if not np.array_equal(supplied_horizon, actual_horizon):\n            raise ValueError("Supplied horizon does not equal date minus forecast origin.")\n    v["horizon"] = actual_horizon\n    scales = seasonal_scales(t, seasonality_days, target, group_cols)\n    v = v.merge(scales, on=list(group_cols), how="left", sort=False, validate="many_to_one")\n    usable_scale = v["mase_scale"].where(v["mase_scale"] > 0)\n    v["scaled_absolute_error"] = v["absolute_error"] / usable_scale\n    entity_rows = []\n    for keys, frame in v.groupby(list(group_cols), observed=True, dropna=False):\n        keys = keys if isinstance(keys, tuple) else (keys,)\n        stats = regression_metrics(frame[target], frame["prediction"])\n        entity_rows.append({\n            **dict(zip(group_cols, keys)),\n            "rows": len(frame),\n            **stats,\n            "mase_scale": frame["mase_scale"].iloc[0],\n            "seasonal_pairs": frame["seasonal_pairs"].iloc[0],\n            "mase": frame["scaled_absolute_error"].mean(),\n        })\n    per_entity = pd.DataFrame.from_records(entity_rows)\n    per_step = pd.DataFrame.from_records([\n        {"horizon": int(h), "rows": len(frame), **regression_metrics(frame[target], frame["prediction"]), "mase": frame["scaled_absolute_error"].mean()}\n        for h, frame in v.groupby("horizon", sort=True)\n    ])\n    acf = residual_autocorrelation(v, group_cols=group_cols)\n    summary = {\n        **regression_metrics(y, p),\n        "mase_macro": float(per_entity["mase"].mean()),\n        "mase_row_weighted": float(v["scaled_absolute_error"].mean()),\n        "mase_series_coverage": float(per_entity["mase"].notna().mean()),\n        "mase_row_coverage": float(v["scaled_absolute_error"].notna().mean()),\n        "mase_undefined_series": int(per_entity["mase"].isna().sum()),\n        "mase_seasonality_days": seasonality_days,\n        "origin": str(cutoff.date()),\n        "forecast_rows": len(v),\n    }\n    for lag, frame in acf.groupby("lag_days"):\n        finite = frame["acf"].dropna()\n        summary[f"acf_{lag}d_macro"] = float(finite.mean())\n        summary[f"acf_{lag}d_abs_gt_0_1_fraction"] = float((finite.abs() > 0.1).mean()) if len(finite) else float("nan")\n        summary[f"acf_{lag}d_defined_series"] = int(len(finite))\n    return {"summary": summary, "per_entity": per_entity, "residual_acf": acf, "per_step": per_step}\n', 'src/models.py': '"""Scratch-trained CPU tree baselines and a small tabular neural baseline."""\nimport numpy as np\nimport sys\nfrom .models_v2 import ResidualTree\n\ndef lightgbm_parameters(config,large=True):\n    """Common parameters, with explicit overrides from a saved tuning plan."""\n    params=dict(objective=\'regression\',n_estimators=220 if large else 150,\n                learning_rate=.055,num_leaves=63 if large else 31,min_child_samples=80,\n                reg_lambda=3,colsample_bytree=.9,subsample=.9,subsample_freq=1,\n                random_state=config[\'seed\'],n_jobs=config[\'threads\'],verbosity=-1,linear_tree=False)\n    overrides=config.get(\'lightgbm_params\',{})\n    allowed={\'n_estimators\',\'learning_rate\',\'num_leaves\',\'max_depth\',\'min_child_samples\',\n             \'min_child_weight\',\'reg_lambda\',\'reg_alpha\',\'colsample_bytree\',\'subsample\',\n             \'subsample_freq\',\'min_split_gain\'}\n    if set(overrides)-allowed:raise ValueError(f\'Unsupported LightGBM overrides: {set(overrides)-allowed}\')\n    params.update(overrides)\n    return params\n\nclass NeuralRegressor:\n    """MLP with training-only scaling and missing masks; not TabM/RevIN."""\n    def __init__(self,seed=2026,threads=4,device=\'cpu\',epochs=10):\n        self.seed,self.threads,self.device,self.epochs=seed,threads,device,epochs\n\n    def fit(self,x,y):\n        import torch\n        # Apple OpenMP runtimes used by tree wheels can deadlock with Torch\'s\n        # indexed tensor kernels. One Torch CPU thread avoids this interaction.\n        torch.manual_seed(self.seed); torch.set_num_threads(1 if sys.platform==\'darwin\' else self.threads)\n        if self.device.startswith(\'cuda\') and not torch.cuda.is_available():\n            raise RuntimeError(\'CUDA requested but unavailable\')\n        self.columns=[c for c in x.columns if c in [\'station_id\',\'hour\',\'dow\',\'canonical_line\',\'horizon\',\'month\',\'date_index\',\'weekend\',\'hour_sin\',\'hour_cos\',\'dow_sin\',\'dow_cos\',\'day_of_year_sin\',\'day_of_year_cos\',\'history_days\',\'cold_start\',\'origin_lag_1\',\'origin_lag_7\',\'origin_lag_28\',\'seasonal7\',\'seasonal7_alight\',\'recent_mean\',\'recent_std\',\'recent_alight\',\'baseline\',\'is_holiday\',\'holiday_type\',\'is_holiday_eve\',\'is_after_holiday\']]\n        a=x[self.columns].to_numpy(dtype=\'float32\')\n        self.fill=np.nanmedian(a,axis=0)\n        self.fill=np.nan_to_num(self.fill,nan=0).astype(\'float32\')\n        filled=np.where(np.isnan(a),self.fill,a)\n        self.mean=filled.mean(axis=0); self.std=filled.std(axis=0).clip(min=1)\n        z=self.transform(x)\n        # y is log1p(count). Squared error is exactly the square of RMSLE.\n        target=y-np.log1p(x.baseline.to_numpy())\n        self.net=torch.nn.Sequential(torch.nn.Linear(z.shape[1],96),torch.nn.ReLU(),torch.nn.Linear(96,64),torch.nn.ReLU(),torch.nn.Linear(64,1)).to(self.device)\n        optimizer=torch.optim.AdamW(self.net.parameters(),lr=0.001,weight_decay=0.001)\n        use_amp=self.device.startswith(\'cuda\')\n        scaler=torch.amp.GradScaler(\'cuda\',enabled=use_amp)\n        rng=np.random.default_rng(self.seed)\n        n=min(len(z),160000)\n        idx=rng.choice(len(z),n,replace=False)\n        xt=torch.from_numpy(z[idx]); yt=torch.tensor(target[idx],dtype=torch.float32)\n        self.losses=[]\n        self.net.train()\n        for epoch in range(self.epochs):\n            perm=torch.randperm(n); losses=[]\n            for ids in perm.split(2048):\n                xb=xt[ids].to(self.device); yb=yt[ids].to(self.device)\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=\'cuda\',dtype=torch.float16,enabled=use_amp):\n                    pred=self.net(xb).squeeze(1);loss=torch.nn.functional.mse_loss(pred,yb)\n                if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite neural loss\')\n                scaler.scale(loss).backward()\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(self.net.parameters(),1.0,error_if_nonfinite=True)\n                scaler.step(optimizer);scaler.update()\n                losses.append(float(loss.detach().cpu()))\n            self.losses.append(float(np.sqrt(np.mean(losses))))\n        self.net.to(\'cpu\');self.net.eval()\n        return self\n\n    def transform(self,x):\n        a=x[self.columns].to_numpy(dtype=\'float32\')\n        mask=np.isnan(a)\n        z=(np.where(mask,self.fill,a)-self.mean)/self.std\n        return np.concatenate([z,mask.astype(\'float32\')],axis=1).astype(\'float32\')\n\n    def predict(self,x):\n        import torch\n        z=self.transform(x);parts=[]\n        self.net.eval()\n        with torch.no_grad():\n            for start in range(0,len(x),8192):\n                parts.append(self.net(torch.from_numpy(z[start:start+8192])).squeeze(1).numpy())\n        return np.log1p(x.baseline.to_numpy())+np.concatenate(parts)\n\nclass Model:\n    def __init__(self,name,config):\n        self.name,self.config=name,config\n\n    def _frame(self,x):\n        if self.name==\'catboost\':\n            z=x.copy()\n            for col in [\'station_id\',\'canonical_line\',\'hour\',\'dow\']:z[col]=z[col].astype(\'int32\').astype(str)\n            return z\n        return x\n\n    def fit(self,x,y):\n        name=self.name;cfg=self.config\n        if name in [\'seasonal7\',\'weekday_mean\']:return self\n        if cfg[\'training_loss\']!=\'log_mse\':\n            raise ValueError(\'SINCON official RMSLE requires this implementation to train log_mse.\')\n        y=np.log1p(y)\n        if name.startswith(\'lightgbm\'):\n            from lightgbm import LGBMRegressor\n            large=name.endswith(\'large\')\n            self.estimator=LGBMRegressor(**lightgbm_parameters(cfg,large))\n            self.estimator.fit(x,y,categorical_feature=[\'station_id\',\'canonical_line\',\'hour\',\'dow\'])\n        elif name==\'catboost\':\n            from catboost import CatBoostRegressor\n            self.estimator=CatBoostRegressor(iterations=220,depth=7,learning_rate=0.07,loss_function=\'RMSE\',l2_leaf_reg=5,random_seed=cfg[\'seed\'],thread_count=cfg[\'threads\'],verbose=False,allow_writing_files=False)\n            self.estimator.fit(self._frame(x),y,cat_features=[\'station_id\',\'canonical_line\',\'hour\',\'dow\'])\n        elif name==\'xgboost\':\n            from xgboost import XGBRegressor\n            self.estimator=XGBRegressor(objective=\'reg:squarederror\',n_estimators=180,max_depth=6,learning_rate=0.06,min_child_weight=30,reg_lambda=5,subsample=0.9,colsample_bytree=0.85,tree_method=\'hist\',n_jobs=cfg[\'threads\'],random_state=cfg[\'seed\'])\n            self.estimator.fit(x,y)\n        elif name==\'mlp\':\n            self.estimator=NeuralRegressor(cfg[\'seed\'],cfg[\'threads\'],cfg[\'device\']).fit(x,y)\n        else:raise ValueError(f\'Unknown model {name}\')\n        return self\n\n    def predict(self,x):\n        if self.name in [\'seasonal7\',\'weekday_mean\']:return x[self.name].to_numpy(dtype=\'float64\')\n        pred=np.expm1(np.maximum(0,self.estimator.predict(self._frame(x)))).astype(\'float64\')\n        if not np.isfinite(pred).all():raise ValueError(f\'{self.name}: nonfinite predictions\')\n        return pred\n', 'src/models_v2.py': '"""Residual tree wrappers; torch is isolated to the TabM worker process."""\nimport os\nimport random\nimport numpy as np\nimport pandas as pd\nfrom .features_v2 import CATS\n\ndef seed_everything(seed):\n    random.seed(seed);np.random.seed(seed)\n\ndef devices():\n    # Probe in a fresh interpreter: loading macOS torch and LightGBM OpenMP\n    # runtimes together can crash native dataset construction.\n    import subprocess,sys,json\n    result=subprocess.run([sys.executable,\'-c\',"import torch,json;print(json.dumps({\'cuda_count\':torch.cuda.device_count(),\'mps\':torch.backends.mps.is_available()}))"],capture_output=True,text=True,check=True)\n    return json.loads(result.stdout)\n\nclass ResidualTree:\n    def __init__(self,name,cfg,seed=2026,iterations=None,hardware=None):\n        self.name=name;self.cfg=cfg;self.seed=seed;self.iterations=iterations\n        self.hardware=hardware or {\'cuda_count\':0,\'mps\':False}\n    def frame(self,x,fit=False):\n        x=x.copy();self.cats=[c for c in CATS if c in x]\n        if self.name==\'catboost\':\n            for c in self.cats:x[c]=x[c].fillna(-1).astype(\'int32\').astype(str)\n        else:\n            if fit:self.categories={c:sorted(int(v) for v in x[c].dropna().unique()) for c in self.cats}\n            for c in self.cats:x[c]=pd.Categorical(x[c],categories=self.categories[c])\n        return x\n    def fit(self,x,y,xvalid=None,yvalid=None):\n        seed_everything(self.seed);x=self.frame(x,fit=True)\n        xv=None if xvalid is None else self.frame(xvalid)\n        p=dict(self.cfg[self.name]);threads=self.cfg[\'threads\'];patience=self.cfg[\'patience\']\n        gpu=self.hardware[\'cuda_count\']>0 and self.cfg[\'device\']!=\'cpu\'\n        self.device=\'cpu\';self.history={}\n        if self.name==\'lightgbm\':\n            import lightgbm as lgb\n            if self.iterations is not None:p[\'n_estimators\']=self.iterations\n            self.model=lgb.LGBMRegressor(**p,objective=\'regression_l2\',random_state=self.seed,n_jobs=threads,verbosity=-1,deterministic=True,force_col_wise=True)\n            callbacks=[lgb.log_evaluation(100),lgb.record_evaluation(self.history)]\n            if xv is not None:callbacks.append(lgb.early_stopping(patience,verbose=True))\n            self.model.fit(x,y[:,0],eval_set=None if xv is None else [(xv,yvalid[:,0])],eval_metric=\'rmse\',callbacks=callbacks,categorical_feature=self.cats)\n            self.best_iteration=int(self.model.best_iteration_ or p[\'n_estimators\'])\n        elif self.name==\'catboost\':\n            from catboost import CatBoostRegressor\n            if self.iterations is not None:p[\'iterations\']=self.iterations\n            if gpu:p.update(task_type=\'GPU\',devices=\'0\');self.device=\'cuda:0\'\n            self.model=CatBoostRegressor(**p,loss_function=\'RMSE\',random_seed=self.seed,thread_count=threads,allow_writing_files=False,verbose=100)\n            self.model.fit(x,y[:,0],cat_features=self.cats,eval_set=None if xv is None else (xv,yvalid[:,0]),early_stopping_rounds=patience if xv is not None else None)\n            self.best_iteration=int(self.model.get_best_iteration()+1) if xv is not None else int(p[\'iterations\'])\n            self.history=self.model.get_evals_result()\n        elif self.name==\'xgboost\':\n            from .xgb_compat import InferenceConsistentXGBRegressor as XGBRegressor\n            if self.iterations is not None:p[\'n_estimators\']=self.iterations\n            device=f\'cuda:{min(1,self.hardware["cuda_count"]-1)}\' if gpu else \'cpu\';self.device=device\n            self.model=XGBRegressor(**p,objective=\'reg:squarederror\',eval_metric=\'rmse\',random_state=self.seed,n_jobs=threads,device=device,early_stopping_rounds=patience if xv is not None else None)\n            self.model.fit(x,y[:,0],eval_set=None if xv is None else [(xv,yvalid[:,0])],verbose=100)\n            self.best_iteration=int(self.model.best_iteration+1) if xv is not None else int(p[\'n_estimators\'])\n            self.history=self.model.evals_result() if xv is not None else {}\n        else:raise ValueError(self.name)\n        return self\n    def predict_log(self,x):\n        residual=self.model.predict(self.frame(x))\n        return np.maximum(0,x.anchor_board_log.to_numpy(dtype=\'float64\')+residual)\n    def to_cpu(self):\n        if self.name==\'xgboost\':self.model.set_params(device=\'cpu\')\n        self.device=\'cpu\';return self\n', 'src/models_v3.py': '"""v3 regularized trees; v2 checkpoints and feature schemas remain unchanged."""\nfrom .models_v2 import ResidualTree as ResidualTreeV2, devices\n\nclass ResidualTree(ResidualTreeV2):\n    def frame(self, x, fit=False):\n        if self.name in (\'lightgbm\', \'xgboost\'):\n            x = x.drop(columns=[\'station_hour\', \'station_eff_dow\'], errors=\'ignore\')\n        return super().frame(x, fit=fit)\n', 'src/nn_preprocessor.py': '"""Fold-fitted robust continuous scaling, missing masks and unknown categories."""\nimport numpy as np\nfrom .features_v2 import CATS\n\nclass NNPreprocessor:\n    def fit(self,x):\n        self.cats=[c for c in CATS if c in x];self.cont=[c for c in x if c not in self.cats]\n        a=x[self.cont].to_numpy(dtype=\'float32\')\n        self.median=np.nan_to_num(np.nanmedian(a,axis=0),nan=0)\n        self.iqr=np.nan_to_num(np.nanpercentile(a,75,axis=0)-np.nanpercentile(a,25,axis=0),nan=0)\n        self.maps={c:{int(v):i+1 for i,v in enumerate(sorted(x[c].dropna().unique()))} for c in self.cats}\n        self.cardinalities=[len(self.maps[c])+1 for c in self.cats]\n        return self\n    def transform(self,x):\n        a=x[self.cont].to_numpy(dtype=\'float32\');mask=~np.isfinite(a)\n        a=np.clip((np.where(mask,self.median,a)-self.median)/(self.iqr+1e-5),-10,10)\n        continuous=np.concatenate([a,mask.astype(\'float32\')],axis=1).astype(\'float32\')\n        categorical=np.column_stack([x[c].map(self.maps[c]).fillna(0).to_numpy(dtype=\'int64\') for c in self.cats])\n        return continuous,categorical\n', 'src/nn_preprocessor_v3.py': '"""Origin-anchor imputation for prior-year log levels, retaining missing masks."""\nimport numpy as np\nfrom .nn_preprocessor import NNPreprocessor\nfrom .features_v2 import CATS\n\nclass NNPreprocessorV3(NNPreprocessor):\n    def values(self, x):\n        a = x[self.cont].to_numpy(dtype=\'float32\', copy=True)\n        mask = ~np.isfinite(a)\n        for j, name in enumerate(self.cont):\n            # Only level features: delta/missing flags must not receive a log level.\n            if name in (\'year364_clean_log\', \'year364_board_log\', \'year364_alight_log\'):\n                anchor = \'anchor_alight_log\' if \'alight\' in name else \'anchor_board_log\'\n                if anchor in x:\n                    a[mask[:, j], j] = x[anchor].to_numpy()[mask[:, j]]\n        return a, mask\n\n    def fit(self, x):\n        self.cats = [c for c in CATS if c in x]\n        self.cont = [c for c in x if c not in self.cats]\n        a, _ = self.values(x)\n        a[~np.isfinite(a)] = np.nan\n        self.median = np.nan_to_num(np.nanmedian(a, axis=0), nan=0)\n        self.iqr = np.nan_to_num(np.nanpercentile(a, 75, axis=0) - np.nanpercentile(a, 25, axis=0), nan=0)\n        # Unseen event flags must not activate arbitrary network weights.\n        # Preserve rare but nonconstant columns even when their IQR is zero.\n        filled = np.where(np.isfinite(a), a, self.median)\n        self.constant = np.ptp(filled, axis=0) == 0\n        self.maps = {c: {int(v): i + 1 for i, v in enumerate(sorted(x[c].dropna().unique()))} for c in self.cats}\n        self.cardinalities = [len(self.maps[c]) + 1 for c in self.cats]\n        return self\n\n    def transform(self, x):\n        a, original_mask = self.values(x)\n        a = np.clip((np.where(np.isfinite(a), a, self.median) - self.median) / (self.iqr + 1e-5), -10, 10)\n        a[:, self.constant] = 0\n        num = np.concatenate([a, original_mask.astype(\'float32\')], axis=1).astype(\'float32\')\n        cat = np.column_stack([x[c].map(self.maps[c]).fillna(0).to_numpy(dtype=\'int64\') for c in self.cats])\n        return num, cat\n', 'src/pipeline.py': '"""Reproducible EDA -> fixed temporal CV -> fit -> verified CSV pipeline."""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport resource\nimport sys\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom .features import load_data,make_features,training_matrix\nfrom .models import Model\nfrom .ensemble import fit_ensemble,predict_ensemble\nfrom .metrics import regression_metrics,forecast_diagnostics\nfrom .validate_submission import validate_submission\n\nROOT=Path(__file__).resolve().parents[1]\n\ndef clean(value):\n    if isinstance(value,dict):return {str(k):clean(v) for k,v in value.items()}\n    if isinstance(value,(list,tuple)):return [clean(v) for v in value]\n    if isinstance(value,(np.integer,)):return int(value)\n    if isinstance(value,(np.floating,float)):return float(value) if np.isfinite(value) else None\n    if isinstance(value,(np.bool_,)):return bool(value)\n    return value\n\ndef save_json(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    path.write_text(json.dumps(clean(value),ensure_ascii=False,indent=2,allow_nan=False)+\'\\n\',encoding=\'utf-8\')\n\ndef environment():\n    versions={}\n    for name in [\'numpy\',\'pandas\',\'polars\',\'scikit-learn\',\'scipy\',\'lightgbm\',\'catboost\',\'xgboost\',\'torch\',\'joblib\']:\n        try:versions[name]=importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:versions[name]=None\n    return {\'python\':sys.version,\'platform\':platform.platform(),\'packages\':versions}\n\ndef diagnostics(history,valid,prediction,origin,path):\n    result=forecast_diagnostics(history.loc[history.date<=origin],valid,prediction,origin=origin)\n    for key in [\'per_entity\',\'residual_acf\',\'per_step\']:\n        result[key].to_csv(path/f\'{key}.csv\',index=False)\n    monthly=[]\n    v=valid.copy();v[\'prediction\']=prediction\n    for month,rows in v.groupby(v.date.dt.to_period(\'M\')):\n        monthly.append({\'month\':str(month),\'rows\':len(rows),**regression_metrics(rows.boardings,rows.prediction)})\n    pd.DataFrame(monthly).to_csv(path/\'per_month.csv\',index=False)\n    # Day-block bootstrap reflects the competition\'s date-level public/private grouping.\n    daily=v.assign(log_se=(np.log1p(v.prediction)-np.log1p(v.boardings))**2).groupby(\'date\').log_se.agg([\'sum\',\'size\'])\n    rng=np.random.default_rng(2026)\n    values=[]\n    for _ in range(500):\n        ix=rng.integers(0,len(daily),size=len(daily));sample=daily.iloc[ix]\n        values.append(float(np.sqrt(sample[\'sum\'].sum()/sample[\'size\'].sum())))\n    result[\'summary\'][\'day_bootstrap_rmsle_ci95\']=[float(np.quantile(values,.025)),float(np.quantile(values,.975))]\n    result[\'summary\'][\'day_bootstrap_note\']=\'500 date-block resamples; descriptive conditional interval, not a guarantee for future months.\'\n    save_json(path/\'summary.json\',result[\'summary\'])\n    return result[\'summary\']\n\ndef export_predictions(data,predictions,output,reference=None):\n    output.mkdir(parents=True,exist_ok=True)\n    full=data[\'test\'][[\'example_id\']].copy();full[\'boardings\']=predictions\n    full.to_csv(output/\'submission.csv\',index=False,float_format=\'%.8f\')\n    # Keep original stale template as evidence; this secondary file is diagnostic only.\n    subset=data[\'sample_submission\'][[\'example_id\']].merge(full,on=\'example_id\',how=\'left\',validate=\'one_to_one\')\n    subset.to_csv(output/\'template_diagnostic_DO_NOT_SUBMIT.csv\',index=False,float_format=\'%.8f\')\n    full_report=validate_submission(output/\'submission.csv\',data[\'test\'],data[\'sample_submission\'],scope=\'test\',reference_predictions=reference)\n    full_report[\'official_expected_rows\']=157194\n    full_report[\'official_scope_source\']=\'User-provided Overview/Data/Rules: 123 days and all test IDs; local sample is stale.\'\n    if len(full)!=157194:raise ValueError(\'Official SINCON row count must be 157194\')\n    save_json(output/\'submission_validation.json\',full_report)\n    rich=data[\'test\'].copy();rich[\'boardings\']=predictions\n    rich.to_csv(output/\'predictions_with_keys.csv\',index=False,float_format=\'%.8f\')\n    return full_report\n\ndef run(args):\n    import joblib\n    start=time.perf_counter()\n    cfg=json.loads(args.config.read_text())\n    if cfg[\'selection_metric\']!=\'rmsle\' or cfg[\'training_loss\']!=\'log_mse\':raise ValueError(\'Official metric/loss configuration mismatch\')\n    if len(cfg[\'fold_origins\'])!=len(cfg[\'fold_roles\']):raise ValueError(\'Fold role mismatch\')\n    if cfg[\'fold_roles\']!=[\'development\',\'development\',\'audit\']:raise ValueError(\'Expected two development folds followed by one evaluation fold\')\n    data=load_data(args.data_dir);train=data[\'train\'];meta=data[\'metaData\']\n    output=args.output;output.mkdir(parents=True,exist_ok=True)\n    save_json(output/\'config_resolved.json\',cfg)\n    save_json(output/\'environment.json\',environment())\n    fingerprints={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in args.data_dir.glob(\'*.csv\')}\n    save_json(output/\'input_sha256.json\',fingerprints)\n    fold_predictions=[];fold_truth=[];fold_frames=[];metrics=[];audits=[];run_times=[]\n    ensemble_plan=None;development_pred=None\n    for fold,(date,role) in enumerate(zip(cfg[\'fold_origins\'],cfg[\'fold_roles\'])):\n        origin=pd.Timestamp(date);end=origin+pd.Timedelta(days=cfg[\'horizon_days\'])\n        if end>train.date.max():raise ValueError(\'Validation extends beyond observed training\')\n        valid=train.loc[(train.date>origin)&(train.date<=end)].copy().reset_index(drop=True)\n        print(f\'Fold {fold+1} ({role}) origin={date}; validation={len(valid):,}\',flush=True)\n        t=time.perf_counter()\n        x,y,anchor_audit=training_matrix(train,meta,origin,cfg[\'horizon_days\'],cfg[\'origin_stride_days\'],cfg[\'max_training_rows\'],cfg[\'seed\'])\n        xv=make_features(train,valid,meta,origin)\n        assert x.columns.equals(xv.columns)\n        fold_path=output/f\'fold_{fold+1}\';fold_path.mkdir(exist_ok=True)\n        save_json(fold_path/\'training_origins.json\',anchor_audit)\n        audits.append({\'fold\':fold+1,\'role\':role,\'origin\':date,\'validation_start\':str(valid.date.min().date()),\'validation_end\':str(valid.date.max().date()),\'validation_rows\':len(valid),\'training_sample_rows\':len(y),\'feature_count\':x.shape[1],\'training_matrix_mb\':float(x.memory_usage(deep=True).sum()/1e6),\'training_anchor_max\':anchor_audit[-1][\'origin\'],\'training_label_end_max\':max(a[\'label_end\'] for a in anchor_audit),\'full_horizon_embargo\':True})\n        print(f\'  Features={x.shape}, created in {time.perf_counter()-t:.1f}s\',flush=True)\n        preds={}\n        for name in cfg[\'models\']:\n            t=time.perf_counter();model=Model(name,cfg).fit(x,y)\n            preds[name]=model.predict(xv)\n            elapsed=time.perf_counter()-t\n            if not np.isfinite(preds[name]).all():raise ValueError(f\'{name}: nonfinite output\')\n            row={\'fold\':fold+1,\'role\':role,\'model\':name,\'seconds\':elapsed,**regression_metrics(valid.boardings,preds[name])}\n            metrics.append(row);run_times.append({\'phase\':\'cv\',\'fold\':fold+1,\'model\':name,\'seconds\':elapsed})\n            print(f\'  {name:18s} RMSLE={row["rmsle"]:.6f} MAE={row["mae"]:.3f} ({elapsed:.1f}s)\',flush=True)\n            del model\n        frame=valid[[\'date\',\'station_id\',\'hour\',\'boardings\']].copy()\n        frame[\'horizon\']=(valid.date-origin).dt.days\n        for name,pred in preds.items():frame[name]=pred\n        fold_predictions.append(preds);fold_truth.append(valid.boardings.to_numpy());fold_frames.append(frame)\n        if role==\'development\' and cfg[\'fold_roles\'][fold+1]==\'audit\':\n            dev_y=np.concatenate(fold_truth)\n            dev_preds={name:np.concatenate([p[name] for p in fold_predictions]) for name in cfg[\'models\']}\n            ensemble_plan,corr=fit_ensemble(dev_y,dev_preds,cfg[\'selection_metric\'],cfg[\'correlation_cutoff\'],cfg.get(\'ensemble_policy\'))\n            ensemble_plan[\'fit_folds\']=[i+1 for i in range(fold+1)]\n            ensemble_plan[\'audit_fold_used_for_tuning\']=cfg.get(\'evaluation_context\',{}).get(\'policy_selection_has_seen_audit\',False)\n            ensemble_plan[\'audit_fold_used_for_weight_fit\']=False\n            ensemble_plan[\'evaluation_context\']=cfg.get(\'evaluation_context\',{\'status\':\'original_audit\'})\n            save_json(output/\'ensemble_plan.json\',ensemble_plan)\n            corr.to_csv(output/\'development_log_residual_correlations.csv\')\n            development_pred=predict_ensemble(dev_preds,ensemble_plan)\n            print(\'  Development-only ensemble frozen:\',ensemble_plan[\'weights\'],flush=True)\n        if role==\'audit\':\n            if ensemble_plan is None:raise ValueError(\'No development-only ensemble plan\')\n            pred=predict_ensemble(preds,ensemble_plan);frame[\'ensemble\']=pred\n            summary=diagnostics(train,valid,pred,origin,fold_path)\n            metrics.append({\'fold\':fold+1,\'role\':role,\'model\':\'ensemble\',**summary})\n            print(f\'  AUDIT ensemble RMSLE={summary["rmsle"]:.6f}\',flush=True)\n            save_json(output/\'audit_summary.json\',summary)\n        frame.to_csv(fold_path/\'oof.csv.gz\',index=False,compression=\'gzip\')\n        pd.DataFrame(metrics).to_csv(output/\'cv_scores.csv\',index=False)\n        save_json(output/\'cv_folds.json\',audits)\n        del x,y,xv\n    for fold in [0,1]:\n        frame=fold_frames[fold]\n        pred=predict_ensemble(fold_predictions[fold],ensemble_plan)\n        path=output/f\'fold_{fold+1}\'\n        diagnostics(train,frame,pred,pd.Timestamp(cfg[\'fold_origins\'][fold]),path)\n        frame[\'ensemble\']=pred\n        frame.to_csv(path/\'oof.csv.gz\',index=False,compression=\'gzip\')\n    print(\'Fitting selected nonzero-weight models on all observed data...\',flush=True)\n    cutoff=train.date.max()\n    x,y,final_audit=training_matrix(train,meta,cutoff,cfg[\'horizon_days\'],cfg[\'origin_stride_days\'],cfg[\'max_training_rows\'],cfg[\'seed\'])\n    xt=make_features(train,data[\'test\'],meta,cutoff)\n    save_json(output/\'final_training_origins.json\',final_audit)\n    models={};test_preds={}\n    for name,weight in ensemble_plan[\'weights\'].items():\n        if weight<=0:continue\n        t=time.perf_counter();model=Model(name,cfg).fit(x,y)\n        test_preds[name]=model.predict(xt);models[name]=model\n        elapsed=time.perf_counter()-t;run_times.append({\'phase\':\'final_fit_predict\',\'model\':name,\'seconds\':elapsed})\n        print(f\'  Final {name}: weight={weight:.5f}, {elapsed:.1f}s\',flush=True)\n    predictions=predict_ensemble(test_preds,ensemble_plan)\n    bundle={\'config\':cfg,\'ensemble\':ensemble_plan,\'models\':models,\'features\':x.columns.tolist(),\'training_cutoff\':str(cutoff.date()),\'input_sha256\':fingerprints}\n    joblib.dump(bundle,output/\'model_bundle.joblib\',compress=3)\n    export_predictions(data,predictions,output,fold_frames[-1][\'ensemble\'].to_numpy())\n    # Record a strong single-model alternative without using holdout to choose it.\n    best_name=min(ensemble_plan[\'individual_scores\'],key=ensemble_plan[\'individual_scores\'].get)\n    if best_name in test_preds:\n        alternative=data[\'test\'][[\'example_id\']].copy();alternative[\'boardings\']=test_preds[best_name]\n        alternative.to_csv(output/\'submission_best_single.csv\',index=False,float_format=\'%.8f\')\n        validate_submission(output/\'submission_best_single.csv\',data[\'test\'],data[\'sample_submission\'],scope=\'test\')\n    primary_name=cfg.get(\'ensemble_policy\',{}).get(\'primary_model\')\n    if primary_name in test_preds:\n        alternative=data[\'test\'][[\'example_id\']].copy();alternative[\'boardings\']=test_preds[primary_name]\n        alternative.to_csv(output/\'submission_primary.csv\',index=False,float_format=\'%.8f\')\n        validate_submission(output/\'submission_primary.csv\',data[\'test\'],data[\'sample_submission\'],scope=\'test\')\n    peak=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n    peak_mb=peak/(1024*1024) if sys.platform==\'darwin\' else peak/1024\n    save_json(output/\'run_summary.json\',{\'status\':\'completed\',\'official_metric\':\'rmsle\',\'elapsed_seconds\':time.perf_counter()-start,\'peak_rss_mb\':peak_mb,\'feature_count\':x.shape[1],\'models_evaluated\':cfg[\'models\'],\'models_deployed\':list(models),\'timings\':run_times,\'external_data\':[\'preannounced Korean calendar facts\'],\'pretrained_weights\':False,\'submission_rows\':len(predictions),\'submission_source\':\'all test IDs\',\'audit_rmsle\':metrics[-1][\'rmsle\']})\n    print(f\'Completed: {output / "submission.csv"}; {len(predictions):,} rows\',flush=True)\n\ndef infer(args):\n    import joblib\n    data=load_data(args.data_dir)\n    bundle=joblib.load(args.model) # Load only this project\'s trusted locally generated artifact.\n    if str(data[\'train\'].date.max().date())!=bundle[\'training_cutoff\']:raise ValueError(\'Training cutoff mismatch\')\n    for filename,digest in bundle[\'input_sha256\'].items():\n        if hashlib.sha256((args.data_dir/filename).read_bytes()).hexdigest()!=digest:raise ValueError(f\'Input changed: {filename}\')\n    x=make_features(data[\'train\'],data[\'test\'],data[\'metaData\'],bundle[\'training_cutoff\'])\n    if x.columns.tolist()!=bundle[\'features\']:raise ValueError(\'Feature schema mismatch\')\n    preds={name:model.predict(x) for name,model in bundle[\'models\'].items()}\n    prediction=predict_ensemble(preds,bundle[\'ensemble\'])\n    if bundle.get(\'postprocessing\'):\n        from .postprocessing import apply_postprocessing\n        prediction=apply_postprocessing(data[\'train\'],data[\'test\'],bundle[\'training_cutoff\'],prediction,bundle[\'postprocessing\'])\n    report=export_predictions(data,prediction,args.output)\n    print(json.dumps(clean(report),ensure_ascii=False,indent=2))\n\ndef main():\n    if len(sys.argv)>1 and sys.argv[1]==\'v2\':\n        from .pipeline_v2 import main as v2_main\n        return v2_main(sys.argv[2:])\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'command\',choices=[\'train\',\'infer\'])\n    parser.add_argument(\'--data-dir\',type=Path,default=ROOT/\'data\'/\'raw\')\n    parser.add_argument(\'--config\',type=Path,default=ROOT/\'configs\'/\'default.json\')\n    parser.add_argument(\'--output\',type=Path,default=ROOT/\'outputs\'/\'lightgbm_tuned\')\n    parser.add_argument(\'--model\',type=Path,default=ROOT/\'outputs\'/\'lightgbm_tuned\'/\'model_bundle.joblib\')\n    args=parser.parse_args()\n    run(args) if args.command==\'train\' else infer(args)\n\nif __name__==\'__main__\':main()\n', 'src/pipeline_v2.py': '"""Resumable four-fold residual CV, two-seed refit and log-space inference."""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom concurrent.futures import ProcessPoolExecutor,ThreadPoolExecutor,as_completed\nimport multiprocessing\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom .features import load_data\nfrom .features_v2 import training_matrix_v2,make_features_v2\nfrom .models_v2 import ResidualTree,devices\nfrom .ensemble_v2 import evaluate_ensemble\nfrom .pipeline import save_json,environment,diagnostics\nfrom .metrics import regression_metrics\n\nROOT=Path(__file__).resolve().parents[1]\nPROTECTED=[\'submission_lgbm_0.247808.csv\',\'outputs/lightgbm_tuned/model_bundle.joblib\',\'configs/lightgbm_tuned.json\']\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda:f.read(1024*1024),b\'\'):h.update(chunk)\n    return h.hexdigest()\n\ndef preserve(root,output):\n    current={name:sha(root/name) for name in PROTECTED if (root/name).exists()}\n    path=output/\'protected_sha256.json\'\n    if path.exists():\n        expected=json.loads(path.read_text())\n        if current!=expected:raise RuntimeError(\'Protected artifact hash changed; refusing to continue\')\n    else:save_json(path,current)\n    return current\n\ndef contract(args,cfg):\n    args.output.mkdir(parents=True,exist_ok=True)\n    if args.output.resolve()!=(ROOT/\'outputs/ensemble_v2\').resolve():raise ValueError(\'v2 writes only outputs/ensemble_v2\')\n    if args.config.resolve()!=(ROOT/\'configs/ensemble_v2.json\').resolve():raise ValueError(\'v2 requires configs/ensemble_v2.json\')\n    preserve(ROOT,args.output)\n    inputs={p.name:sha(p) for p in sorted(args.data_dir.glob(\'*.csv\'))}\n    source={p.name:sha(p) for p in sorted((ROOT/\'src\').glob(\'*.py\'))}\n    fingerprint={\'inputs\':inputs,\'config_sha256\':sha(args.config),\'source_sha256\':source}\n    path=args.output/\'run_fingerprint.json\'\n    if path.exists() and json.loads(path.read_text())!=fingerprint:raise RuntimeError(\'Input/config/source mismatch. Use a fresh versioned run after review; never silently reuse stale caches.\')\n    save_json(path,fingerprint);save_json(args.output/\'config_resolved.json\',cfg)\n    save_json(args.output/\'environment.json\',{**environment(),\'hardware\':devices()})\n\ndef cache_fold(data,cfg,fold,output):\n    path=output/f\'fold_{fold}\';path.mkdir(parents=True,exist_ok=True);cache=path/\'matrix.joblib\'\n    if cache.exists():return cache\n    origin=pd.Timestamp(cfg[\'fold_origins\'][fold-1]);train=data[\'train\']\n    valid=train.loc[(train.date>origin)&(train.date<=origin+pd.Timedelta(days=123))].reset_index(drop=True)\n    if len(valid)!=157194:raise ValueError(f\'Fold {fold} has {len(valid)} rows, expected 157194\')\n    print(f\'FEATURES fold={fold} origin={origin.date()}\',flush=True)\n    x,y,audit,horizons=training_matrix_v2(train,data[\'metaData\'],origin,cfg[\'max_training_rows\'],cfg[\'seed\'])\n    xv=make_features_v2(train,valid,data[\'metaData\'],origin)\n    yv=np.column_stack([np.log1p(valid.boardings)-xv.anchor_board_log,np.log1p(valid.alightings)-xv.anchor_alight_log]).astype(\'float32\')\n    save_json(path/\'feature_audit.json\',{\'origin\':str(origin.date()),\'rows\':len(x),\'valid_rows\':len(xv),\'features\':list(x),\'origins\':audit,\'horizon_counts\':horizons,\'train_year364_available\':float(x.year364_clean_log.notna().mean()),\'valid_year364_available\':float(xv.year364_clean_log.notna().mean()),\'train_nan_fraction\':x.isna().mean().to_dict(),\'max_training_label\':max(a[\'label_max\'] for a in audit)})\n    joblib.dump({\'x\':x,\'y\':y,\'xvalid\':xv,\'yvalid\':yv,\'valid\':valid,\'origin\':origin},cache,compress=0)\n    return cache\n\ndef worker(args,cfg):\n    start=time.perf_counter();matrix=joblib.load(args.cache,mmap_mode=\'r\')\n    seed=args.seed;name=args.model;dest=args.destination;dest.mkdir(parents=True,exist_ok=True)\n    hardware=devices()\n    if name==\'tabm\':\n        from .tabm_v2 import TabMRegressor\n        model=TabMRegressor(cfg,seed,args.iterations,hardware)\n    else:model=ResidualTree(name,cfg,seed,args.iterations,hardware)\n    model.fit(matrix[\'x\'],matrix[\'y\'],matrix.get(\'xvalid\'),matrix.get(\'yvalid\'))\n    if \'xvalid\' in matrix:prediction=model.predict_log(matrix[\'xvalid\'])\n    else:prediction=model.predict_log(matrix[\'xtest\'])\n    np.save(dest/\'prediction_log.npy\',prediction)\n    metadata={\'model\':name,\'seed\':seed,\'best_iteration\':model.best_iteration,\'requested_iterations\':args.iterations,\'seconds\':time.perf_counter()-start,\'device\':model.device,\'train_rows\':len(matrix[\'x\'])}\n    save_json(dest/\'history.json\',model.history)\n    model.to_cpu();joblib.dump(model,dest/\'model.joblib\',compress=3)\n    save_json(dest/\'result.json\',metadata);print(\'COMPLETED\',metadata,flush=True)\n\ndef run_worker(args,cfg,cache,destination,name,seed,iterations=None):\n    if (destination/\'result.json\').exists() and (destination/\'prediction_log.npy\').exists() and (destination/\'model.joblib\').exists():return\n    destination.mkdir(parents=True,exist_ok=True)\n    command=[sys.executable,\'-m\',\'src.pipeline_v2\',\'worker\',\'--config\',str(args.config),\'--cache\',str(cache),\'--destination\',str(destination),\'--model\',name,\'--seed\',str(seed)]\n    if iterations is not None:command+=[\'--iterations\',str(iterations)]\n    env=os.environ.copy();env.update(PYTHONHASHSEED=str(seed),OMP_NUM_THREADS=str(cfg[\'threads\']),OPENBLAS_NUM_THREADS=str(cfg[\'threads\']),VECLIB_MAXIMUM_THREADS=str(cfg[\'threads\']),CUBLAS_WORKSPACE_CONFIG=\':4096:8\')\n    print(\'TRAIN\',destination,flush=True)\n    with (destination/\'train.log\').open(\'w\') as log:\n        subprocess.run(command,cwd=ROOT,env=env,stdout=log,stderr=subprocess.STDOUT,check=True)\n\ndef train_jobs(args,cfg,jobs):\n    """Schedule separate interpreters without sharing a GPU between training jobs.\n\n    CPU/MPS: at most two CPU jobs plus one MPS job. Dual CUDA: one CPU\n    LightGBM queue, one GPU-0 CatBoost/TabM queue, one GPU-1 XGBoost queue.\n    """\n    hardware=devices();cuda=hardware[\'cuda_count\'] if cfg[\'device\']!=\'cpu\' else 0\n    mps=hardware[\'mps\'] and cfg[\'device\']!=\'cpu\';queues={}\n    def queue(name):\n        if cuda:\n            if name==\'lightgbm\':return \'cpu\'\n            return \'cuda1\' if name==\'xgboost\' and cuda>1 else \'cuda0\'\n        return \'mps\' if name==\'tabm\' and mps else \'cpu\'\n    for job in jobs:queues.setdefault(queue(job[2]),[]).append(job)\n    pools={key:ThreadPoolExecutor(max_workers=cfg.get(\'max_cpu_workers\',2) if key==\'cpu\' and not cuda else 1) for key in queues}\n    futures=[]\n    try:\n        for key,items in queues.items():\n            for cache,destination,name,seed,iterations in items:\n                futures.append(pools[key].submit(run_worker,args,cfg,cache,destination,name,seed,iterations))\n        for future in as_completed(futures):future.result()\n    finally:\n        for pool in pools.values():pool.shutdown(wait=True,cancel_futures=True)\n\ndef collect_cv(data,cfg,output):\n    all_frames=[];summaries=[]\n    for fold in range(1,5):\n        matrix=joblib.load(output/f\'fold_{fold}/matrix.joblib\',mmap_mode=\'r\')\n        valid=matrix[\'valid\'];origin=matrix[\'origin\'];frame=valid[[\'date\',\'station_id\',\'hour\',\'boardings\']].copy()\n        frame[\'fold\']=fold;frame[\'origin\']=str(origin.date());frame[\'horizon\']=(frame.date-origin).dt.days\n        frame[\'true_log\']=np.log1p(valid.boardings)\n        for name in cfg[\'models\']:\n            dest=output/f\'fold_{fold}\'/name;pred=np.load(dest/\'prediction_log.npy\');frame[name]=pred\n            reportpath=dest/\'diagnostics\';reportpath.mkdir(exist_ok=True)\n            metric=diagnostics(data[\'train\'],valid,np.expm1(pred),origin,reportpath)\n            summaries.append({\'fold\':fold,\'model\':name,**metric})\n        all_frames.append(frame)\n    oof=pd.concat(all_frames,ignore_index=True)\n    pred,lofo,report,corr=evaluate_ensemble(oof.true_log.to_numpy(),oof[cfg[\'models\']].to_numpy(),oof.fold.to_numpy(),cfg[\'models\'],cfg[\'fold_weights\'])\n    oof[\'ensemble\']=pred;oof[\'lofo_ensemble\']=lofo\n    oof.to_csv(output/\'oof_predictions.csv.gz\',index=False,compression=\'gzip\')\n    corr.to_csv(output/\'residual_correlation.csv\');save_json(output/\'ensemble_report.json\',report)\n    for fold,frame in oof.groupby(\'fold\'):\n        origin=pd.Timestamp(cfg[\'fold_origins\'][fold-1]);dest=output/f\'fold_{fold}\'/\'ensemble\';dest.mkdir(exist_ok=True)\n        metric=diagnostics(data[\'train\'],frame,np.expm1(frame.ensemble),origin,dest)\n        summaries.append({\'fold\':fold,\'model\':\'ensemble\',**metric})\n    comparison=pd.DataFrame(summaries)\n    comparison[\'old_fold4_rmsle\']=np.where(comparison.fold==4,.247808,np.nan)\n    comparison[\'rmsle_delta_vs_old_fold4\']=comparison.rmsle-comparison.old_fold4_rmsle\n    comparison.to_csv(output/\'cv_metrics.csv\',index=False)\n    print(\'CV REPORT\',json.dumps(report,ensure_ascii=False),flush=True)\n    return report\n\ndef final_cache(data,cfg,output):\n    path=output/\'final\';path.mkdir(exist_ok=True);cache=path/\'matrix.joblib\'\n    if cache.exists():return cache\n    x,y,audit,horizons=training_matrix_v2(data[\'train\'],data[\'metaData\'],cfg[\'final_cutoff\'],cfg[\'final_training_rows\'],cfg[\'seed\'])\n    xtest=make_features_v2(data[\'train\'],data[\'test\'],data[\'metaData\'],cfg[\'final_cutoff\'])\n    save_json(path/\'feature_audit.json\',{\'rows\':len(x),\'origins\':audit,\'horizon_counts\':horizons,\'train_year364_available\':float(x.year364_clean_log.notna().mean()),\'test_year364_available\':float(xtest.year364_clean_log.notna().mean())})\n    joblib.dump({\'x\':x,\'y\':y,\'xtest\':xtest},cache,compress=0);return cache\n\ndef write_submission(data,pred,output):\n    test=data[\'test\'];submission=test[[\'example_id\']].copy();submission[\'boardings\']=np.expm1(pred)\n    if len(submission)!=157194 or not submission.example_id.is_unique or not np.isfinite(submission.boardings).all() or (submission.boardings<0).any():raise ValueError(\'Submission integrity failure\')\n    path=output/\'submission_ensemble_v2.csv\';submission.to_csv(path,index=False,float_format=\'%.8f\')\n    read=pd.read_csv(path);integrity={\'rows\':len(read),\'columns\':list(read),\'ids_in_test_order\':bool(read.example_id.equals(test.example_id)),\'unique_ids\':bool(read.example_id.is_unique),\'finite\':bool(np.isfinite(read.boardings).all()),\'nonnegative\':bool((read.boardings>=0).all()),\'mean\':float(read.boardings.mean()),\'min\':float(read.boardings.min()),\'max\':float(read.boardings.max()),\'mean_in_requested_315_345_range\':bool(315<=read.boardings.mean()<=345),\'minimum_below_3\':bool(read.boardings.min()<3),\'quantiles\':read.boardings.quantile([0,.01,.1,.5,.9,.99,1]).to_dict(),\'sha256\':sha(path),\'note\':\'Mean/minimum thresholds are diagnostics only. No mean forcing, floor forcing, bias calibration or domain zeroing was applied.\'}\n    save_json(output/\'submission_validation.json\',integrity)\n    rich=test.copy();rich[\'boardings\']=read.boardings;rich.to_csv(output/\'test_predictions_with_keys.csv\',index=False)\n    rich.groupby(rich.date.dt.to_period(\'M\')).boardings.agg([\'mean\',\'min\',\'max\',\'sum\']).to_csv(output/\'test_monthly_distribution.csv\')\n    preserve(ROOT,output);print(\'SUBMISSION\',integrity,flush=True)\n\ndef run(args,cfg):\n    start=time.perf_counter();contract(args,cfg);data=load_data(args.data_dir)\n    jobs=[]\n    for fold in range(1,5):\n        cache=cache_fold(data,cfg,fold,args.output)\n        for name in cfg[\'models\']:jobs.append((cache,args.output/f\'fold_{fold}\'/name,name,cfg[\'seed\'],None))\n    train_jobs(args,cfg,jobs)\n    report=collect_cv(data,cfg,args.output)\n    if args.cv_only:return\n    cache=final_cache(data,cfg,args.output);predictions={};iteration_plan={};jobs=[]\n    for name in cfg[\'models\']:\n        best=[json.loads((args.output/f\'fold_{f}\'/name/\'result.json\').read_text())[\'best_iteration\'] for f in [3,4]]\n        iterations=max(1,int(np.rint(np.dot(cfg[\'final_iteration_fold_weights\'],best)*cfg[\'final_iteration_multiplier\'])))\n        iteration_plan[name]={\'fold3\':best[0],\'fold4\':best[1],\'final_iterations\':iterations}\n        for seed in cfg[\'final_seeds\']:\n            dest=args.output/\'final\'/name/str(seed)\n            jobs.append((cache,dest,name,seed,iterations))\n    save_json(args.output/\'final_iteration_plan.json\',iteration_plan)\n    train_jobs(args,cfg,jobs)\n    for name in cfg[\'models\']:\n        predictions[name]=np.mean([np.load(args.output/\'final\'/name/str(seed)/\'prediction_log.npy\') for seed in cfg[\'final_seeds\']],axis=0)\n    pred=sum(report[\'weights\'][name]*value for name,value in predictions.items())\n    write_submission(data,pred,args.output)\n    save_json(args.output/\'bundle_manifest.json\',{\'config\':cfg,\'input_sha256\':json.loads((args.output/\'run_fingerprint.json\').read_text())[\'inputs\'],\'weights\':report[\'weights\'],\'models\':{name:[str((Path(\'final\')/name/str(seed)/\'model.joblib\')) for seed in cfg[\'final_seeds\']] for name in cfg[\'models\']},\'feature_columns\':list(joblib.load(cache,mmap_mode=\'r\')[\'x\']),\'seconds\':time.perf_counter()-start})\n\ndef predict_family(task):\n    name,paths,output,xpath,hardware,cfg=task\n    x=joblib.load(xpath,mmap_mode=\'r\');logs=[]\n    for relative in paths:\n        model=joblib.load(Path(output)/relative)\n        if name==\'tabm\':\n            import torch\n            torch.set_num_threads(cfg[\'threads\'])\n            model.device=model.device_name();model.model.to(model.device)\n        elif name==\'xgboost\' and hardware[\'cuda_count\'] and cfg[\'device\']!=\'cpu\':model.model.set_params(device=f\'cuda:{min(1,hardware["cuda_count"]-1)}\')\n        logs.append(model.predict_log(x))\n    return name,np.mean(logs,axis=0)\n\ndef infer(args,cfg):\n    manifest=json.loads((args.output/\'bundle_manifest.json\').read_text())\n    current={p.name:sha(p) for p in sorted(args.data_dir.glob(\'*.csv\'))}\n    if current!=manifest[\'input_sha256\']:raise ValueError(\'Inference input hash mismatch\')\n    data=load_data(args.data_dir);x=make_features_v2(data[\'train\'],data[\'test\'],data[\'metaData\'],cfg[\'final_cutoff\'])\n    if list(x)!=manifest[\'feature_columns\']:raise ValueError(\'Inference feature schema mismatch\')\n    hardware=devices()\n    xpath=args.output/\'final/inference_features.joblib\';joblib.dump(x,xpath,compress=0)\n    # Independent families predict concurrently; LightGBM uses CPU, XGBoost GPU 1,\n    # TabM GPU 0 when available. CatBoost persisted CPU inference is portable.\n    tasks=[(name,manifest[\'models\'][name],str(args.output),str(xpath),hardware,cfg) for name in cfg[\'models\']]\n    with ProcessPoolExecutor(max_workers=4,mp_context=multiprocessing.get_context(\'spawn\')) as pool:predictions=dict(pool.map(predict_family,tasks))\n    pred=sum(manifest[\'weights\'][name]*value for name,value in predictions.items())\n    write_submission(data,pred,args.output)\n\ndef main(argv=None):\n    parser=argparse.ArgumentParser(description=__doc__);parser.add_argument(\'command\',choices=[\'run\',\'infer\',\'worker\'])\n    parser.add_argument(\'--config\',type=Path,default=ROOT/\'configs/ensemble_v2.json\');parser.add_argument(\'--data-dir\',type=Path,default=ROOT/\'data/raw\');parser.add_argument(\'--output\',type=Path,default=ROOT/\'outputs/ensemble_v2\')\n    parser.add_argument(\'--cv-only\',action=\'store_true\');parser.add_argument(\'--cache\',type=Path);parser.add_argument(\'--destination\',type=Path);parser.add_argument(\'--model\');parser.add_argument(\'--seed\',type=int,default=2026);parser.add_argument(\'--iterations\',type=int)\n    args=parser.parse_args(argv);cfg=json.loads(args.config.read_text())\n    if args.command==\'worker\':worker(args,cfg)\n    elif args.command==\'infer\':infer(args,cfg)\n    else:run(args,cfg)\n\nif __name__==\'__main__\':main()\n', 'src/pipeline_v3.py': '"""Resumable four-fold residual CV, two-seed refit and log-space inference."""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom concurrent.futures import ProcessPoolExecutor,ThreadPoolExecutor,as_completed\nimport multiprocessing\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom .features import load_data\nfrom .features_v3 import training_matrix_v3,make_features_v3\nfrom .models_v3 import ResidualTree,devices\nfrom .ensemble_v3 import evaluate_ensemble\nfrom .pipeline import save_json,environment,diagnostics\nfrom .metrics import regression_metrics\n\nROOT=Path(__file__).resolve().parents[1]\nPROTECTED=[\'submission_lgbm_0.247808.csv\',\'outputs/lightgbm_tuned/model_bundle.joblib\',\'configs/lightgbm_tuned.json\']\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda:f.read(1024*1024),b\'\'):h.update(chunk)\n    return h.hexdigest()\n\ndef preserve(root,output):\n    current={name:sha(root/name) for name in PROTECTED if (root/name).exists()}\n    path=output/\'protected_sha256.json\'\n    if path.exists():\n        expected=json.loads(path.read_text())\n        if current!=expected:raise RuntimeError(\'Protected artifact hash changed; refusing to continue\')\n    else:save_json(path,current)\n    return current\n\ndef contract(args,cfg):\n    args.output.mkdir(parents=True,exist_ok=True)\n    if args.output.resolve()!=(ROOT/\'outputs/ensemble_v3\').resolve():raise ValueError(\'v3 writes only outputs/ensemble_v3\')\n    if args.config.resolve()!=(ROOT/\'configs/ensemble_v3.json\').resolve():raise ValueError(\'v3 requires configs/ensemble_v3.json\')\n    preserve(ROOT,args.output)\n    inputs={p.name:sha(p) for p in sorted(args.data_dir.glob(\'*.csv\'))}\n    source={p.name:sha(p) for p in sorted([*(ROOT/\'src\').glob(\'*.py\'), *(ROOT/\'src\').glob(\'*.json\')])}\n    fingerprint={\'inputs\':inputs,\'config_sha256\':sha(args.config),\'source_sha256\':source}\n    path=args.output/\'run_fingerprint.json\'\n    if path.exists() and json.loads(path.read_text())!=fingerprint:raise RuntimeError(\'Input/config/source mismatch. Use a fresh versioned run after review; never silently reuse stale caches.\')\n    save_json(path,fingerprint);save_json(args.output/\'config_resolved.json\',cfg)\n    save_json(args.output/\'environment.json\',{**environment(),\'hardware\':devices()})\n\ndef cache_fold(data,cfg,fold,output):\n    path=output/f\'fold_{fold}\';path.mkdir(parents=True,exist_ok=True);cache=path/\'matrix.joblib\'\n    if cache.exists():return cache\n    origin=pd.Timestamp(cfg[\'fold_origins\'][fold-1]);train=data[\'train\']\n    valid=train.loc[(train.date>origin)&(train.date<=origin+pd.Timedelta(days=123))].reset_index(drop=True)\n    if len(valid)!=157194:raise ValueError(f\'Fold {fold} has {len(valid)} rows, expected 157194\')\n    print(f\'FEATURES fold={fold} origin={origin.date()}\',flush=True)\n    x,y,audit,horizons=training_matrix_v3(train,data[\'metaData\'],origin,cfg[\'max_training_rows\'],cfg[\'seed\'])\n    xv=make_features_v3(train,valid,data[\'metaData\'],origin)\n    yv=np.column_stack([np.log1p(valid.boardings)-xv.anchor_board_log,np.log1p(valid.alightings)-xv.anchor_alight_log]).astype(\'float32\')\n    save_json(path/\'feature_audit.json\',{\'origin\':str(origin.date()),\'rows\':len(x),\'valid_rows\':len(xv),\'features\':list(x),\'origins\':audit,\'horizon_counts\':horizons,\'train_year364_available\':float(x.year364_clean_log.notna().mean()),\'valid_year364_available\':float(xv.year364_clean_log.notna().mean()),\'train_nan_fraction\':x.isna().mean().to_dict(),\'max_training_label\':max(a[\'label_max\'] for a in audit)})\n    joblib.dump({\'x\':x,\'y\':y,\'xvalid\':xv,\'yvalid\':yv,\'valid\':valid,\'origin\':origin},cache,compress=0)\n    return cache\n\ndef worker(args,cfg):\n    start=time.perf_counter();matrix=joblib.load(args.cache,mmap_mode=\'r\')\n    seed=args.seed;name=args.model;dest=args.destination;dest.mkdir(parents=True,exist_ok=True)\n    hardware=devices()\n    if name==\'tabm\':\n        from .tabm_v3 import TabMRegressor\n        model=TabMRegressor(cfg,seed,args.iterations,hardware)\n    else:model=ResidualTree(name,cfg,seed,args.iterations,hardware)\n    model.fit(matrix[\'x\'],matrix[\'y\'],matrix.get(\'xvalid\'),matrix.get(\'yvalid\'))\n    if \'xvalid\' in matrix:prediction=model.predict_log(matrix[\'xvalid\'])\n    else:prediction=model.predict_log(matrix[\'xtest\'])\n    np.save(dest/\'prediction_log.npy\',prediction)\n    metadata={\'model\':name,\'seed\':seed,\'best_iteration\':model.best_iteration,\'requested_iterations\':args.iterations,\'seconds\':time.perf_counter()-start,\'device\':model.device,\'train_rows\':len(matrix[\'x\'])}\n    save_json(dest/\'history.json\',model.history)\n    model.to_cpu();joblib.dump(model,dest/\'model.joblib\',compress=3)\n    save_json(dest/\'result.json\',metadata);print(\'COMPLETED\',metadata,flush=True)\n\ndef run_worker(args,cfg,cache,destination,name,seed,iterations=None):\n    if (destination/\'result.json\').exists() and (destination/\'prediction_log.npy\').exists() and (destination/\'model.joblib\').exists():return\n    destination.mkdir(parents=True,exist_ok=True)\n    command=[sys.executable,\'-m\',\'src.pipeline_v3\',\'worker\',\'--config\',str(args.config),\'--cache\',str(cache),\'--destination\',str(destination),\'--model\',name,\'--seed\',str(seed)]\n    if iterations is not None:command+=[\'--iterations\',str(iterations)]\n    env=os.environ.copy();env.update(PYTHONHASHSEED=str(seed),OMP_NUM_THREADS=str(cfg[\'threads\']),OPENBLAS_NUM_THREADS=str(cfg[\'threads\']),VECLIB_MAXIMUM_THREADS=str(cfg[\'threads\']),CUBLAS_WORKSPACE_CONFIG=\':4096:8\')\n    print(\'TRAIN\',destination,flush=True)\n    with (destination/\'train.log\').open(\'w\') as log:\n        subprocess.run(command,cwd=ROOT,env=env,stdout=log,stderr=subprocess.STDOUT,check=True)\n\ndef train_jobs(args,cfg,jobs):\n    """Schedule separate interpreters without sharing a GPU between training jobs.\n\n    CPU/MPS: at most two CPU jobs plus one MPS job. Dual CUDA: one CPU\n    LightGBM queue, one GPU-0 CatBoost/TabM queue, one GPU-1 XGBoost queue.\n    """\n    hardware=devices();cuda=hardware[\'cuda_count\'] if cfg[\'device\']!=\'cpu\' else 0\n    mps=hardware[\'mps\'] and cfg[\'device\']!=\'cpu\';queues={}\n    def queue(name):\n        if cuda:\n            if name==\'lightgbm\':return \'cpu\'\n            return \'cuda1\' if name==\'xgboost\' and cuda>1 else \'cuda0\'\n        return \'mps\' if name==\'tabm\' and mps else \'cpu\'\n    for job in jobs:queues.setdefault(queue(job[2]),[]).append(job)\n    pools={key:ThreadPoolExecutor(max_workers=cfg.get(\'max_cpu_workers\',2) if key==\'cpu\' and not cuda else 1) for key in queues}\n    futures=[]\n    try:\n        for key,items in queues.items():\n            for cache,destination,name,seed,iterations in items:\n                futures.append(pools[key].submit(run_worker,args,cfg,cache,destination,name,seed,iterations))\n        for future in as_completed(futures):future.result()\n    finally:\n        for pool in pools.values():pool.shutdown(wait=True,cancel_futures=True)\n\ndef collect_cv(data,cfg,output):\n    all_frames=[];summaries=[]\n    for fold in range(1,5):\n        matrix=joblib.load(output/f\'fold_{fold}/matrix.joblib\',mmap_mode=\'r\')\n        valid=matrix[\'valid\'];origin=matrix[\'origin\'];frame=valid[[\'date\',\'station_id\',\'hour\',\'boardings\']].copy()\n        frame[\'fold\']=fold;frame[\'origin\']=str(origin.date());frame[\'horizon\']=(frame.date-origin).dt.days\n        frame[\'true_log\']=np.log1p(valid.boardings)\n        for name in cfg[\'models\']:\n            dest=output/f\'fold_{fold}\'/name;pred=np.load(dest/\'prediction_log.npy\');frame[name]=pred\n            reportpath=dest/\'diagnostics\';reportpath.mkdir(exist_ok=True)\n            metric=diagnostics(data[\'train\'],valid,np.expm1(pred),origin,reportpath)\n            summaries.append({\'fold\':fold,\'model\':name,**metric})\n        all_frames.append(frame)\n    oof=pd.concat(all_frames,ignore_index=True)\n    pred,lofo,report,corr=evaluate_ensemble(oof.true_log.to_numpy(),oof[cfg[\'models\']].to_numpy(),oof.fold.to_numpy(),cfg[\'models\'],cfg[\'fold_weights\'],oof.date.to_numpy())\n    oof[\'ensemble\']=pred;oof[\'lofo_ensemble\']=lofo\n    oof.to_csv(output/\'oof_predictions.csv.gz\',index=False,compression=\'gzip\')\n    corr.to_csv(output/\'residual_correlation.csv\');save_json(output/\'ensemble_report.json\',report)\n    for fold,frame in oof.groupby(\'fold\'):\n        origin=pd.Timestamp(cfg[\'fold_origins\'][fold-1]);dest=output/f\'fold_{fold}\'/\'ensemble\';dest.mkdir(exist_ok=True)\n        metric=diagnostics(data[\'train\'],frame,np.expm1(frame.ensemble),origin,dest)\n        summaries.append({\'fold\':fold,\'model\':\'ensemble\',**metric})\n    comparison=pd.DataFrame(summaries)\n    comparison[\'old_fold4_rmsle\']=np.where(comparison.fold==4,.247808,np.nan)\n    comparison[\'rmsle_delta_vs_old_fold4\']=comparison.rmsle-comparison.old_fold4_rmsle\n    comparison.to_csv(output/\'cv_metrics.csv\',index=False)\n    print(\'CV REPORT\',json.dumps(report,ensure_ascii=False),flush=True)\n    return report\n\ndef final_cache(data,cfg,output):\n    path=output/\'final\';path.mkdir(exist_ok=True);cache=path/\'matrix.joblib\'\n    if cache.exists():return cache\n    x,y,audit,horizons=training_matrix_v3(data[\'train\'],data[\'metaData\'],cfg[\'final_cutoff\'],cfg[\'final_training_rows\'],cfg[\'seed\'])\n    xtest=make_features_v3(data[\'train\'],data[\'test\'],data[\'metaData\'],cfg[\'final_cutoff\'])\n    save_json(path/\'feature_audit.json\',{\'rows\':len(x),\'origins\':audit,\'horizon_counts\':horizons,\'train_year364_available\':float(x.year364_clean_log.notna().mean()),\'test_year364_available\':float(xtest.year364_clean_log.notna().mean())})\n    joblib.dump({\'x\':x,\'y\':y,\'xtest\':xtest},cache,compress=0);return cache\n\ndef write_submission(data,pred,output):\n    test=data[\'test\'];submission=test[[\'example_id\']].copy();submission[\'boardings\']=np.expm1(pred)\n    if len(submission)!=157194 or not submission.example_id.is_unique or not np.isfinite(submission.boardings).all() or (submission.boardings<0).any():raise ValueError(\'Submission integrity failure\')\n    path=output/\'submission_ensemble_v3.csv\';submission.to_csv(path,index=False,float_format=\'%.8f\')\n    read=pd.read_csv(path);integrity={\'rows\':len(read),\'columns\':list(read),\'ids_in_test_order\':bool(read.example_id.equals(test.example_id)),\'unique_ids\':bool(read.example_id.is_unique),\'finite\':bool(np.isfinite(read.boardings).all()),\'nonnegative\':bool((read.boardings>=0).all()),\'mean\':float(read.boardings.mean()),\'min\':float(read.boardings.min()),\'max\':float(read.boardings.max()),\'mean_in_requested_315_345_range\':bool(315<=read.boardings.mean()<=345),\'minimum_below_3\':bool(read.boardings.min()<3),\'quantiles\':read.boardings.quantile([0,.01,.1,.5,.9,.99,1]).to_dict(),\'sha256\':sha(path),\'note\':\'Mean/minimum thresholds are diagnostics only. No mean forcing, floor forcing, bias calibration or domain zeroing was applied.\'}\n    save_json(output/\'submission_validation.json\',integrity)\n    rich=test.copy();rich[\'boardings\']=read.boardings;rich.to_csv(output/\'test_predictions_with_keys.csv\',index=False)\n    rich.groupby(rich.date.dt.to_period(\'M\')).boardings.agg([\'mean\',\'min\',\'max\',\'sum\']).to_csv(output/\'test_monthly_distribution.csv\')\n    preserve(ROOT,output);print(\'SUBMISSION\',integrity,flush=True)\n\ndef write_candidates(data, predictions, report, output):\n    """Two-seed family submissions and all-fold blend for controlled comparison."""\n    dest=output/\'candidates\';dest.mkdir(exist_ok=True)\n    values=dict(predictions)\n    values[\'all_fold_blend\']=sum(report[\'all_fold_comparison\'][\'weights\'][name]*p for name,p in predictions.items())\n    audits={}\n    for name,p in values.items():\n        frame=data[\'test\'][[\'example_id\']].copy();frame[\'boardings\']=np.expm1(p)\n        if not np.isfinite(frame.boardings).all() or (frame.boardings<0).any():raise ValueError(name)\n        path=dest/f\'submission_{name}.csv\';frame.to_csv(path,index=False,float_format=\'%.8f\')\n        audits[name]={\'rows\':len(frame),\'mean\':float(frame.boardings.mean()),\'min\':float(frame.boardings.min()),\'sha256\':sha(path)}\n    save_json(dest/\'validation.json\',audits)\n\ndef _cache_task(task):\n    data_dir,cfg,fold,output=task\n    return cache_fold(load_data(data_dir),cfg,fold,output)\n\ndef run(args,cfg):\n    start=time.perf_counter();contract(args,cfg);data=load_data(args.data_dir)\n    jobs=[]\n    with ProcessPoolExecutor(max_workers=cfg.get(\'feature_workers\', 1),mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        list(pool.map(_cache_task,[(args.data_dir,cfg,fold,args.output) for fold in range(1,5)]))\n    for fold in range(1,5):\n        cache=args.output/f\'fold_{fold}/matrix.joblib\'\n        for name in cfg[\'models\']:jobs.append((cache,args.output/f\'fold_{fold}\'/name,name,cfg[\'seed\'],None))\n    train_jobs(args,cfg,jobs)\n    report=collect_cv(data,cfg,args.output)\n    if args.cv_only:return\n    cache=final_cache(data,cfg,args.output);predictions={};iteration_plan={};jobs=[]\n    for name in cfg[\'models\']:\n        best=[json.loads((args.output/f\'fold_{f}\'/name/\'result.json\').read_text())[\'best_iteration\'] for f in [3,4]]\n        iterations=max(1,int(np.rint(np.dot(cfg[\'final_iteration_fold_weights\'],best)*cfg[\'final_iteration_multiplier\'])))\n        iteration_plan[name]={\'fold3\':best[0],\'fold4\':best[1],\'final_iterations\':iterations}\n        for seed in cfg[\'final_seeds\']:\n            dest=args.output/\'final\'/name/str(seed)\n            jobs.append((cache,dest,name,seed,iterations))\n    save_json(args.output/\'final_iteration_plan.json\',iteration_plan)\n    train_jobs(args,cfg,jobs)\n    for name in cfg[\'models\']:\n        predictions[name]=np.mean([np.load(args.output/\'final\'/name/str(seed)/\'prediction_log.npy\') for seed in cfg[\'final_seeds\']],axis=0)\n    write_candidates(data,predictions,report,args.output)\n    pred=sum(report[\'weights\'][name]*value for name,value in predictions.items())\n    write_submission(data,pred,args.output)\n    save_json(args.output/\'bundle_manifest.json\',{\'config\':cfg,\'input_sha256\':json.loads((args.output/\'run_fingerprint.json\').read_text())[\'inputs\'],\'weights\':report[\'weights\'],\'models\':{name:[str((Path(\'final\')/name/str(seed)/\'model.joblib\')) for seed in cfg[\'final_seeds\']] for name in cfg[\'models\']},\'feature_columns\':list(joblib.load(cache,mmap_mode=\'r\')[\'x\']),\'seconds\':time.perf_counter()-start})\n\ndef predict_family(task):\n    name,paths,output,xpath,hardware,cfg=task\n    x=joblib.load(xpath,mmap_mode=\'r\');logs=[]\n    for relative in paths:\n        model=joblib.load(Path(output)/relative)\n        if name==\'tabm\':\n            import torch\n            torch.set_num_threads(cfg[\'threads\'])\n            model.device=model.device_name();model.model.to(model.device)\n        elif name==\'xgboost\' and hardware[\'cuda_count\'] and cfg[\'device\']!=\'cpu\':model.model.set_params(device=f\'cuda:{min(1,hardware["cuda_count"]-1)}\')\n        logs.append(model.predict_log(x))\n    return name,np.mean(logs,axis=0)\n\ndef infer(args,cfg):\n    manifest=json.loads((args.output/\'bundle_manifest.json\').read_text())\n    current={p.name:sha(p) for p in sorted(args.data_dir.glob(\'*.csv\'))}\n    if current!=manifest[\'input_sha256\']:raise ValueError(\'Inference input hash mismatch\')\n    data=load_data(args.data_dir);x=make_features_v3(data[\'train\'],data[\'test\'],data[\'metaData\'],cfg[\'final_cutoff\'])\n    if list(x)!=manifest[\'feature_columns\']:raise ValueError(\'Inference feature schema mismatch\')\n    hardware=devices()\n    xpath=args.output/\'final/inference_features.joblib\';joblib.dump(x,xpath,compress=0)\n    # Independent families predict concurrently; LightGBM uses CPU, XGBoost GPU 1,\n    # TabM GPU 0 when available. CatBoost persisted CPU inference is portable.\n    tasks=[(name,manifest[\'models\'][name],str(args.output),str(xpath),hardware,cfg) for name in cfg[\'models\']]\n    with ProcessPoolExecutor(max_workers=4,mp_context=multiprocessing.get_context(\'spawn\')) as pool:predictions=dict(pool.map(predict_family,tasks))\n    pred=sum(manifest[\'weights\'][name]*value for name,value in predictions.items())\n    write_submission(data,pred,args.output)\n\ndef main(argv=None):\n    parser=argparse.ArgumentParser(description=__doc__);parser.add_argument(\'command\',choices=[\'run\',\'infer\',\'worker\'])\n    parser.add_argument(\'--config\',type=Path,default=ROOT/\'configs/ensemble_v3.json\');parser.add_argument(\'--data-dir\',type=Path,default=ROOT/\'data/raw\');parser.add_argument(\'--output\',type=Path,default=ROOT/\'outputs/ensemble_v3\')\n    parser.add_argument(\'--cv-only\',action=\'store_true\');parser.add_argument(\'--cache\',type=Path);parser.add_argument(\'--destination\',type=Path);parser.add_argument(\'--model\');parser.add_argument(\'--seed\',type=int,default=2026);parser.add_argument(\'--iterations\',type=int)\n    args=parser.parse_args(argv);cfg=json.loads(args.config.read_text())\n    if args.command==\'worker\':worker(args,cfg)\n    elif args.command==\'infer\':infer(args,cfg)\n    else:run(args,cfg)\n\nif __name__==\'__main__\':main()\n', 'src/postprocessing.py': '"""Cutoff-safe, holiday-preserving corrections for nonnegative demand forecasts."""\nfrom __future__ import annotations\nimport numpy as np\nimport pandas as pd\nfrom .holiday_calendar import calendar_features\n\nANCHOR_WINDOWS=(28,56,112)\n\ndef build_context(history,rows,origin):\n    origin=pd.Timestamp(origin)\n    query=rows[[\'date\',\'station_id\',\'hour\']].copy().reset_index(drop=True)\n    query[\'date\']=pd.to_datetime(query.date)\n    if (query.date<=origin).any():raise ValueError(\'Postprocessing queries must follow the origin\')\n    past=history.loc[pd.to_datetime(history.date)<=origin,[\'date\',\'station_id\',\'hour\',\'boardings\']].copy()\n    past[\'date\']=pd.to_datetime(past.date)\n    if past.empty:raise ValueError(\'No observations before origin\')\n    query[\'dow\']=query.date.dt.dayofweek\n    query[\'is_holiday\']=calendar_features(query.date,as_of=origin).is_holiday.to_numpy(dtype=bool)\n    past[\'dow\']=past.date.dt.dayofweek\n    past[\'is_holiday\']=calendar_features(past.date,as_of=origin).is_holiday.to_numpy(dtype=bool)\n    past[\'log_boardings\']=np.log1p(past.boardings)\n    keys=[\'station_id\',\'hour\',\'dow\']\n    index=pd.MultiIndex.from_frame(query[keys])\n    for days in ANCHOR_WINDOWS:\n        available=past.loc[(past.date>origin-pd.Timedelta(days=days))&~past.is_holiday]\n        profile=available.groupby(keys).log_boardings.mean()\n        query[f\'anchor_{days}\']=profile.reindex(index).to_numpy()\n    for days in [56,112]:\n        available=past.loc[past.date>origin-pd.Timedelta(days=days)]\n        cap=available.groupby([\'station_id\',\'hour\']).log_boardings.quantile(.95)\n        query[f\'cap95_{days}\']=cap.reindex(pd.MultiIndex.from_frame(query[[\'station_id\',\'hour\']])).to_numpy()\n    return query\n\ndef candidate_specs():\n    calibrations=[{\'kind\':\'none\',\'scale\':0.}]\n    for group in [\'hour\',\'hour_dow\']:\n        for scale in [.25,.5]:calibrations.append({\'kind\':group,\'scale\':scale})\n    anchors=[{\'days\':0,\'weight\':0.}]+[{\'days\':d,\'weight\':w} for d in ANCHOR_WINDOWS for w in [.05,.1,.2]]\n    candidates=[]\n    for cal in calibrations:\n        for anchor in anchors:\n            name=f\'{cal["kind"]}_{cal["scale"]:g}_anchor{anchor["days"]}_{anchor["weight"]:g}\'\n            candidates.append({\'name\':name,\'calibration\':cal,\'anchor\':anchor,\'cap_days\':0,\'eligible\':True})\n    for scale in [.5,1.]:\n        candidates.append({\'name\':f\'global_bias_{scale:g}\',\'calibration\':{\'kind\':\'global\',\'scale\':scale},\'anchor\':{\'days\':0,\'weight\':0.},\'cap_days\':0,\'eligible\':True})\n    for days in [56,112]:\n        candidates.append({\'name\':f\'diagnostic_cap95_{days}\',\'calibration\':{\'kind\':\'none\',\'scale\':0.},\'anchor\':{\'days\':0,\'weight\':0.},\'cap_days\':days,\'eligible\':False})\n    return candidates\n\ndef group_keys(context,kind):\n    if kind==\'hour\':return context.hour.astype(str).to_numpy()\n    if kind==\'hour_dow\':return (context.hour.astype(str)+\'|\'+context.dow.astype(str)).to_numpy()\n    raise ValueError(f\'Unknown calibration group: {kind}\')\n\ndef fit_plan(context,predictions,actual,spec,fold_ids=None):\n    pred=np.asarray(predictions,dtype=np.float64);actual=np.asarray(actual,dtype=np.float64)\n    if len(context)!=len(pred) or pred.shape!=actual.shape:raise ValueError(\'Postprocessing fit shape mismatch\')\n    if not np.isfinite(pred).all() or np.any(pred<0) or not np.isfinite(actual).all() or np.any(actual<0):\n        raise ValueError(\'Finite nonnegative fit inputs required\')\n    kind=spec[\'calibration\'][\'kind\']\n    result={\'spec\':spec,\'offsets\':{},\'global_offset\':0.,\'shrinkage_pseudo_rows\':1000,\'offset_clip\':.1,\n            \'holiday_policy\':\'retain_exact_model_prediction\',\'missing_anchor_policy\':\'retain_calibrated_model_prediction\'}\n    if kind==\'none\':return result\n    residual=np.log1p(actual)-np.log1p(pred)\n    eligible=~context.is_holiday.to_numpy(dtype=bool)\n    if not eligible.any():return result\n    if kind==\'global\':\n        result[\'global_offset\']=float(np.clip(residual[eligible].mean(),-.1,.1)*spec[\'calibration\'][\'scale\'])\n        return result\n    folds=np.zeros(len(context),dtype=int) if fold_ids is None else np.asarray(fold_ids)\n    data=pd.DataFrame({\'key\':group_keys(context,kind),\'fold\':folds,\'residual\':residual}).loc[eligible].copy()\n    # Remove the seasonal/global bias separately per calibration fold so a sign\n    # reversal between development periods is not transported into test forecasts.\n    data[\'centered\']=data.residual-data.groupby(\'fold\').residual.transform(\'mean\')\n    grouped=data.groupby(\'key\').centered.agg([\'sum\',\'size\'])\n    shifts=(grouped[\'sum\']/(grouped[\'size\']+1000)).clip(-.1,.1)*spec[\'calibration\'][\'scale\']\n    result[\'offsets\']={str(k):float(v) for k,v in shifts.items()}\n    return result\n\ndef apply_plan(context,predictions,plan):\n    pred=np.asarray(predictions,dtype=np.float64)\n    if pred.ndim!=1 or len(context)!=len(pred) or not np.isfinite(pred).all() or np.any(pred<0):\n        raise ValueError(\'Finite nonnegative predictions matching query rows required\')\n    spec=plan[\'spec\'];kind=spec[\'calibration\'][\'kind\']\n    if kind==\'none\' and spec[\'anchor\'][\'weight\']==0 and spec[\'cap_days\']==0:return pred.copy()\n    z=np.log1p(pred)\n    if kind==\'global\':z=z+plan[\'global_offset\']\n    elif kind!=\'none\':z=z+pd.Series(group_keys(context,kind)).map(plan[\'offsets\']).fillna(0).to_numpy()\n    anchor=spec[\'anchor\']\n    if anchor[\'weight\']:\n        if not 0<=anchor[\'weight\']<=.2:raise ValueError(\'Historical anchor share must not exceed 20%\')\n        historical=context[f\'anchor_{anchor["days"]}\'].to_numpy()\n        usable=np.isfinite(historical)\n        z[usable]=(1-anchor[\'weight\'])*z[usable]+anchor[\'weight\']*historical[usable]\n    if spec[\'cap_days\']:\n        cap=context[f\'cap95_{spec["cap_days"]}\'].to_numpy()\n        usable=np.isfinite(cap);z[usable]=np.minimum(z[usable],cap[usable])\n    out=np.expm1(np.maximum(z,0))\n    # No closure or zero-demand rule exists for any of the target hours/holidays.\n    holiday=context.is_holiday.to_numpy(dtype=bool);out[holiday]=pred[holiday]\n    if not np.isfinite(out).all():raise FloatingPointError(\'Nonfinite postprocessed output\')\n    return out\n\ndef apply_postprocessing(history,rows,origin,predictions,plan):\n    return apply_plan(build_context(history,rows,origin),predictions,plan)\n', 'src/tabm_v2.py': '"""Pure PyTorch multitask BatchEnsemble with periodic numerical embeddings.\n\nThis is the requested TabM-style architecture, not the upstream TabM package.\n"""\nimport copy\nimport time\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom .nn_preprocessor import NNPreprocessor\nfrom .models_v2 import seed_everything\n\nclass PeriodicEmbedding(nn.Module):\n    def __init__(self,n,frequencies):\n        super().__init__();self.weight=nn.Parameter(torch.randn(n,frequencies)*.1)\n    def forward(self,x):\n        phase=2*torch.pi*x[:,:,None]*self.weight[None,:,:]\n        return torch.cat([phase.sin(),phase.cos()],dim=-1).flatten(1)\n\nclass BatchEnsembleBlock(nn.Module):\n    def __init__(self,inputs,hidden,k,dropout):\n        super().__init__();self.linear=nn.Linear(inputs,hidden,bias=False)\n        self.r=nn.Parameter(torch.empty(k,inputs).bernoulli_(.5).mul_(2).sub_(1))\n        self.s=nn.Parameter(torch.empty(k,hidden).bernoulli_(.5).mul_(2).sub_(1))\n        self.bias=nn.Parameter(torch.zeros(k,hidden));self.norm=nn.LayerNorm(hidden)\n        self.activation=nn.SiLU();self.dropout=nn.Dropout(dropout)\n    def forward(self,x):\n        return self.dropout(self.activation(self.norm(self.linear(x*self.r)*self.s+self.bias)))\n\nclass MultiTaskTabM(nn.Module):\n    def __init__(self,n_cont,cardinalities,k=8,hidden=256,blocks=3,frequencies=4,dropout=.15):\n        super().__init__();self.k=k\n        self.raw_cont=n_cont//2\n        self.numeric=PeriodicEmbedding(self.raw_cont,frequencies)\n        self.embeddings=nn.ModuleList([nn.Embedding(c,min(16,max(2,int(np.ceil(np.sqrt(c))))),padding_idx=0) for c in cardinalities])\n        width=self.raw_cont*frequencies*2+self.raw_cont+sum(e.embedding_dim for e in self.embeddings)\n        self.blocks=nn.ModuleList([BatchEnsembleBlock(width if i==0 else hidden,hidden,k,dropout) for i in range(blocks)])\n        self.head=nn.Linear(hidden,2)\n    def forward(self,num,cat):\n        x=torch.cat([self.numeric(num[:,:self.raw_cont]),num[:,self.raw_cont:]]+[e(cat[:,i]) for i,e in enumerate(self.embeddings)],dim=1)\n        x=x[:,None,:].expand(-1,self.k,-1)\n        for block in self.blocks:x=block(x)\n        return self.head(x)\n\ndef multitask_loss(pred,target,auxiliary_weight=.2):\n    error=(pred-target[:,None,:]).square()\n    # Compute each head\'s loss before averaging. Opposing heads cannot cancel.\n    per_head=error[:,:,0].mean(dim=0)+auxiliary_weight*error[:,:,1].mean(dim=0)\n    return per_head.mean()\n\nclass TabMRegressor:\n    def __init__(self,cfg,seed=2026,iterations=None,hardware=None):\n        self.cfg=cfg;self.seed=seed;self.iterations=iterations;self.hardware=hardware or {}\n    def device_name(self):\n        if self.cfg[\'device\']==\'cpu\':return \'cpu\'\n        if torch.cuda.is_available():return \'cuda:0\'\n        if torch.backends.mps.is_available():return \'mps\'\n        return \'cpu\'\n    def fit(self,x,y,xvalid=None,yvalid=None):\n        seed_everything(self.seed);torch.manual_seed(self.seed);torch.set_num_threads(self.cfg[\'threads\'])\n        if torch.cuda.is_available():torch.cuda.manual_seed_all(self.seed)\n        torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\n        self.device=self.device_name();p=self.cfg[\'tabm\'];amp=self.device.startswith(\'cuda\')\n        self.preprocessor=NNPreprocessor().fit(x)\n        num,cat=self.preprocessor.transform(x);target=torch.from_numpy(y)\n        self.arch={key:p[key] for key in [\'k\',\'hidden\',\'blocks\',\'frequencies\',\'dropout\']}\n        self.arch.update(n_cont=num.shape[1],cardinalities=self.preprocessor.cardinalities)\n        self.model=MultiTaskTabM(**self.arch).to(self.device)\n        num=torch.from_numpy(num);cat=torch.from_numpy(cat)\n        optimizer=torch.optim.AdamW(self.model.parameters(),lr=p[\'learning_rate\'],weight_decay=p[\'weight_decay\'])\n        epochs=self.iterations or p[\'epochs\'];scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)\n        scaler=torch.amp.GradScaler(\'cuda\',enabled=amp)\n        self.history=[];best=float(\'inf\');bad=0;best_state=None\n        valid_prepared=None if xvalid is None else self.preprocessor.transform(xvalid)\n        valid_true=None if yvalid is None else yvalid[:,0]+xvalid.anchor_board_log.to_numpy()\n        batch=p[\'batch_size\'];rng=np.random.default_rng(self.seed)\n        for epoch in range(1,epochs+1):\n            start=time.perf_counter();self.model.train();total=0;order=rng.permutation(len(x))\n            for offset in range(0,len(x),batch):\n                ix=torch.from_numpy(order[offset:offset+batch]);a=num[ix].to(self.device);b=cat[ix].to(self.device);t=target[ix].to(self.device)\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=\'cuda\' if amp else \'cpu\',dtype=torch.float16,enabled=amp):\n                    loss=multitask_loss(self.model(a,b),t,p[\'auxiliary_weight\'])\n                if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite TabM loss\')\n                scaler.scale(loss).backward();scaler.unscale_(optimizer)\n                nn.utils.clip_grad_norm_(self.model.parameters(),5.0)\n                scaler.step(optimizer);scaler.update();total+=loss.item()*len(ix)\n            scheduler.step();val=None\n            if xvalid is not None:\n                pred=self._predict_prepared(valid_prepared,xvalid.anchor_board_log.to_numpy())\n                val=float(np.sqrt(np.mean((pred-valid_true)**2)))\n                if val<best-1e-7:\n                    best=val;bad=0;self.best_iteration=epoch\n                    best_state={k:v.detach().cpu().clone() for k,v in self.model.state_dict().items()}\n                else:bad+=1\n            else:self.best_iteration=epoch\n            row={\'epoch\':epoch,\'train_loss\':total/len(x),\'valid_rmsle\':val,\'seconds\':time.perf_counter()-start,\'learning_rate\':optimizer.param_groups[0][\'lr\']}\n            self.history.append(row);print(row,flush=True)\n            if xvalid is not None and bad>=p[\'patience\']:break\n        if best_state is not None:self.model.load_state_dict(best_state)\n        return self\n    def _predict_prepared(self,prepared,anchor):\n        self.model.eval();num,cat=prepared;out=[];batch=self.cfg[\'tabm\'][\'batch_size\']\n        with torch.no_grad():\n            for start in range(0,len(num),batch):\n                a=torch.from_numpy(num[start:start+batch]).to(self.device);b=torch.from_numpy(cat[start:start+batch]).to(self.device)\n                residual=self.model(a,b)[:,:,0].float().cpu().numpy()\n                # Clip each member in log space BEFORE the K-member average.\n                out.append(np.maximum(0,anchor[start:start+batch,None]+residual).mean(axis=1))\n        return np.concatenate(out).astype(\'float64\')\n    def predict_log(self,x):return self._predict_prepared(self.preprocessor.transform(x),x.anchor_board_log.to_numpy())\n    def to_cpu(self):self.model.to(\'cpu\');self.device=\'cpu\';return self\n', 'src/tabm_v3.py': '"""Pure PyTorch multitask BatchEnsemble with periodic numerical embeddings.\n\nThis is the requested TabM-style architecture, not the upstream TabM package.\n"""\nimport copy\nimport time\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom .nn_preprocessor_v3 import NNPreprocessorV3 as NNPreprocessor\nfrom .models_v2 import seed_everything\n\nclass PeriodicEmbedding(nn.Module):\n    def __init__(self,n,frequencies):\n        super().__init__();self.weight=nn.Parameter(torch.randn(n,frequencies)*.1)\n    def forward(self,x):\n        phase=2*torch.pi*x[:,:,None]*self.weight[None,:,:]\n        return torch.cat([phase.sin(),phase.cos()],dim=-1).flatten(1)\n\nclass BatchEnsembleBlock(nn.Module):\n    def __init__(self,inputs,hidden,k,dropout):\n        super().__init__();self.linear=nn.Linear(inputs,hidden,bias=False)\n        self.r=nn.Parameter(torch.empty(k,inputs).bernoulli_(.5).mul_(2).sub_(1))\n        self.s=nn.Parameter(torch.empty(k,hidden).bernoulli_(.5).mul_(2).sub_(1))\n        self.bias=nn.Parameter(torch.zeros(k,hidden));self.norm=nn.LayerNorm(hidden)\n        self.activation=nn.SiLU();self.dropout=nn.Dropout(dropout)\n    def forward(self,x):\n        return self.dropout(self.activation(self.norm(self.linear(x*self.r)*self.s+self.bias)))\n\nclass MultiTaskTabM(nn.Module):\n    def __init__(self,n_cont,cardinalities,k=8,hidden=256,blocks=3,frequencies=4,dropout=.15):\n        super().__init__();self.k=k\n        self.raw_cont=n_cont//2\n        self.numeric=PeriodicEmbedding(self.raw_cont,frequencies)\n        self.embeddings=nn.ModuleList([nn.Embedding(c,min(16,max(2,int(np.ceil(np.sqrt(c))))),padding_idx=0) for c in cardinalities])\n        width=self.raw_cont*frequencies*2+self.raw_cont+sum(e.embedding_dim for e in self.embeddings)\n        self.blocks=nn.ModuleList([BatchEnsembleBlock(width if i==0 else hidden,hidden,k,dropout) for i in range(blocks)])\n        self.head=nn.Linear(hidden,2)\n    def forward(self,num,cat):\n        x=torch.cat([self.numeric(num[:,:self.raw_cont]),num[:,self.raw_cont:]]+[e(cat[:,i]) for i,e in enumerate(self.embeddings)],dim=1)\n        x=x[:,None,:].expand(-1,self.k,-1)\n        for block in self.blocks:x=block(x)\n        return self.head(x)\n\ndef multitask_loss(pred,target,auxiliary_weight=.2):\n    error=(pred-target[:,None,:]).square()\n    # Compute each head\'s loss before averaging. Opposing heads cannot cancel.\n    per_head=error[:,:,0].mean(dim=0)+auxiliary_weight*error[:,:,1].mean(dim=0)\n    return per_head.mean()\n\nclass TabMRegressor:\n    def __init__(self,cfg,seed=2026,iterations=None,hardware=None):\n        self.cfg=cfg;self.seed=seed;self.iterations=iterations;self.hardware=hardware or {}\n    def device_name(self):\n        if self.cfg[\'device\']==\'cpu\':return \'cpu\'\n        if torch.cuda.is_available():return \'cuda:0\'\n        if torch.backends.mps.is_available():return \'mps\'\n        return \'cpu\'\n    def fit(self,x,y,xvalid=None,yvalid=None):\n        seed_everything(self.seed);torch.manual_seed(self.seed);torch.set_num_threads(self.cfg[\'threads\'])\n        if torch.cuda.is_available():torch.cuda.manual_seed_all(self.seed)\n        torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\n        self.device=self.device_name();p=self.cfg[\'tabm\'];amp=self.device.startswith(\'cuda\')\n        self.preprocessor=NNPreprocessor().fit(x)\n        num,cat=self.preprocessor.transform(x);target=torch.from_numpy(y)\n        self.arch={key:p[key] for key in [\'k\',\'hidden\',\'blocks\',\'frequencies\',\'dropout\']}\n        self.arch.update(n_cont=num.shape[1],cardinalities=self.preprocessor.cardinalities)\n        self.model=MultiTaskTabM(**self.arch).to(self.device)\n        num=torch.from_numpy(num);cat=torch.from_numpy(cat)\n        optimizer=torch.optim.AdamW(self.model.parameters(),lr=p[\'learning_rate\'],weight_decay=p[\'weight_decay\'])\n        epochs=self.iterations or p[\'epochs\'];scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)\n        scaler=torch.amp.GradScaler(\'cuda\',enabled=amp)\n        self.history=[];best=float(\'inf\');bad=0;best_state=None\n        valid_prepared=None if xvalid is None else self.preprocessor.transform(xvalid)\n        valid_true=None if yvalid is None else yvalid[:,0]+xvalid.anchor_board_log.to_numpy()\n        batch=p[\'batch_size\'];rng=np.random.default_rng(self.seed)\n        for epoch in range(1,epochs+1):\n            start=time.perf_counter();self.model.train();total=0;order=rng.permutation(len(x))\n            for offset in range(0,len(x),batch):\n                ix=torch.from_numpy(order[offset:offset+batch]);a=num[ix].to(self.device);b=cat[ix].to(self.device);t=target[ix].to(self.device)\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=\'cuda\' if amp else \'cpu\',dtype=torch.float16,enabled=amp):\n                    loss=multitask_loss(self.model(a,b),t,p[\'auxiliary_weight\'])\n                if not torch.isfinite(loss):raise FloatingPointError(\'Nonfinite TabM loss\')\n                scaler.scale(loss).backward();scaler.unscale_(optimizer)\n                nn.utils.clip_grad_norm_(self.model.parameters(),5.0)\n                scaler.step(optimizer);scaler.update();total+=loss.item()*len(ix)\n            scheduler.step();val=None\n            if xvalid is not None:\n                pred=self._predict_prepared(valid_prepared,xvalid.anchor_board_log.to_numpy())\n                val=float(np.sqrt(np.mean((pred-valid_true)**2)))\n                if val<best-1e-7:\n                    best=val;bad=0;self.best_iteration=epoch\n                    best_state={k:v.detach().cpu().clone() for k,v in self.model.state_dict().items()}\n                else:bad+=1\n            else:self.best_iteration=epoch\n            row={\'epoch\':epoch,\'train_loss\':total/len(x),\'valid_rmsle\':val,\'seconds\':time.perf_counter()-start,\'learning_rate\':optimizer.param_groups[0][\'lr\']}\n            self.history.append(row);print(row,flush=True)\n            if xvalid is not None and bad>=p[\'patience\']:break\n        if best_state is not None:self.model.load_state_dict(best_state)\n        return self\n    def _predict_prepared(self,prepared,anchor):\n        self.model.eval();num,cat=prepared;out=[];batch=self.cfg[\'tabm\'][\'batch_size\']\n        with torch.no_grad():\n            for start in range(0,len(num),batch):\n                a=torch.from_numpy(num[start:start+batch]).to(self.device);b=torch.from_numpy(cat[start:start+batch]).to(self.device)\n                residual=self.model(a,b)[:,:,0].float().cpu().numpy()\n                # Clip each member in log space BEFORE the K-member average.\n                out.append(np.maximum(0,anchor[start:start+batch,None]+residual).mean(axis=1))\n        return np.concatenate(out).astype(\'float64\')\n    def predict_log(self,x):return self._predict_prepared(self.preprocessor.transform(x),x.anchor_board_log.to_numpy())\n    def to_cpu(self):self.model.to(\'cpu\');self.device=\'cpu\';return self\n', 'src/tune_lightgbm.py': '"""Bounded LightGBM search with temporal early stopping and frozen-round refits."""\nfrom __future__ import annotations\nimport argparse\nimport copy\nimport gc\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom lightgbm import LGBMRegressor,early_stopping,record_evaluation\nfrom .features import load_data,make_features,training_matrix\nfrom .models import Model,lightgbm_parameters\nfrom .pipeline import save_json,environment,diagnostics,export_predictions\nfrom .metrics import regression_metrics\n\nROOT=Path(__file__).resolve().parents[1]\nCATEGORICAL=[\'station_id\',\'canonical_line\',\'hour\',\'dow\']\n\ndef clipped_log_rmsle(y_log,pred_log):\n    """Targets are already log1p transformed; match deployed zero clipping."""\n    return \'rmsle\',float(np.sqrt(np.mean((np.maximum(np.asarray(pred_log),0)-np.asarray(y_log))**2))),False\n\ndef choose_common_round(curves,row_counts,max_rounds=1000):\n    if len(curves)!=len(row_counts) or not curves or any(n<=0 for n in row_counts):\n        raise ValueError(\'Nonempty curves and positive row counts required\')\n    common=min(max_rounds,*(len(c) for c in curves))\n    if common<1:raise ValueError(\'No common observed iterations\')\n    values=np.array([c[:common] for c in curves],dtype=np.float64)\n    if not np.isfinite(values).all() or np.any(values<0):raise ValueError(\'Invalid validation curve\')\n    pooled=np.sqrt(np.average(values**2,axis=0,weights=np.asarray(row_counts)))\n    iteration=int(np.argmin(pooled))+1\n    return iteration,float(pooled[iteration-1]),common,pooled\n\ndef candidate_parameters(seed=2026,trials=16):\n    base=dict(learning_rate=.055,num_leaves=63,min_child_samples=80,reg_lambda=3.,\n              reg_alpha=0.,max_depth=-1,colsample_bytree=.9,subsample=.9,subsample_freq=1,min_split_gain=0.)\n    variants=[{},dict(learning_rate=.03,min_child_samples=160,reg_lambda=10.,max_depth=10),\n              dict(learning_rate=.08,num_leaves=31,min_child_samples=160,reg_lambda=10.,max_depth=8),\n              dict(learning_rate=.03,num_leaves=127,min_child_samples=160,reg_lambda=10.,max_depth=12)]\n    candidates=[dict(base,**v) for v in variants]\n    rng=np.random.default_rng(seed)\n    while len(candidates)<trials:\n        proposed=dict(base,learning_rate=float(rng.choice([.025,.04,.055,.08])),\n                      num_leaves=int(rng.choice([31,63,95,127])),\n                      min_child_samples=int(rng.choice([80,160,320])),\n                      reg_lambda=float(rng.choice([3.,10.,30.])),\n                      reg_alpha=float(rng.choice([0.,.1,1.])),\n                      max_depth=int(rng.choice([8,10,12,-1])),\n                      colsample_bytree=float(rng.choice([.75,.9,1.])),\n                      subsample=float(rng.choice([.75,.9])),\n                      min_split_gain=float(rng.choice([0.,.01])))\n        if proposed not in candidates:candidates.append(proposed)\n    return candidates[:trials]\n\ndef prepare_fold(train,meta,origin,cfg):\n    origin=pd.Timestamp(origin)\n    history=train.loc[train.date<=origin]\n    valid=train.loc[(train.date>origin)&(train.date<=origin+pd.Timedelta(days=cfg[\'horizon_days\']))].copy().reset_index(drop=True)\n    if valid.empty or valid.date.max()!=origin+pd.Timedelta(days=cfg[\'horizon_days\']):\n        raise ValueError(\'Complete 123-day validation required\')\n    known=valid.station_id.isin(history.station_id.unique()).to_numpy()\n    x,y,anchors=training_matrix(train,meta,origin,cfg[\'horizon_days\'],cfg[\'origin_stride_days\'],cfg[\'max_training_rows\'],cfg[\'seed\'])\n    xv=make_features(train,valid,meta,origin)\n    assert x.columns.equals(xv.columns)\n    assert max(pd.Timestamp(a[\'label_end\']) for a in anchors)<=origin\n    return {\'origin\':origin,\'valid\':valid,\'known\':known,\'x\':x,\'y\':y,\'xv\':xv,\'anchors\':anchors}\n\ndef search_development(train,meta,cfg,output,trials,max_rounds,patience):\n    if max_rounds!=1000:raise ValueError(\'User requested a hard cap of 1000 rounds\')\n    dev_end=pd.Timestamp(cfg[\'fold_origins\'][1])+pd.Timedelta(days=cfg[\'horizon_days\'])\n    development=train.loc[train.date<=dev_end].copy()\n    folds=[prepare_fold(development,meta,origin,cfg) for origin in cfg[\'fold_origins\'][:2]]\n    candidates=candidate_parameters(cfg[\'seed\'],trials)\n    design={\'optimizer\':\'LightGBM gradient/tree boosting, not AdamW\',\n            \'max_boosting_rounds\':max_rounds,\'early_stopping_rounds\':patience,\'min_delta\':1e-5,\n            \'metric\':\'RMSLE after deployed zero clipping in log space\',\'trials\':trials,\n            \'candidate_parameters\':candidates,\'development_label_end\':str(dev_end.date()),\n            \'selection\':\'minimum row-weighted pooled log-MSE on common observed iteration prefix\',\n            \'station_policy\':\'validation stations observed on or before each origin; actual test has no unseen station\',\n            \'global_optimum_guaranteed\':False,\n            \'folds\':[{\'origin\':str(f[\'origin\'].date()),\'all_rows\':len(f[\'valid\']),\'selection_rows\':int(f[\'known\'].sum()),\'excluded_cold_rows\':int((~f[\'known\']).sum()),\'training_rows\':len(f[\'y\'])} for f in folds]}\n    save_json(output/\'search_design.json\',design)\n    rows=[];best=None\n    for trial,parameters in enumerate(candidates,1):\n        t=time.perf_counter();curves=[];fold_info=[]\n        folder=output/\'search\'/f\'trial_{trial:02d}\';folder.mkdir(parents=True,exist_ok=True)\n        print(f\'Trial {trial}/{trials}: leaves={parameters["num_leaves"]}, lr={parameters["learning_rate"]}, L2={parameters["reg_lambda"]}\',flush=True)\n        for i,f in enumerate(folds,1):\n            fit_cfg=dict(cfg,lightgbm_params=dict(parameters,n_estimators=max_rounds))\n            params=lightgbm_parameters(fit_cfg);params[\'metric\']=\'None\'\n            model=LGBMRegressor(**params)\n            mask=f[\'known\'];yv=np.log1p(f[\'valid\'].loc[mask,\'boardings\'].to_numpy(dtype=np.float64))\n            history={}\n            model.fit(f[\'x\'],np.log1p(f[\'y\']),categorical_feature=CATEGORICAL,\n                      eval_set=[(f[\'x\'],np.log1p(f[\'y\'])),(f[\'xv\'].loc[mask],yv)],\n                      eval_names=[\'training\',\'validation\'],eval_metric=clipped_log_rmsle,\n                      callbacks=[record_evaluation(history),early_stopping(patience,first_metric_only=True,verbose=False,min_delta=1e-5)])\n            curve=history[\'validation\'][\'rmsle\'];curves.append(curve)\n            pd.DataFrame({\'iteration\':np.arange(1,len(curve)+1),\'training_rmsle\':history[\'training\'][\'rmsle\'],\'validation_rmsle\':curve}).to_csv(folder/f\'fold_{i}_learning_curve.csv\',index=False)\n            info={\'fold\':i,\'best_iteration\':model.best_iteration_,\'recorded_iterations\':len(curve),\n                  \'retained_trees\':model.n_estimators_,\'early_stopped_before_cap\':len(curve)<max_rounds,\n                  \'cap_reached\':len(curve)==max_rounds,\'best_validation_rmsle\':model.best_score_[\'validation\'][\'rmsle\']}\n            fold_info.append(info)\n            print(f\'  dev{i}: best={model.best_iteration_}, observed={len(curve)}, RMSLE={info["best_validation_rmsle"]:.6f}\',flush=True)\n            del model;gc.collect()\n        iteration,value,common,pooled=choose_common_round(curves,[int(f[\'known\'].sum()) for f in folds],max_rounds)\n        pd.DataFrame({\'iteration\':np.arange(1,common+1),\'pooled_rmsle\':pooled}).to_csv(folder/\'common_validation_curve.csv\',index=False)\n        row={\'trial\':trial,\'selected_iteration\':iteration,\'pooled_development_rmsle\':value,\'common_observed_iterations\':common,\'seconds\':time.perf_counter()-t,**parameters}\n        rows.append(row)\n        save_json(folder/\'trial.json\',{\'summary\':row,\'folds\':fold_info})\n        pd.DataFrame(rows).to_csv(output/\'search_results.csv\',index=False)\n        if best is None or (value,iteration)<(best[\'pooled_development_rmsle\'],best[\'selected_iteration\']):\n            best={**row,\'parameters\':parameters,\'folds\':fold_info}\n        print(f\'  common selection={iteration}, pooled RMSLE={value:.6f} ({row["seconds"]:.1f}s)\',flush=True)\n    save_json(output/\'selected_plan.json\',{\'selection_frozen_before_fold3\':True,\'selection_uses_fold3_labels\':False,\n                                         \'existing_fold3_previously_analyzed\':True,\'global_optimum_guaranteed\':False,\n                                         \'search_design\':design,\'best_trial\':best})\n    return best,folds\n\ndef fit_frozen_fold(f,cfg,path,train):\n    path.mkdir(parents=True,exist_ok=True)\n    model=Model(\'lightgbm_large\',cfg).fit(f[\'x\'],f[\'y\'])\n    pred=model.predict(f[\'xv\'])\n    frame=f[\'valid\'][[\'date\',\'station_id\',\'hour\',\'boardings\']].copy()\n    frame[\'horizon\']=(frame.date-f[\'origin\']).dt.days\n    frame[\'known_at_origin\']=f[\'known\'];frame[\'lightgbm_large\']=pred;frame[\'ensemble\']=pred\n    frame.to_csv(path/\'oof.csv.gz\',index=False,compression=\'gzip\')\n    save_json(path/\'training_origins.json\',f[\'anchors\'])\n    summary=diagnostics(train,f[\'valid\'],pred,f[\'origin\'],path)\n    known=regression_metrics(f[\'valid\'].loc[f[\'known\'],\'boardings\'],pred[f[\'known\']])\n    save_json(path/\'known_station_metrics.json\',known)\n    save_json(path/\'fit_metadata.json\',{\'actual_iterations\':model.estimator.n_estimators_,\'rounds_frozen_from_development\':True})\n    return frame,summary,known\n\ndef run(args):\n    started=time.perf_counter();out=args.output\n    if (out/\'model_bundle.joblib\').exists():raise FileExistsError(\'Choose a new output directory to preserve the previous tuned run\')\n    out.mkdir(parents=True,exist_ok=True)\n    cfg=json.loads(args.config.read_text())\n    data=load_data(args.data_dir);train=data[\'train\'];meta=data[\'metaData\']\n    fingerprint={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in args.data_dir.glob(\'*.csv\')}\n    save_json(out/\'input_sha256.json\',fingerprint);save_json(out/\'environment.json\',environment())\n    best,dev_folds=search_development(train,meta,cfg,out,args.trials,args.max_rounds,args.patience)\n    selected=copy.deepcopy(cfg)\n    selected[\'models\']=[\'lightgbm_large\']\n    selected[\'lightgbm_params\']=dict(best[\'parameters\'],n_estimators=best[\'selected_iteration\'])\n    selected[\'ensemble_policy\']={\'method\':\'primary_constrained_log_mse\',\'primary_model\':\'lightgbm_large\',\'primary_min_weight\':1.,\'bias_correction\':False}\n    selected[\'tuning\']={\'method\':\'16-candidate bounded development search with early stopping\',\'max_rounds\':1000,\'patience\':args.patience,\'min_delta\':1e-5,\'actual_trials\':args.trials,\'selected_iteration\':best[\'selected_iteration\'],\'selected_trial\':best[\'trial\'],\'search_output\':str(out)}\n    selected[\'evaluation_context\']={\'status\':\'reused_validation_after_prior_diagnostics\',\'policy_selection_has_seen_audit\':True,\'weight_fit_uses_audit\':False,\'hyperparameter_selection_uses_audit\':False,\'note\':\'Hyperparameters and common boosting iteration selected using first two development folds, known stations only. Fold 3 already examined in earlier work; reused comparison only.\'}\n    save_json(out/\'config_resolved.json\',selected)\n    # Development refits are needed because an early-stopped booster may retain\n    # fewer trees than the common selected round recorded in its patience window.\n    metrics=[];fold_records=[];known_scores=[]\n    for i,f in enumerate(dev_folds,1):\n        frame,summary,known=fit_frozen_fold(f,selected,out/f\'fold_{i}\',train)\n        metrics.append({\'fold\':i,\'role\':\'development\',\'model\':\'lightgbm_large\',**summary})\n        known_scores.append((len(frame.loc[frame.known_at_origin]),known[\'rmsle\']))\n        fold_records.append({\'fold\':i,\'origin\':str(f[\'origin\'].date()),\'rows\':len(frame),\'selection_rows\':int(f[\'known\'].sum())})\n    confirmed=float(np.sqrt(sum(n*v*v for n,v in known_scores)/sum(n for n,v in known_scores)))\n    if abs(confirmed-best[\'pooled_development_rmsle\'])>2e-7:raise AssertionError(\'Fixed-round refits did not reproduce selected development score\')\n    del dev_folds;gc.collect()\n    # The third fold is accessed for evaluation only after selection is frozen.\n    audit=prepare_fold(train,meta,selected[\'fold_origins\'][2],selected)\n    frame,summary,known=fit_frozen_fold(audit,selected,out/\'fold_3\',train)\n    save_json(out/\'audit_summary.json\',summary)\n    metrics.append({\'fold\':3,\'role\':\'reused_comparison\',\'model\':\'lightgbm_large\',**summary})\n    fold_records.append({\'fold\':3,\'origin\':str(audit[\'origin\'].date()),\'rows\':len(frame),\'selection_rows\':0})\n    pd.DataFrame(metrics).to_csv(out/\'cv_scores.csv\',index=False);save_json(out/\'cv_folds.json\',fold_records)\n    reference=frame.lightgbm_large.to_numpy()\n    print(f\'Frozen-round comparison RMSLE={summary["rmsle"]:.9f}; round={best["selected_iteration"]}\',flush=True)\n    del audit;gc.collect()\n    cutoff=train.date.max()\n    x,y,anchors=training_matrix(train,meta,cutoff,selected[\'horizon_days\'],selected[\'origin_stride_days\'],selected[\'max_training_rows\'],selected[\'seed\'])\n    xt=make_features(train,data[\'test\'],meta,cutoff)\n    model=Model(\'lightgbm_large\',selected).fit(x,y)\n    pred=model.predict(xt)\n    plan={\'weights\':{\'lightgbm_large\':1.},\'bias\':0.,\'blend_space\':\'log1p\',\'bias_space\':\'log1p\',\n          \'retained\':[\'lightgbm_large\'],\'excluded\':{},\'metric\':\'rmsle\',\'fit_folds\':[1,2],\n          \'evaluation_context\':selected[\'evaluation_context\']}\n    bundle={\'config\':selected,\'ensemble\':plan,\'models\':{\'lightgbm_large\':model},\'features\':x.columns.tolist(),\n            \'training_cutoff\':str(cutoff.date()),\'input_sha256\':fingerprint}\n    joblib.dump(bundle,out/\'model_bundle.joblib\',compress=3)\n    model.estimator.booster_.save_model(str(out/\'lightgbm_tuned.txt\'))\n    save_json(out/\'ensemble_plan.json\',plan);save_json(out/\'final_training_origins.json\',anchors)\n    export_predictions(data,pred,out,reference)\n    loaded=joblib.load(out/\'model_bundle.joblib\')\n    restored=loaded[\'models\'][\'lightgbm_large\'].predict(xt)\n    np.testing.assert_array_equal(pred,restored)\n    export_predictions(data,restored,out/\'reloaded\',reference)\n    old_hash=hashlib.sha256((out/\'submission.csv\').read_bytes()).hexdigest()\n    assert old_hash==hashlib.sha256((out/\'reloaded/submission.csv\').read_bytes()).hexdigest()\n    save_json(out/\'verification.json\',{\'reload_bitwise_identical\':True,\'submission_sha256\':old_hash,\n                                      \'selected_development_rmsle\':best[\'pooled_development_rmsle\'],\n                                      \'refitted_development_rmsle\':confirmed,\'actual_final_trees\':model.estimator.n_estimators_,\n                                      \'hard_cap_satisfied\':model.estimator.n_estimators_<=1000,\'final_fit_uses_frozen_rounds\':True,\n                                      \'fold3_not_used_for_early_stopping_or_parameter_selection\':True})\n    save_json(out/\'run_summary.json\',{\'status\':\'completed\',\'elapsed_seconds\':time.perf_counter()-started,\n                                    \'models_deployed\':[\'lightgbm_large\'],\'feature_count\':len(x.columns),\n                                    \'final_training_rows\':len(y),\'max_search_rounds\':1000,\n                                    \'selected_rounds\':best[\'selected_iteration\'],\'selected_trial\':best[\'trial\'],\n                                    \'trials\':args.trials,\'development_rmsle\':confirmed,\'audit_rmsle\':summary[\'rmsle\'],\n                                    \'previous_lightgbm_audit_rmsle\':.2539725321304833,\'global_optimum_guaranteed\':False,\n                                    \'submission_rows\':len(pred),\'optimizer\':\'gradient tree boosting; not AdamW\'})\n    print(f\'Completed: {out}; selected rounds={best["selected_iteration"]}; RMSLE={summary["rmsle"]:.9f}\',flush=True)\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--data-dir\',type=Path,default=ROOT/\'data/raw\')\n    p.add_argument(\'--config\',type=Path,default=ROOT/\'configs/three_tree.json\')\n    p.add_argument(\'--output\',type=Path,default=ROOT/\'outputs/lightgbm_tuned\')\n    p.add_argument(\'--trials\',type=int,default=16)\n    p.add_argument(\'--max-rounds\',type=int,default=1000)\n    p.add_argument(\'--patience\',type=int,default=100)\n    args=p.parse_args()\n    if args.trials<1 or args.patience<1:raise ValueError(\'Trials and patience must be positive\')\n    run(args)\n\nif __name__==\'__main__\':main()\n', 'src/validate_submission.py': '"""Strict SINCON submission validation; never repairs or prunes input rows."""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\nEXPECTED_COLUMNS = ["example_id", "boardings"]\n\n\ndef _read(frame_or_path) -> pd.DataFrame:\n    if isinstance(frame_or_path, pd.DataFrame):\n        return frame_or_path.copy()\n    return pd.read_csv(frame_or_path, dtype={"example_id": "string"})\n\n\ndef _ids(frame: pd.DataFrame, name: str) -> pd.Series:\n    if "example_id" not in frame:\n        raise ValueError(f"{name}: example_id column is missing.")\n    ids = frame["example_id"]\n    if ids.isna().any() or ids.astype(str).str.strip().eq("").any():\n        raise ValueError(f"{name}: example_id contains missing/empty values.")\n    if not ids.map(lambda x: isinstance(x, str)).all():\n        raise ValueError(f"{name}: example_id must be strings; preserve original identifiers.")\n    if ids.duplicated().any():\n        raise ValueError(f"{name}: duplicate example_id values.")\n    return ids.astype(str).reset_index(drop=True)\n\n\ndef _distribution(values) -> dict:\n    v = np.asarray(values, dtype=float).reshape(-1)\n    if len(v) == 0 or not np.isfinite(v).all():\n        raise ValueError("Distribution input must contain finite predictions.")\n    return {\n        "count": len(v), "min": float(v.min()), "mean": float(v.mean()),\n        "p01": float(np.quantile(v, 0.01)), "p50": float(np.quantile(v, 0.5)),\n        "p99": float(np.quantile(v, 0.99)), "max": float(v.max()),\n        "zero_fraction": float((v == 0).mean()),\n    }\n\n\ndef validate_submission(submission, test, sample, scope: str = "test", reference_predictions=None) -> dict:\n    """Validate exact schema, scope-specific ID order, values, and scope mismatch.\n\n    ``test`` is the officially prescribed 123-day scope and expects every test\n    ID. ``template`` checks the stale 92-day sample only for diagnostics; passing\n    that check does not make its file ready for the official full-test scope.\n    The discrepancy is reported in either mode and never silently resolved.\n    """\n    if scope not in ("template", "test"):\n        raise ValueError("scope must be \'template\' or \'test\'.")\n    sub, tst, tmpl = (_read(x) for x in (submission, test, sample))\n    if list(sub.columns) != EXPECTED_COLUMNS:\n        raise ValueError(f"Submission columns/order must equal {EXPECTED_COLUMNS}, got {list(sub.columns)}.")\n    if list(tmpl.columns) != EXPECTED_COLUMNS:\n        raise ValueError(f"Sample columns/order must equal {EXPECTED_COLUMNS}.")\n    sub_ids, test_ids, sample_ids = _ids(sub, "submission"), _ids(tst, "test"), _ids(tmpl, "sample")\n    if len(sub) == 0:\n        raise ValueError("Submission cannot be empty.")\n    expected = sample_ids if scope == "template" else test_ids\n    if len(sub_ids) != len(expected):\n        raise ValueError(f"Submission has {len(sub_ids)} rows but {scope} scope requires {len(expected)}.")\n    if not np.array_equal(sub_ids.to_numpy(), expected.to_numpy()):\n        raise ValueError(f"Submission example_id values/order differ from {scope} scope.")\n    pred = pd.to_numeric(sub["boardings"], errors="raise").to_numpy(dtype=float)\n    if not np.isfinite(pred).all():\n        raise ValueError("Submission predictions contain NaN or infinite values.")\n    if (pred < 0).any():\n        raise ValueError("Submission predictions contain negative boardings.")\n    test_set, sample_set = set(test_ids), set(sample_ids)\n    missing_from_template = test_set - sample_set\n    unknown_template_ids = sample_set - test_set\n    if unknown_template_ids:\n        raise ValueError(f"Sample contains {len(unknown_template_ids)} IDs absent from test.")\n    warnings = []\n    scope_agrees = np.array_equal(test_ids.to_numpy(), sample_ids.to_numpy())\n    if not scope_agrees:\n        warnings.append(\n            f"Stale sample template: sample={len(sample_ids)} rows, test={len(test_ids)} rows; "\n            f"{len(missing_from_template)} test IDs are absent from the sample. "\n            "Official submission scope is the full test ID sequence. "\n            + ("Validated that full test scope." if scope == "test" else "Template scope is diagnostic only and is not the official submission scope.")\n        )\n    mismatch = {"template_rows": len(sample_ids), "test_rows": len(test_ids), "test_ids_absent_from_template": len(missing_from_template)}\n    if "date" in tst and missing_from_template:\n        dates = tst.loc[tst["example_id"].astype(str).isin(missing_from_template), "date"].astype(str)\n        mismatch.update({"missing_period_start": dates.min(), "missing_period_end": dates.max()})\n    result = {\n        "valid": True, "scope": scope, "rows": len(sub), "columns": list(sub.columns),\n        "official_scope": "test", "matches_official_scope": scope == "test",\n        "exact_id_order": True, "finite_nonnegative": True,\n        "template_matches_test": bool(scope_agrees), "scope_discrepancy": mismatch,\n        "warnings": warnings, "prediction_distribution": _distribution(pred),\n    }\n    if reference_predictions is not None:\n        ref = _distribution(reference_predictions)\n        result["reference_prediction_distribution"] = ref\n        result["distribution_comparison"] = {\n            "mean_ratio": result["prediction_distribution"]["mean"] / ref["mean"] if ref["mean"] != 0 else None,\n            "median_ratio": result["prediction_distribution"]["p50"] / ref["p50"] if ref["p50"] != 0 else None,\n            "note": "Diagnostic only: forecast dates/composition differ; ratios are not automatic rejection rules.",\n        }\n    return result\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--submission", required=True, type=Path)\n    parser.add_argument("--test", required=True, type=Path)\n    parser.add_argument("--sample", required=True, type=Path)\n    parser.add_argument("--scope", choices=("template", "test"), default="test")\n    parser.add_argument("--reference", type=Path, help="Optional validation prediction CSV.")\n    parser.add_argument("--reference-column", default="prediction")\n    parser.add_argument("--report", type=Path, help="Optional JSON output path.")\n    args = parser.parse_args(argv)\n    try:\n        ref = None\n        if args.reference:\n            reference = pd.read_csv(args.reference)\n            ref = reference[args.reference_column].to_numpy()\n        result = validate_submission(args.submission, args.test, args.sample, args.scope, ref)\n    except (ValueError, KeyError, OSError) as error:\n        print(f"Submission validation failed: {error}", file=sys.stderr)\n        return 2\n    content = json.dumps(result, indent=2, ensure_ascii=False, allow_nan=False)\n    print(content)\n    if args.report:\n        args.report.parent.mkdir(parents=True, exist_ok=True)\n        args.report.write_text(content + "\\n", encoding="utf-8")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/xgb_compat.py': '"""Use raw validation data so early stopping measures the inference path.\n\nTraining keeps QuantileDMatrix/hist. The installed XGBoost 3.4.1 produced\ndifferent predictions with reference-category validation metadata versus\nraw DMatrix/inplace prediction in this dataset. ResidualTree.frame fixes\nidentical categorical levels/codes for training, validation and inference,\nso validation needs no additional reference-category recoding. This adapter\nchanges only validation construction, not training or hyperparameters.\n"""\nfrom xgboost import XGBRegressor,DMatrix\n\nclass InferenceConsistentXGBRegressor(XGBRegressor):\n    def _create_dmatrix(self,ref,**kwargs):\n        if ref is not None:\n            kwargs[\'feature_types\']=None\n            return DMatrix(**kwargs,nthread=self.n_jobs)\n        return super()._create_dmatrix(ref=ref,**kwargs)\n', 'tests/test_ensemble.py': '"""OOF blend constraints, duplicate filtering, and MAE bias correction."""\nfrom pathlib import Path\nimport sys\nimport unittest\n\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom src.ensemble import fit_ensemble, predict_ensemble\nfrom src.metrics import score\n\n\nclass EnsembleTests(unittest.TestCase):\n    def test_single_model_identity_preserves_exact_predictions(self):\n        values=np.array([.123456785,13.456789125,1000.123456785],dtype=np.float64)\n        plan={\'weights\':{\'only\':1.,\'unused\':0.},\'bias\':0.,\'blend_space\':\'log1p\'}\n        np.testing.assert_array_equal(predict_ensemble({\'only\':values},plan),values)\n\n    def test_primary_constraint_and_no_bias_match_boundary_optimum(self):\n        truth=np.expm1(np.array([2.,3.,4.,5.]))\n        predictions={\'large\':np.expm1(np.log1p(truth)+.4),\n                     \'cat\':np.expm1(np.log1p(truth)-.1),\n                     \'xgb\':np.expm1(np.log1p(truth)+.6)}\n        policy={\'method\':\'primary_constrained_log_mse\',\'primary_model\':\'large\',\n                \'primary_min_weight\':.5,\'bias_correction\':False}\n        plan,_=fit_ensemble(truth,predictions,policy=policy)\n        # Unconstrained optimum can cancel bias with large=.2, cat=.8.\n        # Requiring large>=.5 makes [.5,.5,0] the known constrained optimum.\n        np.testing.assert_allclose([plan[\'weights\'][n] for n in predictions],[.5,.5,0],atol=1e-8)\n        self.assertEqual(plan[\'bias\'],0.)\n        self.assertEqual(plan[\'excluded\'],{})\n        self.assertAlmostEqual(plan[\'development_score\'],.15,places=10)\n\n    def test_primary_blend_keeps_correlated_models_and_is_order_invariant(self):\n        truth=np.expm1(np.array([2.,3.,4.,5.]))\n        error=np.array([.1,-.2,.3,-.1])\n        predictions={\'large\':np.expm1(np.log1p(truth)+error),\n                     \'cat\':np.expm1(np.log1p(truth)+error*.9),\n                     \'xgb\':np.expm1(np.log1p(truth)-error*.7)}\n        policy={\'method\':\'primary_constrained_log_mse\',\'primary_model\':\'large\',\n                \'primary_min_weight\':.5,\'bias_correction\':False}\n        plan,corr=fit_ensemble(truth,predictions,policy=policy)\n        other,_=fit_ensemble(truth,dict(reversed(list(predictions.items()))),policy=policy)\n        self.assertGreater(corr.loc[\'large\',\'cat\'],.95)\n        self.assertEqual(set(plan[\'retained\']),set(predictions))\n        self.assertAlmostEqual(sum(plan[\'weights\'].values()),1.,places=12)\n        self.assertGreaterEqual(plan[\'weights\'][\'large\'],.5-1e-12)\n        np.testing.assert_allclose(predict_ensemble(predictions,plan),predict_ensemble(predictions,other),rtol=1e-8)\n\n    def test_primary_blend_rejects_invalid_inputs(self):\n        policy={\'method\':\'primary_constrained_log_mse\',\'primary_model\':\'large\',\'primary_min_weight\':.5}\n        with self.assertRaises(ValueError):fit_ensemble([1.,2.],{\'large\':np.array([1.,np.nan])},policy=policy)\n        with self.assertRaises(ValueError):fit_ensemble([1.,2.],{\'large\':np.array([-1.,2.])},policy=policy)\n        with self.assertRaises(ValueError):fit_ensemble([1.,2.],{\'cat\':np.array([1.,2.])},policy=policy)\n        with self.assertRaises(ValueError):fit_ensemble([1.,2.],{\'large\':np.array([1.,2.])},metric=\'mae\',policy=policy)\n\n    def test_convex_weights_and_duplicate_residual_filter(self):\n        truth = np.array([10., 15., 20., 25., 30., 35., 40., 45.])\n        error = np.array([1., -3., 4., -1., 2., -4., 3., -2.])\n        predictions = {"model_a": truth + error, "duplicate_a": truth + error, "model_b": truth - error * .8 + 0.4}\n        plan, correlation = fit_ensemble(truth, predictions, metric="mae")\n        self.assertTrue(all(weight >= 0 for weight in plan["weights"].values()))\n        self.assertAlmostEqual(sum(plan["weights"].values()), 1., places=12)\n        self.assertLessEqual(len(set(plan["retained"]) & {"model_a", "duplicate_a"}), 1)\n        for i, left in enumerate(plan["retained"]):\n            for right in plan["retained"][i + 1:]:\n                self.assertFalse(correlation.loc[left, right] > .95)\n        blended = predict_ensemble(predictions, plan)\n        self.assertTrue(np.isfinite(blended).all())\n        self.assertTrue((blended >= 0).all())\n        self.assertLessEqual(score(truth, blended, \'mae\'), min(score(truth, p, \'mae\') for p in predictions.values()))\n\n    def test_mae_bias_uses_median_and_improves_objective(self):\n        truth = np.array([10., 20., 30., 40., 50.])\n        # Residual mean is -0.2 while the median correction is +5.\n        predictions = {"only": truth + np.array([-5., -5., -5., -5., 19.])}\n        plan, _ = fit_ensemble(truth, predictions, metric="mae")\n        self.assertEqual(plan["bias"], 5.)\n        result = predict_ensemble(predictions, plan)\n        self.assertLess(score(truth, result, \'mae\'), score(truth, predictions["only"], \'mae\'))\n        self.assertAlmostEqual(plan["development_score"], score(truth, result, \'mae\'))\n\n    def test_rmse_does_not_apply_mae_median_correction(self):\n        truth = np.array([10., 20., 30.])\n        predictions = {"only": truth - 5}\n        plan, _ = fit_ensemble(truth, predictions, metric="rmse")\n        self.assertEqual(plan["bias"], 0.)\n\n    def test_rmsle_blends_and_calibrates_in_log1p_space(self):\n        truth_log = np.array([1., 2., 3., 4., 5.])\n        truth = np.expm1(truth_log)\n        residual_log = np.array([-.2, -.2, -.2, -.2, .6])\n        predictions = {"only": np.expm1(truth_log + residual_log)}\n        plan, _ = fit_ensemble(truth, predictions)\n        self.assertEqual(plan["metric"], "rmsle")\n        self.assertEqual(plan["blend_space"], "log1p")\n        self.assertEqual(plan["bias_space"], "log1p")\n        # RMSLE minimizes squared error in log space, so use mean, not median.\n        self.assertAlmostEqual(plan["bias"], .04, places=12)\n        expected = np.expm1(truth_log + residual_log + .04)\n        actual = predict_ensemble(predictions, plan)\n        np.testing.assert_allclose(actual, expected)\n        self.assertLess(score(truth, actual, "rmsle"), score(truth, predictions["only"], "rmsle"))\n\n    def test_predict_ensemble_uses_weighted_geometric_mean_of_one_plus_counts(self):\n        predictions = {"a": np.array([0., 3., 8.]), "b": np.array([3., 8., 15.])}\n        plan = {"weights": {"a": .25, "b": .75}, "bias": .1, "blend_space": "log1p"}\n        expected = np.exp(.1) * (1 + predictions["a"])**.25 * (1 + predictions["b"])**.75 - 1\n        np.testing.assert_allclose(predict_ensemble(predictions, plan), expected)\n        plan["bias"] = -10.\n        np.testing.assert_array_equal(predict_ensemble(predictions, plan), np.zeros(3))\n\n    def test_constant_residual_duplicates_are_removed_even_when_correlation_undefined(self):\n        truth = np.array([10., 20., 30.])\n        predictions = {"a": truth + 5, "duplicate": truth + 5}\n        plan, correlation = fit_ensemble(truth, predictions, metric="mae")\n        self.assertTrue(np.isnan(correlation.loc["a", "duplicate"]))\n        self.assertEqual(len(plan["retained"]), 1)\n        self.assertEqual(len(plan["excluded"]), 1)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'tests/test_ensemble_v2.py': "import unittest\nimport pickle\nimport numpy as np\nimport pandas as pd\nfrom src.features_v2 import cleaned,calendar,make_features_v2,partial_origins,sampled_pairs\nfrom src.nn_preprocessor import NNPreprocessor\nfrom src.ensemble_v2 import convex_weights,evaluate_ensemble\n\ndef fixture(start='2025-01-01',end='2026-06-30',stations=(3110,3111,3107)):\n    index=pd.MultiIndex.from_product([pd.date_range(start,end),stations,[6,7]],names=['date','station_id','hour'])\n    frame=index.to_frame(index=False);frame['boardings']=10.;frame['alightings']=20.\n    meta=pd.DataFrame({'station_id':stations,'canonical_line':1})\n    return frame,meta\n\nclass FeatureTests(unittest.TestCase):\n    def test_partial_and_latest_origins(self):\n        origins=partial_origins('2026-06-30')\n        self.assertEqual(origins[0],pd.Timestamp('2025-02-11'))\n        self.assertEqual(origins[-1],pd.Timestamp('2026-06-16'))\n        frame,_=fixture();pairs,origins=sampled_pairs(frame,'2026-06-30',12300)\n        dates=frame.loc[pairs.row,'date'].reset_index(drop=True)\n        os=pd.Series([origins[i] for i in pairs.origin_index])\n        self.assertTrue((dates>os).all());self.assertTrue((dates<=pd.Timestamp('2026-06-30')).all())\n        self.assertTrue(((dates-os).dt.days==pairs.horizon).all())\n        self.assertEqual(pairs.horizon.nunique(),123)\n        self.assertLessEqual(pairs.horizon.value_counts().max()-pairs.horizon.value_counts().min(),1)\n        self.assertTrue((pairs.origin_index==len(origins)-1).any())\n    def test_cleaning_opening_holiday_break(self):\n        frame,_=fixture();h=cleaned(frame,'2025-08-30')\n        self.assertFalse(h.loc[h.date=='2025-05-01','clean'].any())\n        self.assertFalse(h.loc[h.date=='2025-08-15','clean'].any())\n        self.assertFalse(h.loc[h.station_id.eq(3107)&h.date.between('2025-06-28','2025-06-30'),'clean'].any())\n        self.assertFalse(h.loc[h.station_id.isin([3110,3111])&(h.date<'2025-07-01'),'clean'].any())\n        self.assertTrue(h.loc[h.station_id.eq(3107)&h.date.eq('2025-07-01'),'clean'].all())\n    def test_future_invariance_and_holiday_anchor(self):\n        frame,meta=fixture();frame.loc[frame.date=='2025-08-15','boardings']=1e8\n        query=pd.DataFrame({'date':pd.to_datetime(['2025-10-03','2025-09-01']),'station_id':[3110,3110],'hour':[6,6]})\n        a=make_features_v2(frame,query,meta,'2025-08-30')\n        frame.loc[frame.date>'2025-08-30',['boardings','alightings']]=1e12\n        b=make_features_v2(frame,query,meta,'2025-08-30')\n        pd.testing.assert_frame_equal(a,b)\n        self.assertEqual(a.eff_dow.iloc[0],6)\n        np.testing.assert_allclose(a.anchor_board_log,np.log1p(10),rtol=1e-6)\n    def test_as_of_election_not_known_early(self):\n        early=calendar(pd.to_datetime(['2025-06-03']),'2025-02-11')\n        later=calendar(pd.to_datetime(['2025-06-03']),'2025-05-31')\n        self.assertEqual(int(early.is_election_day.iloc[0]),0)\n        self.assertEqual(int(later.is_election_day.iloc[0]),1)\n    def test_year_holiday_replaced(self):\n        frame,meta=fixture();frame.loc[frame.date=='2025-10-06','boardings']=10000\n        q=pd.DataFrame({'date':pd.to_datetime(['2026-10-05']),'station_id':[3110],'hour':[6]})\n        x=make_features_v2(frame,q,meta,'2026-06-30')\n        self.assertAlmostEqual(x.year364_clean_log.iloc[0],np.log1p(10),places=5)\n    def test_elasticity_uses_clean_ordinary_regime(self):\n        frame,meta=fixture(stations=(3110,3107))\n        frame.loc[(frame.station_id==3110)&(frame.date<'2025-07-01'),'boardings']=1e8\n        frame.loc[frame.date=='2025-05-01','boardings']=1e9\n        query=pd.DataFrame({'date':pd.to_datetime(['2025-09-01','2025-09-01']),'station_id':[3110,3107],'hour':[6,6]})\n        x=make_features_v2(frame,query,meta,'2025-08-30')\n        np.testing.assert_allclose(x.holiday_elasticity,0,atol=1e-6)\n\nclass NeuralAndEnsembleTests(unittest.TestCase):\n    def test_xgboost_validation_matches_inference(self):\n        import subprocess,sys,textwrap\n        code=textwrap.dedent('''\n            import numpy as np,pandas as pd\n            from src.xgb_compat import InferenceConsistentXGBRegressor\n            rng=np.random.default_rng(12)\n            tr=pd.DataFrame({'a':rng.normal(size=600),'b':pd.Categorical(rng.integers(0,5,600),categories=range(5))})\n            va=pd.DataFrame({'a':rng.normal(1,2,size=180),'b':pd.Categorical(rng.integers(0,5,180),categories=range(5))})\n            y=tr.a.to_numpy()*.3+tr.b.astype(int).to_numpy()*.2\n            v=va.a.to_numpy()*.3+va.b.astype(int).to_numpy()*.2\n            model=InferenceConsistentXGBRegressor(tree_method='hist',enable_categorical=True,n_estimators=25,max_depth=3,n_jobs=1,early_stopping_rounds=5,eval_metric='rmse',device='cpu')\n            model.fit(tr,y,eval_set=[(va,v)],verbose=False)\n            actual=np.sqrt(np.mean((model.predict(va)-v)**2))\n            recorded=model.evals_result()['validation_0']['rmse'][model.best_iteration]\n            assert abs(actual-recorded)<1e-6,(actual,recorded)\n        ''')\n        subprocess.run([sys.executable,'-c',code],check=True,capture_output=True,text=True)\n    def test_fold_preprocessor_unknown_mask_serialization(self):\n        x=pd.DataFrame({'station_id':[1,2,1],'value':[0.,2.,np.nan]})\n        p=pickle.loads(pickle.dumps(NNPreprocessor().fit(x)))\n        num,cat=p.transform(pd.DataFrame({'station_id':[3,1],'value':[np.nan,1e9]}))\n        self.assertEqual(cat[0,0],0);self.assertEqual(num[0,-1],1);self.assertEqual(num[1,0],10)\n        self.assertTrue(np.isfinite(num).all())\n    def test_tabm_shape_and_non_cancelling_loss(self):\n        import torch\n        from src.tabm_v2 import MultiTaskTabM,multitask_loss\n        torch.set_num_threads(1);model=MultiTaskTabM(6,[5,3],k=8,hidden=16)\n        out=model(torch.randn(7,6),torch.zeros(7,2,dtype=torch.long))\n        self.assertEqual(tuple(out.shape),(7,8,2))\n        p=torch.tensor([[[1.,2.],[-1.,-2.]]],requires_grad=True)\n        loss=multitask_loss(p,torch.zeros(1,2));self.assertAlmostEqual(loss.item(),1.8,places=5)\n        loss.backward();self.assertGreater(p.grad.abs().sum().item(),0)\n    def test_convex_and_lofo(self):\n        y=np.linspace(0,2,80);pred=np.column_stack([y+.1,y-.2,y+.3,y-.4]);fold=np.repeat(np.arange(1,5),20)\n        w=convex_weights(y,pred);self.assertAlmostEqual(w.sum(),1,places=10);self.assertTrue((w>=0).all())\n        self.assertLess(np.mean((pred@w-y)**2),1e-8)\n        _,_,report,_=evaluate_ensemble(y,pred,fold,['a','b','c','d'],[.15,.15,.35,.35])\n        self.assertEqual(len(report['lofo']),4)\n        for row in report['lofo']:self.assertAlmostEqual(sum(row['weights'].values()),1,places=9)\n\nif __name__=='__main__':unittest.main()\n", 'tests/test_ensemble_v3.py': "import unittest, pickle\nfrom unittest.mock import patch\nimport numpy as np\nimport pandas as pd\nfrom src.domain_events import oneoff_mask, add_domain_features, schedules\nfrom src.features_v3 import make_features_v3, cleaned, sampled_pairs\nfrom src.features_v2 import calendar\nfrom src.nn_preprocessor_v3 import NNPreprocessorV3\nfrom src.models_v3 import ResidualTree\nfrom src.ensemble_v3 import evaluate_ensemble\nfrom test_ensemble_v2 import fixture\n\nclass V3Tests(unittest.TestCase):\n    def test_opening_and_exact_outages(self):\n        rows=pd.MultiIndex.from_product([pd.date_range('2025-06-28','2025-06-30'),[3107,3108,3109],range(6,24)],names=['date','station_id','hour']).to_frame(index=False)\n        self.assertEqual(int(oneoff_mask(rows).sum()),162)\n        rows=pd.DataFrame({'date':pd.to_datetime(['2025-08-13']*3),'station_id':[3112]*3,'hour':[11,12,13]})\n        self.assertEqual(oneoff_mask(rows).tolist(),[False,True,True])\n\n    def test_schedule_asof_and_september(self):\n        q=pd.DataFrame({'date':pd.to_datetime(['2026-07-03','2026-09-04','2026-07-31']),'station_id':[3127,3127,3139],'hour':[21]*3})\n        early=pd.concat([q,calendar(q.date,'2025-12-01')],axis=1)\n        late=pd.concat([q,calendar(q.date,'2026-06-30')],axis=1)\n        # Synthetic schedule verifies the generic as-of path without loading\n        # KBO facts excluded from production for unresolved reuse conditions.\n        fixture_schedule={'kbo':[{'known_since':'2025-12-19','coverage_start':'2026-03-28','coverage_end':'2026-09-06','home_dates':['2026-07-03','2026-09-04']}],\n                          'festivals':[{'kind':'pentaport','known_since':'2026-05-31','start':'2026-07-31','end':'2026-08-02'}]}\n        with patch('src.domain_events.schedules',return_value=fixture_schedule):\n            a=add_domain_features(early,'2025-12-01');b=add_domain_features(late,'2026-06-30')\n        self.assertEqual(a.munhak_kbo_scheduled.sum(),0);self.assertEqual(a.songdo_pentaport_scheduled.sum(),0)\n        self.assertEqual(b.munhak_kbo_scheduled.sum(),2);self.assertEqual(b.songdo_pentaport_scheduled.sum(),1)\n        self.assertTrue(b.loc[1,'munhak_schedule_known'])\n        self.assertEqual(schedules()['kbo'],[])\n\n    def test_neural_anchor_imputation_and_mask(self):\n        x=pd.DataFrame({'station_id':[1,1,2,2],'anchor_board_log':[1.,3.,5.,7.],'year364_clean_log':[np.nan]*4,'yoy_seas_log_delta':[0.]*4})\n        p=pickle.loads(pickle.dumps(NNPreprocessorV3().fit(x)))\n        a,mask=p.values(x);idx=p.cont.index('year364_clean_log')\n        np.testing.assert_allclose(a[:,idx],x.anchor_board_log)\n        num,cat=p.transform(x)\n        np.testing.assert_allclose(num[:,idx],num[:,p.cont.index('anchor_board_log')])\n        self.assertTrue((num[:,len(p.cont)+idx]==1).all());self.assertTrue(np.isfinite(num).all())\n        self.assertGreater(p.iqr[idx],0)\n\n    def test_tree_category_separation(self):\n        x=pd.DataFrame({'station_id':[3110.,3111.],'station_hour':[1.,2.],'station_eff_dow':[3.,4.],'value':[np.nan,1.]})\n        for name in ['lightgbm','xgboost','catboost']:\n            model=ResidualTree(name,{});frame=model.frame(x,fit=True)\n            self.assertEqual('station_hour' in frame,name=='catboost')\n            self.assertTrue(pd.isna(frame.value.iloc[0]))\n\n    def test_unseen_event_is_neutral_in_neural_input(self):\n        x=pd.DataFrame({'station_id':[1,1,1], 'event':[0.,0.,0.], 'rare':[0.,0.,1.]})\n        p=NNPreprocessorV3().fit(x)\n        test=x.copy();test['event']=1\n        num,_=p.transform(test)\n        self.assertTrue((num[:,p.cont.index('event')]==0).all())\n        self.assertFalse(p.constant[p.cont.index('rare')])\n\n    def test_future_invariance_smooth_clean_and_sampling(self):\n        h,meta=fixture(stations=(3110,3112,3107))\n        h.loc[(h.station_id==3112)&(h.date=='2025-08-13')&(h.hour==7),'boardings']=10 # normal zero values are not globally removed\n        q=pd.DataFrame({'date':pd.to_datetime(['2026-08-15','2026-09-01']),'station_id':[3110,3107],'hour':[7,7]})\n        a=make_features_v3(h,q,meta,'2026-06-30')\n        np.testing.assert_allclose(a.yoy_smooth15_log_delta,0,atol=1e-6)\n        q2=pd.DataFrame({'date':pd.to_datetime(['2026-05-01']),'station_id':[3110],'hour':[7]})\n        before=make_features_v3(h,q2,meta,'2026-02-27')\n        h.loc[h.date>'2026-02-27',['boardings','alightings']]=1e12\n        after=make_features_v3(h,q2,meta,'2026-02-27')\n        pd.testing.assert_frame_equal(before,after)\n        pairs,origins=sampled_pairs(h,'2026-02-27',1230)\n        labels=h.loc[pairs.row]\n        self.assertFalse(oneoff_mask(labels).any())\n        self.assertTrue((labels.date<=pd.Timestamp('2026-02-27')).all())\n        self.assertEqual(pairs.horizon.nunique(),123)\n\n    def test_recent_weights_ignore_early_labels(self):\n        folds=np.repeat([1,2,3,4],20);y=np.ones(80)\n        p=np.column_stack([y+.1,y-.2]);dates=pd.date_range('2025-01-01',periods=80)\n        _,_,a,_=evaluate_ensemble(y,p,folds,['a','b'],[.15,.15,.35,.35],dates)\n        y[:40]=100\n        _,_,b,_=evaluate_ensemble(y,p,folds,['a','b'],[.15,.15,.35,.35],dates)\n        np.testing.assert_allclose(list(a['weights'].values()),list(b['weights'].values()))\n        self.assertAlmostEqual(sum(b['weights'].values()),1)\n        self.assertEqual(a['selection_rows'],40)\n\nif __name__=='__main__':unittest.main()\n", 'tests/test_features.py': '"""Leakage and horizon-boundary regression tests using independent toy data."""\nfrom pathlib import Path\nimport sys\nimport unittest\n\nimport numpy as np\nimport pandas as pd\nfrom pandas.testing import assert_frame_equal\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom src.features import make_features, training_matrix\n\n\ndef fixture(periods=500):\n    dates = pd.date_range("2025-01-01", periods=periods)\n    records = []\n    for station in [101, 202]:\n        for hour in [6, 17]:\n            for day, date in enumerate(dates):\n                value = station + hour + day * 3\n                records.append((date, station, hour, value, value * 2))\n    history = pd.DataFrame(records, columns=["date", "station_id", "hour", "boardings", "alightings"])\n    metadata = pd.DataFrame({\n        "station_id": [101, 202], "canonical_line": [1, 2],\n        "station_name": ["A", "B"], "first_date": ["2025-01-01"]*2,\n        "last_date": ["2026-06-30"]*2,\n    })\n    return history, metadata\n\n\ndef queries(origin, horizons=(1, 7, 123)):\n    return pd.DataFrame([\n        (pd.Timestamp(origin) + pd.Timedelta(days=horizon), station, hour)\n        for horizon in horizons for station in [101, 202] for hour in [6, 17]\n    ], columns=["date", "station_id", "hour"])\n\n\nclass FeatureTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        cls.history, cls.metadata = fixture()\n        cls.origin = pd.Timestamp("2025-12-31")\n\n    def test_future_target_and_alightings_mutation_cannot_change_features(self):\n        query = queries(self.origin)\n        expected = make_features(self.history, query, self.metadata, self.origin)\n        mutated = self.history.copy()\n        future = mutated.date > self.origin\n        mutated.loc[future, "boardings"] = 999_999_999\n        mutated.loc[future, "alightings"] = -999_999_999\n        actual = make_features(mutated, query, self.metadata, self.origin)\n        assert_frame_equal(expected, actual, check_exact=True)\n        # Supplying hidden labels on query rows must not expose them as features.\n        labelled_query = query.assign(boardings=123_456, alightings=654_321)\n        assert_frame_equal(expected, make_features(self.history, labelled_query, self.metadata, self.origin), check_exact=True)\n\n    def test_query_reordering_keeps_each_feature_attached_to_its_key(self):\n        query = queries(self.origin)\n        expected = make_features(self.history, query, self.metadata, self.origin)\n        permutation = np.array([8, 2, 11, 0, 5, 9, 3, 7, 1, 10, 4, 6])\n        actual = make_features(self.history, query.iloc[permutation], self.metadata, self.origin)\n        assert_frame_equal(expected.iloc[permutation].reset_index(drop=True), actual, check_exact=True)\n\n    def test_seasonal7_repeats_latest_observed_weekday_for_all_horizons(self):\n        query = queries(self.origin)\n        features = make_features(self.history, query, self.metadata, self.origin)\n        for index, row in query.iterrows():\n            observed = self.history.loc[\n                (self.history.station_id == row.station_id)\n                & (self.history.hour == row.hour)\n                & (self.history.date <= self.origin)\n                & (self.history.date.dt.dayofweek == row.date.dayofweek)\n            ].sort_values("date").iloc[-1]\n            self.assertEqual(features.loc[index, "seasonal7"], observed.boardings)\n            self.assertEqual(features.loc[index, "seasonal7_alight"], observed.alightings)\n            self.assertEqual(features.loc[index, "horizon"], (row.date - self.origin).days)\n\n    def test_year364_never_uses_lookup_after_origin(self):\n        origin = pd.Timestamp(\'2025-11-26\')\n        query = queries(origin, horizons=(1, 123, 364, 365, 400))\n        features = make_features(self.history, query, self.metadata, origin)\n        for index, row in query.iterrows():\n            lookup_date = row.date - pd.Timedelta(days=364)\n            if lookup_date > origin or lookup_date < self.history.date.min():\n                self.assertTrue(np.isnan(features.loc[index, "year364"]))\n            else:\n                match = self.history.loc[(self.history.date == lookup_date) & (self.history.station_id == row.station_id) & (self.history.hour == row.hour)]\n                self.assertEqual(features.loc[index, "year364"], match.boardings.iloc[0])\n\n    def test_metadata_last_date_and_future_first_date_do_not_leak(self):\n        query = queries(self.origin)\n        expected = make_features(self.history, query, self.metadata, self.origin)\n        metadata = self.metadata.assign(first_date="2099-01-01", last_date="1900-01-01", station_name="Arbitrary")\n        actual = make_features(self.history, query, metadata, self.origin)\n        assert_frame_equal(expected, actual, check_exact=True)\n        self.assertNotIn("last_date", actual.columns)\n        self.assertNotIn("first_date", actual.columns)\n\n    def test_full_123_day_label_window_cutoff_boundary(self):\n        # First anchor is Jan 28. A cutoff exactly 123 days later permits it;\n        # one day earlier must not shorten the training label window silently.\n        history, metadata = fixture(periods=200)\n        anchor = pd.Timestamp("2025-01-28")\n        cutoff = anchor + pd.Timedelta(days=123)\n        x, y, audit = training_matrix(history, metadata, cutoff, horizon=123, max_rows=10000)\n        self.assertEqual(len(audit), 1)\n        self.assertEqual(audit[0]["origin"], "2025-01-28")\n        self.assertEqual(pd.Timestamp(audit[0]["label_end"]), cutoff)\n        self.assertEqual(x.horizon.min(), 1)\n        self.assertEqual(x.horizon.max(), 123)\n        self.assertEqual(len(x), 123 * 4)\n        self.assertEqual(len(x), len(y))\n        self.assertTrue(all(pd.Timestamp(a["label_end"]) <= cutoff for a in audit))\n        self.assertTrue(all(pd.Timestamp(a["origin"]) + pd.Timedelta(days=123) <= cutoff for a in audit))\n        with self.assertRaisesRegex(ValueError, "Insufficient history"):\n            training_matrix(history, metadata, cutoff - pd.Timedelta(days=1), horizon=123)\n        # Changing labels beyond the cutoff cannot change either X or y.\n        mutated = history.copy()\n        mutated.loc[mutated.date > cutoff, ["boardings", "alightings"]] = 999_999\n        changed_x, changed_y, changed_audit = training_matrix(mutated, metadata, cutoff, horizon=123, max_rows=10000)\n        assert_frame_equal(x, changed_x, check_exact=True)\n        np.testing.assert_array_equal(y, changed_y)\n        self.assertEqual(audit, changed_audit)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'tests/test_holiday_calendar.py': '"""Holiday declarations must not leak across the forecast-origin boundary."""\nfrom pathlib import Path\nimport sys\nimport unittest\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom src.holiday_calendar import calendar_features, holiday_records\n\n\nclass HolidayCalendarTests(unittest.TestCase):\n    def test_temporary_holiday_announcement_boundary(self):\n        dates = ["2025-01-26", "2025-01-27", "2025-01-28"]\n        before = calendar_features(dates, as_of="2025-01-13")\n        after = calendar_features(dates, as_of="2025-01-14")\n        self.assertEqual(before.is_holiday.tolist(), [0, 0, 1])\n        self.assertEqual(after.is_holiday.tolist(), [0, 1, 1])\n        self.assertEqual(before.loc[0, "is_holiday_eve"], 0)\n        self.assertEqual(after.loc[0, "is_holiday_eve"], 1)\n        self.assertEqual(before.loc[1, "holiday_type"], 0)\n\n    def test_2025_presidential_election_was_not_known_in_january(self):\n        self.assertEqual(calendar_features(["2025-06-03"], "2025-01-31").is_holiday.iloc[0], 0)\n        after = calendar_features(["2025-06-03"], "2025-04-08")\n        self.assertEqual(after.is_holiday.iloc[0], 1)\n        self.assertEqual(after.is_election_day.iloc[0], 1)\n\n    def test_labor_day_and_constitution_day_new_designations(self):\n        dates = ["2025-05-01", "2026-05-01", "2025-07-17", "2026-07-17"]\n        final = calendar_features(dates)\n        self.assertEqual(final.is_holiday.tolist(), [0, 1, 0, 1])\n        before = calendar_features(dates, "2026-02-02")\n        self.assertEqual(before.is_holiday.tolist(), [0, 0, 0, 0])\n        self.assertEqual(before.is_labor_day.tolist(), [1, 1, 0, 0])\n\n    def test_no_false_substitutes(self):\n        dates = ["2026-09-26", "2026-09-27", "2026-09-28", "2026-06-08", "2026-10-05"]\n        frame = calendar_features(dates)\n        self.assertEqual(frame.is_holiday.tolist(), [1, 0, 0, 0, 1])\n        self.assertEqual(frame.is_rest_day.tolist(), [1, 1, 0, 0, 1])\n        self.assertEqual(frame.is_substitute_holiday.tolist(), [0, 0, 0, 0, 1])\n\n    def test_year_and_test_scope_counts(self):\n        records = holiday_records()\n        self.assertEqual((records.date.str.startswith("2025")).sum(), 19)\n        self.assertEqual((records.date.str.startswith("2026")).sum(), 22)\n        test = calendar_features(pd.date_range("2026-07-01", "2026-10-31"))\n        self.assertEqual(test.is_holiday.sum(), 9)\n\n    def test_order_index_dtype_and_empty_input(self):\n        dates = pd.Series(["2026-10-05", "2025-05-01", "2026-10-05"], index=[9, 4, 9])\n        frame = calendar_features(dates)\n        self.assertEqual(frame.index.tolist(), [9, 4, 9])\n        self.assertEqual(frame.is_holiday.tolist(), [1, 0, 1])\n        self.assertTrue(all(t == np.dtype("int8") for t in frame.dtypes))\n        self.assertEqual(calendar_features([]).shape, (0, 12))\n\n    def test_unsupported_dates_and_post_cutoff_origin_rejected(self):\n        for dates in [[None], ["2027-01-01"], ["2024-12-31"]]:\n            with self.assertRaises(ValueError):\n                calendar_features(dates)\n        with self.assertRaises(ValueError):\n            calendar_features(["2026-07-17"], "2026-07-01")\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'tests/test_lightgbm_tuning.py': '"""Metric, parameter isolation, and common-round selection checks."""\nimport unittest\nimport numpy as np\nfrom src.tune_lightgbm import clipped_log_rmsle,choose_common_round,candidate_parameters\nfrom src.models import lightgbm_parameters\n\nclass LightGBMTuningTests(unittest.TestCase):\n    def test_metric_matches_official_count_rmsle_with_clipping(self):\n        y=np.array([0.,1.,8.]);p=np.array([-2.,.5,2.5])\n        name,actual,greater=clipped_log_rmsle(np.log1p(y),p)\n        expected=np.sqrt(np.mean((np.log1p(np.expm1(np.maximum(p,0)))-np.log1p(y))**2))\n        self.assertEqual(name,\'rmsle\');self.assertFalse(greater)\n        self.assertAlmostEqual(actual,expected,places=12)\n\n    def test_round_selection_uses_pooled_squared_error_and_common_prefix(self):\n        iteration,score,common,curve=choose_common_round([[0.,.6,0.],[1.,.6]],[1,1])\n        # Arithmetic mean would pick iteration 1; pooled squared error picks 2.\n        self.assertEqual(iteration,2);self.assertEqual(common,2)\n        self.assertAlmostEqual(score,.6)\n        self.assertAlmostEqual(curve[0],np.sqrt(.5))\n\n    def test_selection_respects_hard_cap_and_row_counts(self):\n        a=np.ones(1001);a[-1]=0\n        iteration,_,common,_=choose_common_round([a,a],[2,3],1000)\n        self.assertEqual(common,1000);self.assertEqual(iteration,1)\n        _,_,_,curve=choose_common_round([[1.,2.],[3.,4.]],[1,3])\n        self.assertAlmostEqual(curve[0],np.sqrt(7))\n\n    def test_legacy_parameters_unchanged_and_override_is_explicit(self):\n        cfg={\'seed\':2026,\'threads\':4}\n        self.assertEqual(lightgbm_parameters(cfg)[\'n_estimators\'],220)\n        self.assertEqual(lightgbm_parameters(cfg,False)[\'num_leaves\'],31)\n        changed=lightgbm_parameters(dict(cfg,lightgbm_params={\'n_estimators\':437,\'reg_lambda\':10}))\n        self.assertEqual(changed[\'n_estimators\'],437)\n        self.assertEqual(changed[\'reg_lambda\'],10)\n        with self.assertRaises(ValueError):lightgbm_parameters(dict(cfg,lightgbm_params={\'objective\':\'poisson\'}))\n\n    def test_search_is_deterministic_and_includes_original_parameters(self):\n        a=candidate_parameters(2026,16)\n        self.assertEqual(a,candidate_parameters(2026,16))\n        self.assertEqual(len(a),16);self.assertEqual(a[0][\'num_leaves\'],63)\n        self.assertEqual(a[0][\'learning_rate\'],.055)\n\nif __name__==\'__main__\':unittest.main()\n', 'tests/test_metrics_validation.py': '"""Behavior checks for score conventions, leakage and submission corruption."""\nimport math\nfrom pathlib import Path\nimport sys\nimport unittest\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))\nfrom metrics import forecast_diagnostics, regression_metrics, residual_autocorrelation, score, seasonal_scales\nfrom validate_submission import validate_submission\n\n\nclass MetricTests(unittest.TestCase):\n    def test_score_conventions(self):\n        result = regression_metrics([0, 2, 4], [0, 4, 1])\n        self.assertAlmostEqual(result["mae"], 5 / 3)\n        self.assertAlmostEqual(result["rmse"], math.sqrt(13 / 3))\n        self.assertAlmostEqual(result["mbe"], -1 / 3)\n        self.assertAlmostEqual(result["smape"], (0 + 200 * 2 / 6 + 200 * 3 / 5) / 3)\n        self.assertAlmostEqual(result["rmsle"], math.sqrt(((math.log(5)-math.log(3))**2 + (math.log(2)-math.log(5))**2)/3))\n        self.assertEqual(score([0, 0], [0, 0], "smape"), 0)\n\n    def test_invalid_inputs_rejected(self):\n        for y, p in [([], []), ([1], [1, 2]), ([1], [np.nan]), ([np.inf], [1])]:\n            with self.assertRaises(ValueError):\n                regression_metrics(y, p)\n        with self.assertRaises(ValueError):\n            score([1], [-1], "rmsle")\n\n    def test_official_rmsle_is_root_mean_squared_log1p_error(self):\n        truth = np.expm1(np.array([0., 1., 2.]))\n        predictions = np.expm1(np.array([1., 3., 5.]))\n        expected = math.sqrt((1.0**2 + 2.0**2 + 3.0**2) / 3)\n        self.assertAlmostEqual(score(truth, predictions, "rmsle"), expected, places=12)\n        self.assertAlmostEqual(regression_metrics(truth, predictions)["rmsle"], expected, places=12)\n        self.assertEqual(score([0, 1, 20], [0, 1, 20], "rmsle"), 0)\n        self.assertAlmostEqual(score([0], [1], "rmsle"), math.log(2), places=12)\n\n    def test_seasonal_scale_matches_calendar_not_rows(self):\n        train = pd.DataFrame({"station_id": [1]*3, "hour": [6]*3, "date": ["2025-01-01", "2025-01-08", "2025-01-16"], "boardings": [10, 12, 99]})\n        scales = seasonal_scales(train)\n        self.assertEqual(scales.loc[0, "seasonal_pairs"], 1)\n        self.assertEqual(scales.loc[0, "mase_scale"], 2)\n\n    def test_zero_scales_undefined_and_coverage_reported(self):\n        train = pd.DataFrame({"station_id": [1]*8+[2]*8, "hour": [6]*16, "date": list(pd.date_range("2025-01-01", periods=8))*2, "boardings": list(range(8))+[3]*8})\n        valid = pd.DataFrame({"station_id": [1, 2], "hour": [6, 6], "date": ["2025-01-09"]*2, "boardings": [8, 3]})\n        result = forecast_diagnostics(train, valid, [15, 4])\n        self.assertAlmostEqual(result["summary"]["mase_macro"], 1)\n        self.assertAlmostEqual(result["summary"]["mase_row_weighted"], 1)\n        self.assertEqual(result["summary"]["mase_row_coverage"], 0.5)\n        self.assertEqual(result["summary"]["mase_undefined_series"], 1)\n        self.assertEqual(result["per_step"]["horizon"].tolist(), [1])\n        with self.assertRaisesRegex(ValueError, "leakage"):\n            forecast_diagnostics(train, valid, [15, 4], origin="2025-01-07")\n\n    def test_residual_acf_does_not_cross_groups_or_missing_days(self):\n        frame = pd.DataFrame({"station_id": [1]*4 + [2]*4, "hour": [6]*8, "date": list(pd.date_range("2025-01-01", periods=4))*2, "residual": [1,2,3,4,100,90,80,70]})\n        acf = residual_autocorrelation(frame, lags=(1, 7))\n        self.assertTrue(np.allclose(acf.loc[acf.lag_days.eq(1), "acf"], 1))\n        self.assertEqual(acf.loc[acf.lag_days.eq(1), "pairs"].tolist(), [3, 3])\n        self.assertEqual(acf.loc[acf.lag_days.eq(7), "pairs"].tolist(), [0, 0])\n        self.assertTrue(acf.loc[acf.lag_days.eq(7), "acf"].isna().all())\n\n\nclass SubmissionTests(unittest.TestCase):\n    def setUp(self):\n        self.test = pd.DataFrame({"example_id": ["J0", "J1", "J2"], "date": ["2026-07-01", "2026-09-30", "2026-10-01"]})\n        self.sample = pd.DataFrame({"example_id": ["J0", "J1"], "boardings": [0, 0]})\n        self.sub = pd.DataFrame({"example_id": ["J0", "J1"], "boardings": [1.2, 2.3]})\n\n    def test_template_discrepancy_explicit(self):\n        result = validate_submission(self.sub, self.test, self.sample, scope="template")\n        self.assertTrue(result["valid"])\n        self.assertFalse(result["template_matches_test"])\n        self.assertEqual(result["scope_discrepancy"]["test_ids_absent_from_template"], 1)\n        self.assertEqual(len(result["warnings"]), 1)\n        self.assertFalse(result["matches_official_scope"])\n        self.assertIn("diagnostic only", result["warnings"][0])\n        self.assertEqual(result["scope_discrepancy"]["missing_period_start"], "2026-10-01")\n        with self.assertRaisesRegex(ValueError, "requires 3"):\n            validate_submission(self.sub, self.test, self.sample, scope="test")\n\n    def test_invalid_files_rejected(self):\n        invalid = [self.sub.iloc[::-1], self.sub.assign(boardings=[np.nan, 2]), self.sub.assign(boardings=[np.inf, 2]), self.sub.assign(boardings=[-1, 2]), self.sub.assign(example_id=["J0", "J0"]), self.sub.assign(extra=0), self.sub.assign(example_id=["J0", "BAD"])]\n        for frame in invalid:\n            with self.subTest(frame=frame.to_dict()):\n                with self.assertRaises(ValueError):\n                    validate_submission(frame, self.test, self.sample, scope="template")\n\n    def test_test_scope_and_distribution(self):\n        sub = pd.DataFrame({"example_id": self.test.example_id, "boardings": [1, 2, 3]})\n        result = validate_submission(sub, self.test, self.sample, scope="test", reference_predictions=[1, 1, 1])\n        self.assertEqual(result["rows"], 3)\n        self.assertTrue(result["matches_official_scope"])\n        self.assertEqual(result["distribution_comparison"]["mean_ratio"], 2)\n\n    def test_default_requires_official_full_test_scope(self):\n        with self.assertRaisesRegex(ValueError, "requires 3"):\n            validate_submission(self.sub, self.test, self.sample)\n        full = pd.DataFrame({"example_id": self.test.example_id, "boardings": [1, 2, 3]})\n        result = validate_submission(full, self.test, self.sample)\n        self.assertEqual(result["scope"], "test")\n        self.assertTrue(result["matches_official_scope"])\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'tests/test_postprocessing.py': '"""Cutoff, holiday, centering and row alignment checks for postprocessing."""\nimport unittest\nimport numpy as np\nimport pandas as pd\nfrom src.postprocessing import build_context,fit_plan,apply_plan\n\ndef spec(kind=\'none\',scale=0.,days=28,weight=.2):\n    return {\'name\':\'test\',\'calibration\':{\'kind\':kind,\'scale\':scale},\'anchor\':{\'days\':days,\'weight\':weight},\'cap_days\':0,\'eligible\':True}\n\nclass PostprocessingTests(unittest.TestCase):\n    def setUp(self):\n        dates=pd.date_range(\'2025-01-01\',\'2025-02-05\')\n        self.history=pd.DataFrame({\'date\':dates,\'station_id\':1,\'hour\':6,\'boardings\':100.})\n        # Jan 27 is a preannounced temporary holiday, excluded from Monday profile.\n        self.history.loc[self.history.date==\'2025-01-27\',\'boardings\']=10000.\n        self.query=pd.DataFrame({\'date\':pd.to_datetime([\'2025-02-03\',\'2025-03-01\',\'2025-02-03\']),\'station_id\':[1,1,99],\'hour\':6})\n\n    def test_history_is_cutoff_safe_and_query_targets_are_ignored(self):\n        original=build_context(self.history,self.query,\'2025-01-31\')\n        changed=self.history.copy();changed.loc[changed.date>\'2025-01-31\',\'boardings\']=1e8\n        query=self.query.assign(boardings=-999999)\n        pd.testing.assert_frame_equal(original,build_context(changed,query,\'2025-01-31\'))\n\n    def test_historical_holidays_excluded_and_missing_anchor_falls_back(self):\n        context=build_context(self.history,self.query,\'2025-01-31\')\n        self.assertAlmostEqual(context.anchor_28.iloc[0],np.log1p(100.))\n        self.assertTrue(np.isnan(context.anchor_28.iloc[2]))\n        pred=np.array([200.,1000.,333.])\n        plan=fit_plan(context,pred,pred,spec())\n        result=apply_plan(context,pred,plan)\n        self.assertAlmostEqual(result[0],np.expm1(.8*np.log1p(200)+.2*np.log1p(100)))\n        self.assertEqual(result[1],pred[1]) # March 1 holiday unchanged exactly.\n        self.assertAlmostEqual(result[2],pred[2])\n\n    def test_context_and_group_correction_preserve_query_order(self):\n        context=build_context(self.history,self.query,\'2025-01-31\')\n        pred=np.array([200.,1000.,333.]);plan=fit_plan(context,pred,pred,spec())\n        expected=apply_plan(context,pred,plan)\n        order=[2,0,1]\n        changed=build_context(self.history,self.query.iloc[order],\'2025-01-31\')\n        np.testing.assert_array_equal(apply_plan(changed,pred[order],plan),expected[order])\n\n    def test_per_fold_centering_does_not_transport_global_bias(self):\n        context=pd.DataFrame({\'hour\':[6,7,6,7],\'dow\':[0,0,0,0],\'is_holiday\':False})\n        pred=np.expm1(np.full(4,4.))\n        truth=np.expm1(np.array([4.2,4.2,3.7,3.7]))\n        plan=fit_plan(context,pred,truth,spec(\'hour\',.5,0,0),fold_ids=[1,1,2,2])\n        self.assertTrue(all(abs(v)<1e-12 for v in plan[\'offsets\'].values()))\n\n    def test_identity_exact_and_anchor_share_guard(self):\n        context=build_context(self.history,self.query,\'2025-01-31\')\n        pred=np.array([.123456785,1000.,333.])\n        plan=fit_plan(context,pred,pred,spec(days=0,weight=0))\n        np.testing.assert_array_equal(apply_plan(context,pred,plan),pred)\n        plan[\'spec\'][\'anchor\']={\'days\':28,\'weight\':.5}\n        with self.assertRaises(ValueError):apply_plan(context,pred,plan)\n\nif __name__==\'__main__\':unittest.main()\n'}
SOURCE_HASHES = {'configs/ensemble_v3.json': 'cf880d4f25123f2d81758015cbee5f2c4efa5763b6bd9ca5ae44d16ae141bc88', 'reports/holiday_sources.md': '0988a6cd6d3eda4d703e7a6250c9c38f9c267c4c9a4b05b1796082660ebc910a', 'requirements.txt': '0b72d59bf1f0bb4f67719ece9451c5ee3d3f48a040be818031ea939fcddc21a2', 'src/__init__.py': 'f526452b64e34c742e0f8ce4ab662b851774fc308d3c57f7ac247d51f302d207', 'src/domain_events.py': 'f919f383b98eab35c456061ec340980384c9e789c33fbd39a17816cb91b5795d', 'src/domain_schedule_v3.json': 'dd2f7f7cfdda550740fbc65aaf16050c0922ed748597928fd2a163b6ead73ebc', 'src/eda.py': '9891970efc533f6b3d9510b3d5d49ac409511e7224d02cb7312a481bf28e54df', 'src/ensemble.py': 'd955667ffc968daac36ea27835d0d32fe91eab6eaf47c3bce06f351723ef9a33', 'src/ensemble_v2.py': '226eeb7a8ac6a7c424b428b15295d934e4cb3185da7975675ff5932f4042199c', 'src/ensemble_v3.py': 'a4228c5efd789ec39b89307c6335b825fd5f1882d7dd82ce877dcf71edbeb205', 'src/evaluate_postprocessing.py': '7dc58353fdf7f6dfccd0480e44a0ab4de0ce3570a6b7bd667a51d2d073c4a27b', 'src/features.py': '171d90496c2464b5efc55e25f8efd1cad33db2c01b01613c94359171999faa17', 'src/features_v2.py': '2da7ab555351619f1fa9b92b288d906f161d4f1e19c2554fe7f56d416956f0c3', 'src/features_v3.py': 'fcd0c65d925f312e502ec1fa7e7f286f191b22865136959604b51422b2058eb4', 'src/holiday_calendar.py': '50defb4fb19fcc52a1f35e568deb8d94e5b89ec275e7ace57138cff658a17e64', 'src/metrics.py': '9fdda3b049e583d372401a4ddc6066c81ae8d5b2d1ef4560af4ffd23fa78c571', 'src/models.py': 'fdf47108a6f7aac0632e56f7634f252a023409bbf776c25571c0a7d29c52a302', 'src/models_v2.py': '15fcfb851af4ae76168f4b21bfdbea0bc72a5ab6b421aeecff75cf8fb9b6a80f', 'src/models_v3.py': 'aa3ee4c9c9697ff1201ba6fbf206e702624ba273d5a411a94a9874d6a106b653', 'src/nn_preprocessor.py': 'abe6d243eec067abaa59b7d1eb74538f1fb1b81ce70ab64fd0d315580a6a38bb', 'src/nn_preprocessor_v3.py': 'ad1503a70425b75672f752cddaa87f89d8ab043e080ce3ce5f12c215c7cd66ec', 'src/pipeline.py': 'd51f54f1d827b612f7390da4a7620287c5859f1e3a93ced447fee7ef77d02f77', 'src/pipeline_v2.py': '9788a6a2bd16047736b8c4a2ad8a2bbe6537fca5f90b6e9c43f2363c58466be9', 'src/pipeline_v3.py': 'ef90d9fc6c9e8481a3598fae168052e87032d3bf82cde27a0c2de1b0fd5a3827', 'src/postprocessing.py': '64d8d4aef97156c0f1efdb4e1b5f8c771343ca48c9c168a93cda8604b64c6f5d', 'src/tabm_v2.py': 'ce01e8d3392bc821d5c5caf02f0e740880b20293f8f03ade9d202d269f66cd0e', 'src/tabm_v3.py': 'c5e3a8b4dcc2f2e5bc28e9a69de2c585455a83e42ad377b1c303274062872e16', 'src/tune_lightgbm.py': 'e5acf34fa09925ddff60deb20ad84e74082ec5ba5bfda38908a03e70eeb3c663', 'src/validate_submission.py': '64667df2339bcf0d5fc4466807dc3ef21c892980a81ed862c3f919c3aece3d4e', 'src/xgb_compat.py': '4871dcd95040249acb2ccd8f9136fe6d419b0a99ceafe39e80fa53ef2c96bacf', 'tests/test_ensemble.py': '4321ba681dac343a4f8f466221db347dcf3f40ac2b77ab700a4670de735125cd', 'tests/test_ensemble_v2.py': 'b78d72139c759a97554d49ee51b9aaa2125366afdc59f95f64bcee4771eebc1f', 'tests/test_ensemble_v3.py': '16d27767c34af1cdf7887ac543ad72e8bb3fc2ee3eb34ed785a84317b7d3f36c', 'tests/test_features.py': 'd8a7966d2bc219f8230fd37822cc2893f73a47aa11fe9c24575bfd3472afddd1', 'tests/test_holiday_calendar.py': 'c9e76a9f4936f28bacb42e110a3739822782e8cdbd91a9eac8604a3372e81c48', 'tests/test_lightgbm_tuning.py': '90347ddd2ca9e241604305a6dd8ae659df21bd608016506377489a10871f73e4', 'tests/test_metrics_validation.py': 'ae5e956c969e3b02c55f9f6c934d86fdd19bb0e8dd009c328582fd84f5da5680', 'tests/test_postprocessing.py': '39cdb7d18649f4c2d4e2b09229bb338211da0162f14f7c38dfeed91ec3d6deb1'}
for name, content in EMBEDDED_FILES.items():
    assert hashlib.sha256(content.encode()).hexdigest() == SOURCE_HASHES[name]
    path = PROJECT_ROOT/name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
print('Restored source files:', len(EMBEDDED_FILES))


In [ ]:
DATA_DIR_OVERRIDE = ''  # Colab: /content/sincon-2026
required = ['train.csv', 'test.csv', 'metaData.csv', 'sample_submission.csv']
if DATA_DIR_OVERRIDE:
    DATA_DIR = Path(DATA_DIR_OVERRIDE)
elif IS_KAGGLE:
    candidates = sorted({p.parent for p in Path('/kaggle/input').rglob('train.csv') if all((p.parent/n).exists() for n in required)})
    assert len(candidates) == 1, f'Set DATA_DIR_OVERRIDE; candidates={candidates}'
    DATA_DIR = candidates[0]
else:
    DATA_DIR = Path('/content/sincon-2026')
assert all((DATA_DIR/n).exists() for n in required), 'Prepare all four original CSV files'
OUT = PROJECT_ROOT/'outputs/ensemble_v3'
print('Data:', DATA_DIR, 'Output:', OUT)

## 의존성과 테스트
참조 패키지 버전은 requirements.txt에 고정되어 있습니다. CUDA 런타임에 이미 설치된 PyTorch는
유지하며, 나머지 패키지 버전을 맞춥니다. 오프라인 환경에서는 해당 wheel을 미리 준비하세요.
환경 버전과 장치는 실행 결과 environment.json에 기록됩니다.

In [ ]:
INSTALL_REQUIREMENTS = True
if INSTALL_REQUIREMENTS:
    import importlib.util
    packages = [line for line in (PROJECT_ROOT/'requirements.txt').read_text().splitlines() if line and not line.startswith('torch==')]
    if importlib.util.find_spec('torch') is None:
        packages.append('torch==2.15.0')
    subprocess.run([sys.executable, '-m', 'pip', 'install', *packages], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=PROJECT_ROOT, check=True)

## 전체 실행
동일 코드·설정·입력 해시로 재실행하면 완료한 fold/model을 재사용합니다.
OOF 날짜가 폴드 사이에 중복되므로 LOFO 점수는 완전히 독립적인 holdout 점수가 아닙니다.

In [ ]:
subprocess.run([sys.executable, '-u', '-m', 'src.pipeline_v3', 'run', '--data-dir', str(DATA_DIR)], cwd=PROJECT_ROOT, check=True)
print((OUT/'ensemble_report.json').read_text())
print((OUT/'submission_validation.json').read_text())

In [ ]:
before = (OUT/'submission_ensemble_v3.csv').read_bytes()
subprocess.run([sys.executable, '-m', 'src.pipeline_v3', 'infer', '--data-dir', str(DATA_DIR)], cwd=PROJECT_ROOT, check=True)
import pandas as pd, numpy as np, io
old, new = pd.read_csv(io.BytesIO(before)), pd.read_csv(OUT/'submission_ensemble_v3.csv')
assert old.example_id.equals(new.example_id)
assert np.allclose(old.boardings, new.boardings, rtol=2e-5, atol=1e-5)
print('Reloaded prediction agreement verified. CPU/GPU floating-point differences are tolerated.')
from IPython.display import FileLink, display
display(FileLink(str(OUT/'submission_ensemble_v3.csv')))